# Trabajo Práctico 3: Redes neuronales

Cada sección tiene un responsable (ver `JOURNAL.md`). Agregar celdas solo debajo
de los encabezados propios y no mover ni editar celdas de otras secciones.

## 0. Configuración común

*Acordada en T0 entre los tres. Los cambios posteriores se registran en `JOURNAL.md`
y se avisan al equipo.*

- **Rutas.** Todas se construyen desde la raíz del repositorio, por lo que el notebook
  funciona igual si el kernel arranca en la raíz o en `notebooks/`. Los CSV de
  UNSW-NB15 van en `data/raw/unsw_nb15/` y se versionan en Git (acuerdo del 2026-11-25).
- **Semillas.** Cada sección crea su propio generador con su semilla, por ejemplo
  `rng_s2 = np.random.default_rng(SEEDS["s2_xor"])`, y usa nombres de variables con su
  prefijo. Así sus resultados no dependen de lo que se ejecutó antes.
- **Ejecución reducida.** Con la variable de entorno `TP3_QUICK=1`, las secciones pueden
  reducir épocas o repeticiones con `quick_or_full(reducido, completo)` mientras se
  desarrolla. La ejecución final (T8) se hace sin esa variable, con los valores del
  enunciado.
- **Normalización L2.** `USE_L2 = False`: §4.1 (T4) comparó con y sin L2 y decidió no usarla
  con la regla acordada en T0; §4.2 y §4.4 leen esta bandera.
- **Pruebas.** Cada sección ejecuta solo sus propias clases de prueba con un
  `unittest.TestSuite` explícito, como en §4.3, y usa nombres de clases que no se
  repitan entre secciones.

In [ ]:
import os
import platform
import sys
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
import scipy
import sklearn


def find_project_root(start):
    """Busca hacia arriba desde `start` la carpeta que contiene `JOURNAL.md` y `notebooks/`."""
    start = Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "JOURNAL.md").is_file() and (path / "notebooks").is_dir():
            return path
    raise FileNotFoundError(f"No se encontró la raíz del proyecto desde {start}")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
UNSW_DIR = DATA_RAW_DIR / "unsw_nb15"
RESULTS_FIGURES_DIR = PROJECT_ROOT / "results" / "figures"
RESULTS_TABLES_DIR = PROJECT_ROOT / "results" / "tables"

# Una semilla por sección; cada sección crea su propio generador a partir de la suya.
SEEDS = {
    "s1_perceptron": 1100,
    "s2_xor": 2100,
    "s3_r2": 3100,
    "s4_datos": 4000,
    "s4_1_l2": 4100,
    "s4_2_ensamble": 4200,
    "s4_3_ece": 4300,
    "s4_4_evaluacion": 4400,
}

# Modo de ejecución reducida para el desarrollo (TP3_QUICK=1). T8 se ejecuta sin él.
QUICK = os.environ.get("TP3_QUICK", "0") == "1"


def quick_or_full(quick_value, full_value):
    """Devuelve `quick_value` en ejecución reducida y `full_value` en la ejecución completa."""
    return quick_value if QUICK else full_value


# Decisión de §4.1 (T4, 2026-11-26): sin normalización L2. Con L2, el F1 de la clase ataque fue menor
# en las 30 corridas emparejadas (Wilcoxon p < 0.05); ver §4.1 y JOURNAL.md. §4.2 y §4.4 la usan.
USE_L2 = False

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Python {platform.python_version()} en {platform.platform()} ({platform.machine()})")
print(
    f"numpy {np.__version__} · pandas {pd.__version__} · scipy {scipy.__version__} · "
    f"scikit-learn {sklearn.__version__} · matplotlib {matplotlib.__version__}"
)
print(f"Ejecución reducida (TP3_QUICK): {QUICK}")
if sys.version_info[:3] != (3, 15, 0):
    print("Aviso: la versión acordada en T0 es Python 3.15.0 (ver requirements.txt).")

## 1. Perceptrón de tres capas

*Responsable: P1 (T1).*

### 1.1 Clase `Multilayer_perceptron`

La red tiene tres capas: $D$ neuronas de entrada, $M$ ocultas con tangente hiperbólica y $K$ de salida
con softmax. Sigue el contrato de la red acordado en T0 (`JOURNAL.md`), que también usan §3 y §4.1:

| Método | Qué hace |
| --- | --- |
| `__init__(neurons_per_layer, alpha, max_weights, gamma=0.0, seed=None)` | Guarda $[D, M, K]$, $\alpha$, $\gamma$ y `max_weights`; inicializa los pesos de forma uniforme en $[-\texttt{max\_weights}, \texttt{max\_weights}]$ con un generador propio creado con `seed`. Agrega `gamma`, que la lista de argumentos del enunciado pide aunque la firma impresa lo omite (D2). |
| `forward(X)` | Propagación hacia adelante de la matriz $X$ ($n \times D$, una observación por fila); devuelve la salida $n \times K$ y guarda los pesos netos y las salidas de cada capa. |
| `backpropagate_deltas(X, T)` | Calcula los deltas de las dos capas y, a partir de ellos, el gradiente del error de `get_error` respecto de cada matriz de pesos. |
| `update_weights(X, T)` | Actualiza ambas matrices con AdaGrad y momentum (D5) y recorta los pesos a $\pm$`max_weights`. |
| `train_mlp(num_epochs, X, T, alpha, gamma, X_val=None, T_val=None)` | Repite `update_weights` en lote completo y registra el error de entrenamiento (y de validación, si se pasa) después de cada época, para graficarlo. |
| `get_error(X, T)` | MSE: media de $(y - t)^2$ sobre las $n$ observaciones y las $K$ salidas. |
| `predict(X)` | Etiquetas enteras: argmax de la salida si $K \ge 2$; umbral 0.5 si $K = 1$. |

**Representación matricial.** Como en el material del curso, el sesgo es la fila 0 de cada matriz de
pesos y se agrega una entrada constante igual a 1 a cada capa: $W_o$ es $(D+1) \times M$ y $W_s$ es
$(M+1) \times K$. `T` es una matriz $n \times K$ de ceros y unos: one-hot si $K \ge 2$ y una columna con
1 = clase positiva si $K = 1$.

**Salida con $K = 1$ (D3).** Una softmax de una sola componente vale siempre 1 y su derivada es 0, de
modo que la red no aprendería. Con $K = 1$ la salida es la primera componente de
$\operatorname{softmax}([p, 0])$, que sigue siendo una softmax y es igual a la sigmoide:

$$\operatorname{softmax}([p, 0])_1 = \frac{e^{p}}{e^{p} + e^{0}} = \frac{1}{1 + e^{-p}} = \sigma(p).$$

El docente indicó que una salida igual a 1 con $K = 1$ también era aceptable; se eligió esta forma
porque permite entrenar y comparar las configuraciones de §3.

### 1.2 Propagación, deltas y AdaGrad con momentum

**Propagación hacia adelante.** Con $\tilde X = [\mathbf 1, X]$ ($n \times (D+1)$):

$$A_o = \tilde X W_o, \qquad Y_o = \tanh(A_o), \qquad \tilde Y_o = [\mathbf 1, Y_o], \qquad
A_s = \tilde Y_o W_s, \qquad Y_s = \operatorname{softmax}(A_s),$$

donde la softmax se aplica por fila: $y_{ik} = e^{a_{ik}} / \sum_j e^{a_{ij}}$. Antes de exponenciar se
resta el máximo de cada fila, lo que no cambia el resultado y evita desbordamientos.

**Error.** `get_error` es el MSE promediado sobre las $n$ observaciones y las $K$ salidas:

$$E = \frac{1}{nK} \sum_{i=1}^{n} \sum_{k=1}^{K} (y_{ik} - t_{ik})^2, \qquad
e_{ik} = \frac{\partial E}{\partial y_{ik}} = \frac{2}{nK}\,(y_{ik} - t_{ik}).$$

Los deltas de la clase calculan el gradiente exacto de este $E$, con el promedio y sin el factor
$\tfrac12$ que suele agregarse para simplificar la derivada. Así la prueba de gradiente numérico compara los deltas
directamente con `get_error`. AdaGrad divide cada paso entre la raíz de los gradientes acumulados, por
lo que un factor constante en el error casi no cambia el entrenamiento.

**Delta de salida con softmax y MSE.** Definimos $\delta^s_{ij} = \partial E / \partial a^s_{ij}$. En
una softmax, cada salida depende de todos los netos de su fila; el jacobiano es

$$\frac{\partial y_{ik}}{\partial a_{ij}} = y_{ik}\,(\mathbb 1[k = j] - y_{ij}),$$

y por la regla de la cadena

$$\delta^s_{ij} = \sum_{k=1}^{K} e_{ik}\, y_{ik}\,(\mathbb 1[k = j] - y_{ij})
= y_{ij}\Big(e_{ij} - \sum_{k=1}^{K} e_{ik}\, y_{ik}\Big).$$

Como $\sum_j y_{ij} = 1$, cada fila de $\delta^s$ suma cero; una de las pruebas lo verifica.

**Caso $K = 1$.** Con $y_i = \operatorname{softmax}([p_i, 0])_1 = \sigma(p_i)$, solo la primera
componente entra en el error. La fórmula anterior con $e = [e_i, 0]$ e $y = [y_i, 1 - y_i]$ da

$$\delta^s_i = y_i\,(e_i - e_i\, y_i) = e_i\, y_i\,(1 - y_i),$$

que tiene la forma del delta con sigmoide del material del curso.

**Delta de la capa oculta.** Se retropropaga por $W_s$ sin la fila del sesgo (la neurona constante no
depende de la capa anterior), y la derivada de la tangente hiperbólica es $1 - \tanh^2$:

$$\delta^o_{im} = \big(1 - (y^o_{im})^2\big) \sum_{k=1}^{K} \delta^s_{ik}\,(W_s)_{mk}, \qquad
\delta^o = (1 - Y_o \odot Y_o) \odot \big(\delta^s\, W_s[1{:}, :]^\top\big).$$

**Gradientes.** Como los deltas ya incluyen el factor $1/(nK)$, el gradiente del error medio es la
suma sobre las observaciones:

$$\nabla_{W_s} E = \tilde Y_o^\top\, \delta^s, \qquad \nabla_{W_o} E = \tilde X^\top\, \delta^o.$$

**AdaGrad con momentum (D5).** En cada época, con todo el lote y para cada matriz de pesos $W$ con
gradiente $g$:

$$G \leftarrow G + g \odot g, \qquad v \leftarrow \gamma\, v + \frac{\alpha\, g}{\sqrt{G} + \varepsilon},
\qquad W \leftarrow W - v,$$

y después $W$ se recorta a $[-\texttt{max\_weights}, \texttt{max\_weights}]$. Con $\gamma = 0$ queda AdaGrad
puro (Duchi et al., 2011). $G$ y $v$ empiezan en cero y se conservan entre llamadas a `train_mlp`. Se
usa $\varepsilon = 10^{-8}$, que solo evita dividir entre cero mientras la suma $G$ de un peso es 0 y es
despreciable frente a $\sqrt G$ en cuanto hay gradiente. Es del mismo orden que el $\delta \approx 10^{-7}$
de Goodfellow et al. (2016, algoritmo 8.4). En el primer paso $G = g^2$, de modo que cada peso se
mueve casi exactamente $\alpha \cdot \operatorname{sign}(g)$; una de las pruebas lo usa.

**Recorte y momentum.** El recorte actúa sobre $W$ después del paso y no modifica $v$. Si el gradiente
empuja un peso contra el límite durante varias épocas, el peso queda en $\pm$`max_weights`, pero $v$
sigue acumulando pasos en esa dirección: con $\gamma = 0.9$, hasta unas $1/(1 - \gamma) = 10$ veces el paso
de una época. Cuando el gradiente se invierte, $v$ tarda varias épocas en cambiar de signo y, mientras
tanto, el peso sigue en el límite. En un peso aislado con $\alpha = 0.1$, $\gamma = 0.9$ y 15 épocas de
empuje, el peso quedó 7 épocas más en el borde. Con $\gamma = 0$ no hay retraso, porque $v$ es solo el
paso actual.

Se conserva este comportamiento por dos razones: es el que fija el contrato de la red (D5: recorte
después de la actualización) y con él se obtuvieron los resultados de §2, §3 y §4.1. Una de las pruebas lo
fija, y §4.1 registra qué fracción de los pesos termina en el límite (revisión de P2 de T1a, acordado el


**Qué cambia respecto de la retropropagación vista en clase**

1. *Salida softmax.* El delta de salida usa el jacobiano completo de la softmax, porque cada salida
   depende de todos los netos de su fila. La derivada elemento a elemento $y(1 - y)$ de la sigmoide
   solo reaparece con $K = 1$.
2. *Capa oculta con tangente hiperbólica.* Su derivada es $1 - y^2$ en lugar de $y(1 - y)$.
3. *Error MSE promediado en lote completo.* El gradiente acumula las $n$ observaciones con el factor
   $2/(nK)$ y los pesos se actualizan una vez por época.
4. *Descenso del gradiente adaptativo.* El paso fijo $W \leftarrow W - \alpha \nabla E$ se reemplaza por
   un paso propio de cada peso, $\alpha / (\sqrt{G} + \varepsilon)$, que disminuye en los pesos con
   gradientes históricos grandes. El momentum $\gamma$ suma una fracción del paso anterior.
5. *Recorte.* Después de cada actualización los pesos se limitan a $\pm$`max_weights`.

**Referencias**

- Duchi, J., Hazan, E. y Singer, Y. (2011). Adaptive subgradient methods for online learning and
  stochastic optimization. *Journal of Machine Learning Research*, 12, 2121–2159.
- Goodfellow, I., Bengio, Y. y Courville, A. (2016). *Deep Learning*. MIT Press, sección 8.5.1.

In [ ]:
import numpy as np


class Multilayer_perceptron:
    """Perceptrón de tres capas (entrada, oculta y salida) entrenado con AdaGrad y momentum.

    Arquitectura `[D, M, K]`: D entradas, M neuronas ocultas con tangente hiperbólica
    y K salidas con softmax. Con K = 1 la salida es la primera componente de
    softmax([p, 0]), que equivale a sigmoide(p) (D3 en `JOURNAL.md`).

    Pesos en la representación matricial del curso, con el sesgo en la fila 0:
    `W_o` de forma (D + 1) × M (entrada → oculta) y `W_s` de forma (M + 1) × K
    (oculta → salida).

    Atributos que actualizan los métodos (valen None hasta la primera llamada):

    - `forward`: `X_b` (entrada con la columna de unos), `A_o` y `Y_o` (pesos netos
      y salidas de la capa oculta), `Y_o_b` (salidas ocultas con la columna de unos),
      `A_s` y `Y_s` (pesos netos y salidas de la capa de salida). Reflejan la última
      llamada, incluida la que hace `get_error`.
    - `backpropagate_deltas`: `delta_o` (n × M), `delta_s` (n × K) y los gradientes
      `grad_W_o` y `grad_W_s`, con la forma de sus pesos.
    - AdaGrad y momentum: `G_o`, `G_s` (suma de gradientes al cuadrado) y `v_o`,
      `v_s` (velocidad). Se conservan entre llamadas a `train_mlp`.
    """

    # ε de AdaGrad: evita dividir entre cero mientras la suma G de un peso es 0 (T1a).
    EPSILON = 1e-8

    def __init__(self, neurons_per_layer, alpha, max_weights, gamma=0.0, seed=None):
        """Crea la red e inicializa los pesos de forma uniforme en [−max_weights, max_weights].

        Parámetros
        ----------
        neurons_per_layer : secuencia de tres enteros positivos [D, M, K]
            Neuronas de entrada, de la capa oculta y de salida.
        alpha : float > 0
            Coeficiente de aprendizaje α.
        max_weights : float > 0
            Valor absoluto máximo de cualquier peso; se usa para inicializar y para
            recortar los pesos después de cada actualización.
        gamma : float en [0, 1)
            Coeficiente de momentum γ. Con 0 el entrenamiento es AdaGrad puro.
        seed : int o None
            Semilla del generador propio de la instancia (pesos iniciales).
        """
        capas = [int(n) for n in neurons_per_layer]
        if len(capas) != 3 or any(n != m for n, m in zip(capas, neurons_per_layer)) or min(capas) < 1:
            raise ValueError("neurons_per_layer debe ser [D, M, K] con tres enteros positivos")
        if not max_weights > 0:
            raise ValueError("max_weights debe ser positivo")
        self.neurons_per_layer = capas
        self.D, self.M, self.K = capas
        self.alpha, self.gamma = self._validar_alpha_gamma(alpha, gamma)
        self.max_weights = float(max_weights)
        self.rng = np.random.default_rng(seed)
        self.W_o = self.rng.uniform(-self.max_weights, self.max_weights, (self.D + 1, self.M))
        self.W_s = self.rng.uniform(-self.max_weights, self.max_weights, (self.M + 1, self.K))
        self.G_o, self.G_s = np.zeros_like(self.W_o), np.zeros_like(self.W_s)
        self.v_o, self.v_s = np.zeros_like(self.W_o), np.zeros_like(self.W_s)
        self.X_b = self.A_o = self.Y_o = self.Y_o_b = self.A_s = self.Y_s = None
        self.delta_o = self.delta_s = self.grad_W_o = self.grad_W_s = None

    @staticmethod
    def _validar_alpha_gamma(alpha, gamma):
        if not alpha > 0:
            raise ValueError("alpha debe ser positivo")
        if not 0 <= gamma < 1:
            raise ValueError("gamma debe estar en [0, 1)")
        return float(alpha), float(gamma)

    @staticmethod
    def _softmax(A):
        """Softmax por fila, estable numéricamente (resta el máximo de cada fila)."""
        E = np.exp(A - A.max(axis=1, keepdims=True))
        return E / E.sum(axis=1, keepdims=True)

    def _validar_X(self, X):
        X = np.asarray(X, dtype=float)
        if X.ndim != 2 or X.shape[1] != self.D:
            raise ValueError(f"X debe tener forma n × {self.D}; se recibió {X.shape}")
        return X

    def _validar_T(self, T, n):
        T = np.asarray(T, dtype=float)
        if T.shape != (n, self.K):
            raise ValueError(f"T debe tener forma {(n, self.K)}; se recibió {T.shape}")
        if not np.all((T == 0) | (T == 1)):
            raise ValueError("T debe contener solo 0 y 1")
        if self.K >= 2 and not np.all(T.sum(axis=1) == 1):
            raise ValueError("Con K >= 2, cada fila de T debe ser one-hot")
        return T

    def forward(self, X):
        """Propagación hacia adelante; devuelve la salida n × K y guarda netos y salidas.

        A_o = [1, X] W_o, Y_o = tanh(A_o), A_s = [1, Y_o] W_s y Y_s = softmax(A_s)
        por fila. Con K = 1, Y_s es la primera componente de softmax([A_s, 0]).
        """
        X = self._validar_X(X)
        unos = np.ones((X.shape[0], 1))
        self.X_b = np.hstack([unos, X])
        self.A_o = self.X_b @ self.W_o
        self.Y_o = np.tanh(self.A_o)
        self.Y_o_b = np.hstack([unos, self.Y_o])
        self.A_s = self.Y_o_b @ self.W_s
        if self.K == 1:
            self.Y_s = self._softmax(np.hstack([self.A_s, np.zeros_like(self.A_s)]))[:, :1]
        else:
            self.Y_s = self._softmax(self.A_s)
        return self.Y_s

    def backpropagate_deltas(self, X, T):
        """Calcula los deltas de ambas capas y los gradientes del MSE de `get_error`.

        Los deltas son derivadas del error respecto de los pesos netos:
        delta_s = ∂E/∂A_s (n × K) y delta_o = ∂E/∂A_o (n × M). Ejecuta `forward(X)`.
        Devuelve `(delta_o, delta_s)` y guarda además `grad_W_o` y `grad_W_s`.
        """
        Y = self.forward(X)
        T = self._validar_T(T, Y.shape[0])
        n = Y.shape[0]
        e = 2.0 * (Y - T) / (n * self.K)                      # ∂E/∂Y_s
        if self.K == 1:
            self.delta_s = e * Y * (1.0 - Y)                  # softmax([p, 0]) = sigmoide(p)
        else:
            self.delta_s = Y * (e - np.sum(e * Y, axis=1, keepdims=True))  # jacobiano de softmax
        self.delta_o = (1.0 - self.Y_o ** 2) * (self.delta_s @ self.W_s[1:].T)
        self.grad_W_s = self.Y_o_b.T @ self.delta_s
        self.grad_W_o = self.X_b.T @ self.delta_o
        return self.delta_o, self.delta_s

    def _paso_adagrad(self, W, G, v, grad):
        """Aplica en el lugar un paso de AdaGrad con momentum y el recorte a max_weights."""
        G += grad * grad
        v *= self.gamma
        v += self.alpha * grad / (np.sqrt(G) + self.EPSILON)
        W -= v
        np.clip(W, -self.max_weights, self.max_weights, out=W)

    def update_weights(self, X, T):
        """Actualiza los pesos de ambas capas a partir de los deltas, con AdaGrad y momentum.

        Para cada matriz W con gradiente g: G ← G + g ⊙ g; v ← γ·v + α·g / (√G + ε);
        W ← W − v; y luego W se recorta a [−max_weights, max_weights]. Los gradientes
        se calculan con `backpropagate_deltas(X, T)` antes de modificar los pesos.
        """
        self.backpropagate_deltas(X, T)
        self._paso_adagrad(self.W_o, self.G_o, self.v_o, self.grad_W_o)
        self._paso_adagrad(self.W_s, self.G_s, self.v_s, self.grad_W_s)

    def get_error(self, X, T):
        """MSE: media de (y − t)² sobre las n observaciones y las K salidas."""
        Y = self.forward(X)
        T = self._validar_T(T, Y.shape[0])
        return float(np.mean((Y - T) ** 2))

    def train_mlp(self, num_epochs, X, T, alpha, gamma, X_val=None, T_val=None):
        """Entrena `num_epochs` épocas en lote completo y devuelve el error por época.

        `alpha` y `gamma` reemplazan los valores guardados en la instancia. G y v se
        conservan entre llamadas. Cada valor del historial se mide después de la
        actualización de su época. Devuelve
        `{"train_error": [...], "val_error": [...]}`; `val_error` queda vacía si no
        se pasan `X_val` y `T_val`.
        """
        if int(num_epochs) != num_epochs or num_epochs < 1:
            raise ValueError("num_epochs debe ser un entero positivo")
        if (X_val is None) != (T_val is None):
            raise ValueError("X_val y T_val se pasan juntos")
        self.alpha, self.gamma = self._validar_alpha_gamma(alpha, gamma)
        X = self._validar_X(X)
        T = self._validar_T(T, X.shape[0])
        historial = {"train_error": [], "val_error": []}
        for _ in range(int(num_epochs)):
            self.update_weights(X, T)
            historial["train_error"].append(self.get_error(X, T))
            if X_val is not None:
                historial["val_error"].append(self.get_error(X_val, T_val))
        return historial

    def predict(self, X):
        """Etiquetas enteras de forma (n,): argmax si K >= 2; umbral 0.5 (y >= 0.5 → 1) si K = 1."""
        Y = self.forward(X)
        if self.K == 1:
            return (Y[:, 0] >= 0.5).astype(int)
        return np.argmax(Y, axis=1)

### 1.3 Pruebas unitarias de §1

Las pruebas usan redes pequeñas, valores calculables a mano y datos generados con
`SEEDS["s1_perceptron"]`, y no dependen de otras secciones. La prueba central de los deltas compara
el gradiente que sale de `backpropagate_deltas` con un gradiente numérico de `get_error` por
diferencias centrales, $(E(w + h) - E(w - h)) / 2h$ con $h = 10^{-6}$, peso por peso.

| Función | Prueba | Resultado esperado |
| --- | --- | --- |
| `__init__` | Red $[3, 5, 2]$ con `max_weights` = 0.5 | $W_o$ de $4 \times 5$, $W_s$ de $6 \times 2$, pesos en $[-0.5, 0.5]$, $G = v = 0$, $\gamma = 0$ por omisión |
| | Misma semilla y semilla distinta | Pesos iguales y distintos |
| | Capas, $\alpha$, `max_weights` o $\gamma$ inválidos | `ValueError` |
| `forward` | Red $[1, 1, 2]$ con pesos fijados y $x = 2$ | Netos, salidas y softmax calculados a mano |
| | Red $[4, 6, 3]$ con entradas grandes | Salida $10 \times 3$ finita, filas que suman 1 y formas de los atributos |
| | $K = 1$ | Salida igual a $\sigma(A_s)$ (D3) |
| | $X$ con columnas de más | `ValueError` |
| `backpropagate_deltas` | Gradiente numérico con $K = 3$ | Coincide con `grad_W_o` y `grad_W_s` (tolerancia relativa $10^{-5}$) |
| | Gradiente numérico con $K = 1$ | Coincide con `grad_W_o` y `grad_W_s` |
| | Formas de los deltas con softmax | $\delta^o$ de $n \times M$, $\delta^s$ de $n \times K$, filas de $\delta^s$ que suman 0 |
| `update_weights` | Primer paso con $\gamma = 0$ | $\Delta W = -\alpha g / (\lvert g\rvert + \varepsilon) \approx -\alpha \operatorname{sign}(g)$ |
| | Dos pasos con $\gamma = 0.9$ | $G = g_1^2 + g_2^2$ y $v_2 = \gamma v_1 + \alpha g_2 / (\sqrt{G} + \varepsilon)$ |
| | $\alpha = 5$ con `max_weights` = 0.3 | Ningún peso sale de $\pm 0.3$ y alguno queda en el borde |
| | Peso aislado empujado 15 épocas contra +1 y luego con el gradiente invertido | Con $\gamma = 0.9$, $v$ se acumula y el peso sigue en el límite 3 épocas después; con $\gamma = 0$ sale en el primer paso |
| `get_error` | Pesos en cero (salida uniforme $1/K$) | 0.25 con $K = 1$ y $K = 2$; $(0.75^2 + 3 \cdot 0.25^2)/4$ con $K = 4$ |
| | Datos aleatorios con $K = 3$ | $\sum (y - t)^2 / (nK)$ |
| | `T` de forma $(n,)$ o filas no one-hot | `ValueError` |
| `train_mlp` | Dos nubes separables, 100 épocas, con validación | Historiales de 100 valores, error final < 10 % del inicial y validación bien clasificada |
| | 10 épocas seguidas frente a 4 + 6 épocas | Mismos pesos y errores ($G$ y $v$ se conservan); `val_error` vacía sin validación |
| `predict` | $K = 1$ con $p = 5 \tanh(x)$ | Etiquetas enteras $[0, 0, 1, 1]$ para $x = -2, -0.1, 0, 0.3$ |
| | $K = 3$ con pesos fijados | Argmax $[0, 1, 2]$ |

Antes de integrar T1a se comprobó que las pruebas fallan si se introducen errores a propósito: delta de
salida sin el jacobiano, derivada de la sigmoide en la capa oculta, otra escala del error, sin momentum,
sin acumular $G$, sin recorte, retropropagación por la fila del sesgo y umbral estricto en `predict`.

In [ ]:
import sys
import unittest


def s1_datos_prueba(n, D, K, rng):
    """Datos aleatorios de prueba: X normal n × D y T one-hot (K >= 2) o 0/1 (K = 1)."""
    X = rng.normal(size=(n, D))
    if K == 1:
        return X, rng.integers(0, 2, size=(n, 1)).astype(float)
    return X, np.eye(K)[rng.integers(0, K, size=n)]


def s1_gradiente_numerico(red, W, X, T, h=1e-6):
    """Gradiente de `get_error` respecto de la matriz W (atributo de `red`) por diferencias centrales."""
    grad = np.zeros_like(W)
    for idx in np.ndindex(W.shape):
        original = W[idx]
        W[idx] = original + h
        e_mas = red.get_error(X, T)
        W[idx] = original - h
        e_menos = red.get_error(X, T)
        W[idx] = original
        grad[idx] = (e_mas - e_menos) / (2 * h)
    return grad


class TestMLPConstructor(unittest.TestCase):
    def test_formas_rango_y_estado_inicial(self):
        red = Multilayer_perceptron([3, 5, 2], alpha=0.1, max_weights=0.5, seed=1)
        self.assertEqual(red.W_o.shape, (4, 5))
        self.assertEqual(red.W_s.shape, (6, 2))
        for W in (red.W_o, red.W_s):
            self.assertTrue(np.all(np.abs(W) <= 0.5))
        for M in (red.G_o, red.G_s, red.v_o, red.v_s):
            self.assertFalse(M.any())
        self.assertEqual((red.alpha, red.gamma, red.max_weights), (0.1, 0.0, 0.5))

    def test_semilla_reproducible(self):
        a = Multilayer_perceptron([2, 4, 1], 0.1, 1.0, seed=7)
        b = Multilayer_perceptron([2, 4, 1], 0.1, 1.0, seed=7)
        c = Multilayer_perceptron([2, 4, 1], 0.1, 1.0, seed=8)
        np.testing.assert_array_equal(a.W_o, b.W_o)
        np.testing.assert_array_equal(a.W_s, b.W_s)
        self.assertFalse(np.array_equal(a.W_o, c.W_o))

    def test_rechaza_argumentos_invalidos(self):
        for args in (([2, 3], 0.1, 1.0), ([2, 0, 1], 0.1, 1.0), ([2, 3, 1], 0.0, 1.0),
                     ([2, 3, 1], 0.1, -1.0)):
            with self.assertRaises(ValueError):
                Multilayer_perceptron(*args)
        with self.assertRaises(ValueError):
            Multilayer_perceptron([2, 3, 1], 0.1, 1.0, gamma=1.0)


class TestMLPForward(unittest.TestCase):
    def test_calculo_a_mano(self):
        red = Multilayer_perceptron([1, 1, 2], 0.1, 10.0)
        red.W_o = np.array([[0.5], [1.0]])            # a_o = 0.5 + x
        red.W_s = np.array([[0.0, 1.0], [2.0, 0.0]])  # a_s = [2·y_o, 1]
        y_o = np.tanh(0.5 + 2.0)
        z = np.exp([2 * y_o, 1.0])
        Y = red.forward(np.array([[2.0]]))
        np.testing.assert_allclose(red.A_o, [[2.5]])
        np.testing.assert_allclose(red.Y_o, [[y_o]])
        np.testing.assert_allclose(red.A_s, [[2 * y_o, 1.0]])
        np.testing.assert_allclose(Y, [z / z.sum()])

    def test_salida_softmax_y_formas(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        red = Multilayer_perceptron([4, 6, 3], 0.1, 1.0, seed=SEEDS["s1_perceptron"])
        X = rng.normal(size=(10, 4)) * 50   # entradas grandes: softmax estable
        Y = red.forward(X)
        self.assertEqual(Y.shape, (10, 3))
        self.assertEqual((red.X_b.shape, red.Y_o_b.shape, red.A_s.shape), ((10, 5), (10, 7), (10, 3)))
        np.testing.assert_allclose(Y.sum(axis=1), np.ones(10))
        self.assertTrue(np.all((Y >= 0) & (Y <= 1)) and np.all(np.isfinite(Y)))

    def test_k1_equivale_a_sigmoide(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        red = Multilayer_perceptron([2, 3, 1], 0.1, 2.0, seed=SEEDS["s1_perceptron"])
        Y = red.forward(rng.normal(size=(8, 2)))
        self.assertEqual(Y.shape, (8, 1))
        np.testing.assert_allclose(Y, 1.0 / (1.0 + np.exp(-red.A_s)))

    def test_rechaza_columnas_incorrectas(self):
        red = Multilayer_perceptron([2, 3, 1], 0.1, 1.0)
        with self.assertRaises(ValueError):
            red.forward(np.zeros((4, 3)))


class TestMLPBackpropagateDeltas(unittest.TestCase):
    def _comprobar_gradiente(self, D, M, K):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        red = Multilayer_perceptron([D, M, K], 0.1, 1.0, seed=SEEDS["s1_perceptron"])
        X, T = s1_datos_prueba(7, D, K, rng)
        red.backpropagate_deltas(X, T)
        grad_o, grad_s = red.grad_W_o.copy(), red.grad_W_s.copy()
        np.testing.assert_allclose(grad_o, s1_gradiente_numerico(red, red.W_o, X, T), rtol=1e-5, atol=1e-9)
        np.testing.assert_allclose(grad_s, s1_gradiente_numerico(red, red.W_s, X, T), rtol=1e-5, atol=1e-9)

    def test_gradiente_numerico_softmax(self):
        self._comprobar_gradiente(D=3, M=4, K=3)

    def test_gradiente_numerico_k1(self):
        self._comprobar_gradiente(D=2, M=5, K=1)

    def test_formas_y_suma_cero_de_delta_salida(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        red = Multilayer_perceptron([3, 4, 3], 0.1, 1.0, seed=SEEDS["s1_perceptron"])
        X, T = s1_datos_prueba(6, 3, 3, rng)
        delta_o, delta_s = red.backpropagate_deltas(X, T)
        self.assertEqual((delta_o.shape, delta_s.shape), ((6, 4), (6, 3)))
        # Con softmax, sumar la misma constante a los netos de una fila no cambia la salida:
        np.testing.assert_allclose(delta_s.sum(axis=1), np.zeros(6), atol=1e-15)


class TestMLPUpdateWeights(unittest.TestCase):
    def test_primer_paso_es_alpha_por_signo(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        red = Multilayer_perceptron([2, 3, 2], alpha=0.05, max_weights=10.0, seed=SEEDS["s1_perceptron"])
        X, T = s1_datos_prueba(5, 2, 2, rng)
        W_o, W_s = red.W_o.copy(), red.W_s.copy()
        red.update_weights(X, T)
        for antes, despues, g in ((W_o, red.W_o, red.grad_W_o), (W_s, red.W_s, red.grad_W_s)):
            np.testing.assert_allclose(antes - despues, 0.05 * g / (np.abs(g) + Multilayer_perceptron.EPSILON))
            grande = np.abs(g) > 1e-4
            np.testing.assert_allclose((antes - despues)[grande], 0.05 * np.sign(g[grande]), rtol=1e-3)

    def test_momentum_y_acumulacion(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        red = Multilayer_perceptron([2, 3, 1], alpha=0.1, max_weights=10.0, gamma=0.9, seed=SEEDS["s1_perceptron"])
        X, T = s1_datos_prueba(6, 2, 1, rng)
        red.update_weights(X, T)
        g1, v1 = red.grad_W_s.copy(), red.v_s.copy()
        W_antes = red.W_s.copy()
        red.update_weights(X, T)
        g2 = red.grad_W_s
        v2 = 0.9 * v1 + 0.1 * g2 / (np.sqrt(g1 ** 2 + g2 ** 2) + Multilayer_perceptron.EPSILON)
        np.testing.assert_allclose(red.G_s, g1 ** 2 + g2 ** 2)
        np.testing.assert_allclose(red.v_s, v2)
        np.testing.assert_allclose(W_antes - red.W_s, v2)

    def test_recorte_a_max_weights(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        red = Multilayer_perceptron([2, 3, 2], alpha=5.0, max_weights=0.3, seed=SEEDS["s1_perceptron"])
        X, T = s1_datos_prueba(5, 2, 2, rng)
        red.update_weights(X, T)
        for W in (red.W_o, red.W_s):
            self.assertTrue(np.all(np.abs(W) <= 0.3))
        self.assertTrue(np.any(np.isclose(np.abs(red.W_s), 0.3)))
    def test_recorte_conserva_el_momentum(self):
        # Peso aislado: 15 pasos que lo empujan contra +max_weights y luego el gradiente se invierte.
        def trayectoria(gamma):
            red = Multilayer_perceptron([1, 1, 1], alpha=0.1, max_weights=1.0, gamma=gamma)
            W, G, v = np.array([0.9]), np.zeros(1), np.zeros(1)
            for g in [-1.0] * 15:
                red._paso_adagrad(W, G, v, np.array([g]))
            v_acumulada = -v[0]
            despues = []
            for g in [1.0] * 3:
                red._paso_adagrad(W, G, v, np.array([g]))
                despues.append(W[0])
            return v_acumulada, despues

        v_momentum, despues_momentum = trayectoria(0.9)
        self.assertGreater(v_momentum, 2 * 0.1)                  # v supera dos veces α (el paso máximo de una época)
        self.assertEqual(despues_momentum, [1.0, 1.0, 1.0])      # sigue en el límite tras invertirse g
        _, despues_sin_momentum = trayectoria(0.0)



class TestMLPGetError(unittest.TestCase):
    def test_pesos_cero_dan_salida_uniforme(self):
        X = np.ones((4, 2))
        for K, T, esperado in ((2, np.eye(2)[[0, 1, 1, 0]], 0.25), (1, np.array([[0.], [1.], [1.], [0.]]), 0.25),
                               (4, np.eye(4), ((0.75 ** 2) + 3 * 0.25 ** 2) / 4)):
            red = Multilayer_perceptron([2, 3, K], 0.1, 1.0)
            red.W_o[:] = 0.0
            red.W_s[:] = 0.0
            self.assertAlmostEqual(red.get_error(X, T), esperado)

    def test_coincide_con_la_definicion(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        red = Multilayer_perceptron([3, 4, 3], 0.1, 1.0, seed=SEEDS["s1_perceptron"])
        X, T = s1_datos_prueba(9, 3, 3, rng)
        Y = red.forward(X)
        self.assertAlmostEqual(red.get_error(X, T), np.sum((Y - T) ** 2) / (9 * 3))

    def test_rechaza_t_con_forma_incorrecta(self):
        red = Multilayer_perceptron([2, 3, 1], 0.1, 1.0)
        with self.assertRaises(ValueError):
            red.get_error(np.zeros((4, 2)), np.zeros(4))   # (n,) en lugar de (n, 1)
        red2 = Multilayer_perceptron([2, 3, 2], 0.1, 1.0)
        with self.assertRaises(ValueError):
            red2.get_error(np.zeros((2, 2)), np.array([[1., 1.], [0., 1.]]))  # no es one-hot


class TestMLPTrain(unittest.TestCase):
    @staticmethod
    def _dos_nubes(n, rng):
        """Dos nubes gaussianas separables en R², con etiquetas one-hot."""
        X = np.vstack([rng.normal(-1.5, 0.6, size=(n, 2)), rng.normal(1.5, 0.6, size=(n, 2))])
        T = np.eye(2)[np.repeat([0, 1], n)]
        return X, T

    def test_aprende_problema_separable(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        X, T = self._dos_nubes(40, rng)
        X_val, T_val = self._dos_nubes(20, rng)
        red = Multilayer_perceptron([2, 4, 2], 0.05, 2.0, seed=SEEDS["s1_perceptron"])
        error_inicial = red.get_error(X, T)
        historial = red.train_mlp(100, X, T, alpha=0.05, gamma=0.5, X_val=X_val, T_val=T_val)
        self.assertEqual((len(historial["train_error"]), len(historial["val_error"])), (100, 100))
        self.assertLess(historial["train_error"][-1], 0.1 * error_inicial)
        self.assertAlmostEqual(historial["train_error"][-1], red.get_error(X, T))
        np.testing.assert_array_equal(red.predict(X_val), np.argmax(T_val, axis=1))

    def test_estado_se_conserva_entre_llamadas(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        X, T = s1_datos_prueba(10, 2, 1, rng)
        continua = Multilayer_perceptron([2, 3, 1], 0.1, 1.0, seed=3)
        por_partes = Multilayer_perceptron([2, 3, 1], 0.1, 1.0, seed=3)
        h = continua.train_mlp(10, X, T, alpha=0.02, gamma=0.8)
        h1 = por_partes.train_mlp(4, X, T, alpha=0.02, gamma=0.8)
        h2 = por_partes.train_mlp(6, X, T, alpha=0.02, gamma=0.8)
        np.testing.assert_allclose(continua.W_o, por_partes.W_o)
        np.testing.assert_allclose(continua.W_s, por_partes.W_s)
        np.testing.assert_allclose(h["train_error"], h1["train_error"] + h2["train_error"])
        self.assertEqual(h["val_error"], [])
        self.assertEqual((por_partes.alpha, por_partes.gamma), (0.02, 0.8))


class TestMLPPredict(unittest.TestCase):
    def test_umbral_k1(self):
        red = Multilayer_perceptron([1, 1, 1], 0.1, 10.0)
        red.W_o = np.array([[0.0], [1.0]])   # y_o = tanh(x)
        red.W_s = np.array([[0.0], [5.0]])   # p = 5·tanh(x)
        etiquetas = red.predict(np.array([[-2.0], [-0.1], [0.0], [0.3]]))
        np.testing.assert_array_equal(etiquetas, [0, 0, 1, 1])
        self.assertEqual(etiquetas.dtype.kind, "i")

    def test_argmax_k3(self):
        red = Multilayer_perceptron([2, 2, 3], 0.1, 10.0)
        red.W_o = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]])
        red.W_s = np.array([[0.0, 0.0, 0.5], [3.0, 0.0, 0.0], [0.0, 3.0, 0.0]])
        X = np.array([[2.0, 0.0], [0.0, 2.0], [0.0, 0.0]])
        np.testing.assert_array_equal(red.predict(X), [0, 1, 2])


suite_s1 = unittest.TestSuite()
for caso in (TestMLPConstructor, TestMLPForward, TestMLPBackpropagateDeltas, TestMLPUpdateWeights,
             TestMLPGetError, TestMLPTrain, TestMLPPredict):
    suite_s1.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(caso))
resultado_s1 = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite_s1)
assert resultado_s1.wasSuccessful(), "Fallaron pruebas unitarias de §1"

#### Captura de la prueba de entrenamiento

El enunciado (§1 d.1) pide un pantallazo de una prueba unitaria de `train_mlp`. Se toma en la ejecución
final del notebook (T8), con el código entregado: la salida de la celda anterior, con las líneas de
`TestMLPTrain`, se adjunta aquí como imagen de esta celda Markdown.

*Pendiente de T8.*

## 2. Clasificación de XOR

*Responsable: P1 (T1).*

### 2.1 Planteamiento

Las cuatro entradas booleanas $\vec x \in \{0, 1\}^2$ y su salida esperada forman `s2_X` y `s2_T`:

| $x_1$ | $x_2$ | $x_1 \oplus x_2$ | Fila de `s2_T` (one-hot) |
| --- | --- | --- | --- |
| 0 | 0 | 0 | $[1, 0]$ |
| 0 | 1 | 1 | $[0, 1]$ |
| 1 | 0 | 1 | $[0, 1]$ |
| 1 | 1 | 0 | $[1, 0]$ |

Ninguna recta separa las entradas con salida 1, $(0, 1)$ y $(1, 0)$, de las de salida 0, $(0, 0)$ y
$(1, 1)$: están en las diagonales opuestas del cuadrado. Por eso un clasificador lineal no resuelve XOR
y hace falta la capa oculta. Sus neuronas con tangente hiperbólica transforman las entradas en un
espacio donde las dos clases sí son separables por la capa de salida.

**Decisiones de T1b** (registradas en `JOURNAL.md`, D5):

- *Salida:* $K = 2$ neuronas con softmax y etiquetas one-hot (clase 0 = XOR falso, clase 1 = XOR
  verdadero). Es la softmax de §1 tal cual, sin el caso especial $K = 1$ de §3 (D3), y la predicción es
  el argmax.
- *«Error cero».* Con softmax las salidas nunca valen exactamente 0 o 1, así que el MSE nunca llega a
  0. Además, el recorte a `max_weights` limita cuánto puede acercarse. Se distinguen dos medidas:
  - el **error de clasificación**, el número de entradas que `predict` clasifica mal, que sí llega a 0;
  - el **MSE**, que mide cuán cerca están las salidas de los objetivos.

  La red resuelve XOR cuando no clasifica mal ninguna de las cuatro entradas y el MSE es
  $\le 0.01$, es decir, salidas a unos 0.1 de su objetivo.
- *Convergencia:* la época de convergencia es la primera a partir de la cual ese criterio se cumple
  **hasta la última época**. Una corrida que lo cumple y luego lo pierde no cuenta como convergida en
  ese momento.
- *Sensibilidad a la semilla:* con pocas neuronas ocultas, la convergencia depende de los pesos
  iniciales. Por eso cada configuración se entrena con 20 semillas y se reporta la proporción que
  converge, no una sola corrida.

Todo el entrenamiento de esta sección usa `train_mlp` de §1, en lote completo con las cuatro
observaciones.

### 2.2 Funciones auxiliares y sus pruebas

| Función | Qué hace |
| --- | --- |
| `s2_entrenar_xor(M, alpha, gamma, max_weights, seed, num_epochs)` | Entrena una red $[2, M, 2]$ con `train_mlp`, una época a la vez, y registra el MSE y el número de entradas mal clasificadas después de cada época. |
| `s2_epoca_convergencia(mse, errores, umbral_mse)` | Devuelve la primera época (contada desde 1) a partir de la cual, hasta el final, no hay errores de clasificación y el MSE es $\le$ `umbral_mse`; `None` si la corrida no converge. |
| `s2_dibujar_red(D, M, K, ax)` | Dibuja el diagrama de la red con sus neuronas de sesgo y una línea por peso. |

| Función | Prueba | Resultado esperado |
| --- | --- | --- |
| `s2_epoca_convergencia` | MSE $[0.3, 0.1, 0.05, 0.009, 0.005]$, errores $[2, 1, 0, 0, 0]$ | Época 4 |
| | Cumple en la época 2, falla en la 3 y cumple desde la 4; cumple desde el inicio | Épocas 4 y 1 |
| | El MSE sube del umbral al final, o queda un error con MSE bajo | `None` |
| | Longitudes distintas | `ValueError` |
| `s2_entrenar_xor` | 30 épocas, una a la vez, frente a `train_mlp(30, …)` | Mismo MSE por época, mismos pesos y mismos errores al final |
| | Dos corridas con la misma semilla | MSE idénticos, formas $(12,)$ y errores entre 0 y 4 |
| `s2_dibujar_red` | Red $[2, 4, 2]$ | $3 + 5 + 2 = 10$ neuronas, contando los sesgos |
| | Redes $[2, 4, 2]$ y $[3, 2, 1]$ | Una línea por peso: $3 \cdot 4 + 5 \cdot 2 = 22$ y $4 \cdot 2 + 3 \cdot 1 = 11$ |

In [ ]:
import itertools

import matplotlib.pyplot as plt
from matplotlib.patches import Circle

# Tabla de verdad del XOR: una observación por fila; clase 1 = XOR verdadero.
s2_X = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
s2_y = np.array([0, 1, 1, 0])
s2_T = np.eye(2)[s2_y]            # one-hot, K = 2
s2_umbral_mse = 0.01              # criterio de convergencia acordado en T1b (D5)
s2_colores = {0.0: "#2a78d6", 0.9: "#eb6834"}   # γ = 0 y γ = 0.9


def s2_entrenar_xor(M, alpha, gamma, max_weights, seed, num_epochs, X=s2_X, T=s2_T):
    """Entrena una red [D, M, K] sobre (X, T) y registra, por época, el MSE y los errores de clasificación.

    Llama a `train_mlp` una época a la vez (G y v se conservan entre llamadas, así que equivale a
    entrenar `num_epochs` seguidas) para contar después de cada época cuántas observaciones
    clasifica mal `predict`.

    Retorna
    -------
    dict con 'red' (la red entrenada), 'mse' (forma (num_epochs,)) y 'errores'
    (número de observaciones mal clasificadas, forma (num_epochs,)).
    """
    red = Multilayer_perceptron([X.shape[1], M, T.shape[1]], alpha, max_weights, gamma=gamma, seed=seed)
    etiquetas = np.argmax(T, axis=1)
    mse = np.empty(num_epochs)
    errores = np.empty(num_epochs, dtype=int)
    for epoca in range(num_epochs):
        mse[epoca] = red.train_mlp(1, X, T, alpha, gamma)["train_error"][0]
        errores[epoca] = int(np.sum(red.predict(X) != etiquetas))
    return {"red": red, "mse": mse, "errores": errores}


def s2_epoca_convergencia(mse, errores, umbral_mse=s2_umbral_mse):
    """Época de convergencia (contada desde 1) según el criterio de T1b.

    Es la primera época a partir de la cual, y hasta la última, no hay errores de
    clasificación y el MSE es menor o igual que `umbral_mse`. Devuelve None si el
    criterio no se cumple en la última época (la corrida no converge).
    """
    mse = np.asarray(mse, dtype=float)
    errores = np.asarray(errores)
    if mse.shape != errores.shape or mse.ndim != 1:
        raise ValueError("mse y errores deben ser vectores de la misma longitud")
    cumple = (errores == 0) & (mse <= umbral_mse)
    if cumple.size == 0 or not cumple[-1]:
        return None
    fallas = np.flatnonzero(~cumple)
    return int(fallas[-1]) + 2 if fallas.size else 1


def s2_dibujar_red(D, M, K, ax=None):
    """Dibuja el diagrama de un perceptrón [D, M, K] con las neuronas de sesgo (valor 1).

    Cada capa es una columna de círculos; las líneas son los pesos de `W_o` y `W_s`. Las
    neuronas de sesgo no reciben conexiones. Devuelve el `Axes` usado.
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4))
    tinta, gris = "#0b0b0b", "#8a8984"
    capas = [
        ["1"] + [f"$x_{i}$" for i in range(1, D + 1)],
        ["1"] + [f"$y^o_{{{m}}}$" for m in range(1, M + 1)],
        [f"$y^s_{{{k}}}$" for k in range(1, K + 1)],
    ]
    rellenos = ["#e8e7e2", "#2a78d6", "#eb6834"]
    alto = max(len(c) for c in capas)
    sep = 3.0  # distancia horizontal entre capas
    posiciones = []
    for columna, nombres in enumerate(capas):
        ys = np.linspace(alto - 1, 0, len(nombres)) if len(nombres) > 1 else np.array([(alto - 1) / 2])
        posiciones.append([(columna * sep, y) for y in ys])
    for origen, destino, salta_sesgo in ((0, 1, True), (1, 2, False)):
        for (x0, y0) in posiciones[origen]:
            for j, (x1, y1) in enumerate(posiciones[destino]):
                if salta_sesgo and j == 0:
                    continue  # la neurona de sesgo de la capa oculta no recibe pesos
                ax.plot([x0, x1], [y0, y1], color=gris, linewidth=0.8, zorder=1)
    for columna, nombres in enumerate(capas):
        for i, ((x, y), nombre) in enumerate(zip(posiciones[columna], nombres)):
            es_sesgo = nombre == "1"
            ax.add_patch(Circle((x, y), 0.32, facecolor="white" if es_sesgo else rellenos[columna],
                                edgecolor=tinta if es_sesgo else "white", linewidth=1.5, zorder=2))
            ax.text(x, y, nombre, ha="center", va="center", zorder=3, fontsize=10,
                    color=tinta if es_sesgo or columna == 0 else "white")
    for columna, titulo in enumerate([f"Entrada\n(D = {D})", f"Oculta, tanh\n(M = {M})", f"Salida, softmax\n(K = {K})"]):
        ax.text(columna * sep, alto - 0.45, titulo, ha="center", va="bottom", fontsize=10, color=tinta)
    ax.text(0.5 * sep, -0.9, "$W_o$", ha="center", color=tinta)
    ax.text(1.5 * sep, -0.9, "$W_s$", ha="center", color=tinta)
    ax.set_xlim(-0.8, 2 * sep + 0.8)
    ax.set_ylim(-1.2, alto + 0.7)
    ax.set_aspect("equal")
    ax.axis("off")
    return ax

In [ ]:
import sys
import unittest


class TestXorEpocaConvergencia(unittest.TestCase):
    def test_calculo_a_mano(self):
        mse = [0.30, 0.10, 0.05, 0.009, 0.005]
        errores = [2, 1, 0, 0, 0]
        self.assertEqual(s2_epoca_convergencia(mse, errores, umbral_mse=0.01), 4)

    def test_cuenta_desde_la_ultima_falla(self):
        # Cumple en la época 2, falla en la 3 y vuelve a cumplir desde la 4.
        self.assertEqual(s2_epoca_convergencia([0.2, 0.005, 0.02, 0.004], [1, 0, 0, 0], 0.01), 4)
        self.assertEqual(s2_epoca_convergencia([0.001, 0.001], [0, 0], 0.01), 1)

    def test_sin_convergencia(self):
        self.assertIsNone(s2_epoca_convergencia([0.005, 0.005, 0.02], [0, 0, 0], 0.01))
        self.assertIsNone(s2_epoca_convergencia([0.001, 0.001], [0, 1], 0.01))  # MSE bajo, un error

    def test_rechaza_longitudes_distintas(self):
        with self.assertRaises(ValueError):
            s2_epoca_convergencia([0.1, 0.2], [0])


class TestXorEntrenamiento(unittest.TestCase):
    def test_equivale_a_train_mlp_seguido(self):
        r = s2_entrenar_xor(M=3, alpha=0.1, gamma=0.5, max_weights=2.0, seed=SEEDS["s2_xor"], num_epochs=30)
        red = Multilayer_perceptron([2, 3, 2], 0.1, 2.0, gamma=0.5, seed=SEEDS["s2_xor"])
        historial = red.train_mlp(30, s2_X, s2_T, 0.1, 0.5)
        np.testing.assert_allclose(r["mse"], historial["train_error"])
        np.testing.assert_allclose(r["red"].W_o, red.W_o)
        self.assertEqual(r["errores"][-1], int(np.sum(red.predict(s2_X) != s2_y)))

    def test_formas_y_reproducibilidad(self):
        a = s2_entrenar_xor(2, 0.1, 0.0, 1.0, seed=5, num_epochs=12)
        b = s2_entrenar_xor(2, 0.1, 0.0, 1.0, seed=5, num_epochs=12)
        self.assertEqual((a["mse"].shape, a["errores"].shape), ((12,), (12,)))
        np.testing.assert_array_equal(a["mse"], b["mse"])
        self.assertTrue(np.all((a["errores"] >= 0) & (a["errores"] <= 4)))


class TestXorDiagrama(unittest.TestCase):
    def _contar(self, D, M, K):
        fig, ax = plt.subplots()
        s2_dibujar_red(D, M, K, ax)
        circulos = sum(isinstance(p, Circle) for p in ax.patches)
        lineas = len(ax.lines)
        plt.close(fig)
        return circulos, lineas

    def test_neuronas_incluyen_sesgos(self):
        self.assertEqual(self._contar(2, 4, 2)[0], (2 + 1) + (4 + 1) + 2)

    def test_un_trazo_por_peso(self):
        # Tantas líneas como pesos: (D + 1)·M en W_o y (M + 1)·K en W_s.
        self.assertEqual(self._contar(2, 4, 2)[1], 3 * 4 + 5 * 2)
        self.assertEqual(self._contar(3, 2, 1)[1], 4 * 2 + 3 * 1)


suite_s2 = unittest.TestSuite()
for caso in (TestXorEpocaConvergencia, TestXorEntrenamiento, TestXorDiagrama):
    suite_s2.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(caso))
resultado_s2 = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite_s2)
assert resultado_s2.wasSuccessful(), "Fallaron pruebas unitarias de §2"

### 2.3 Calibración de hiperparámetros

Se prueban todas las combinaciones de $M \in \{2, 4\}$ neuronas ocultas, `max_weights` $\in \{1, 2, 5\}$,
$\alpha \in \{0.01, 0.1, 0.5\}$ y $\gamma \in \{0, 0.9\}$: 36 configuraciones. Cada una se entrena 2000
épocas con 20 semillas, derivadas de `SEEDS["s2_xor"]` y comunes a todas las configuraciones.
`max_weights` es a la vez el rango de la inicialización uniforme y el límite del recorte. En ejecución
reducida (`TP3_QUICK=1`) se usan 5 semillas y 500 épocas.

Para cada configuración, la tabla muestra:

- cuántas semillas convergen según el criterio de 2.1;
- la mediana de la época de convergencia entre las que convergen;
- la mediana del MSE en la última época.

Se elige la configuración con mayor tasa de convergencia. A igual tasa, gana la que converge en menos
épocas (mediana) y después la que tiene menos neuronas ocultas. La figura siguiente muestra las curvas de
aprendizaje de todas las configuraciones probadas.

In [ ]:
import pandas as pd

# Rejilla de hiperparámetros: M, max_weights, α y γ (36 configuraciones).
s2_valores_M = [2, 4]
s2_valores_max_weights = [1.0, 2.0, 5.0]
s2_valores_alpha = [0.01, 0.1, 0.5]
s2_valores_gamma = [0.0, 0.9]
s2_num_semillas = quick_or_full(5, 20)
s2_num_epocas = quick_or_full(500, 2000)
s2_semillas = np.random.default_rng(SEEDS["s2_xor"]).integers(0, 2**31 - 1, size=s2_num_semillas)

s2_curvas = {}        # (M, max_weights, alpha, gamma) -> matriz semillas × épocas con el MSE
s2_filas = []
for s2_config in itertools.product(s2_valores_M, s2_valores_max_weights, s2_valores_alpha, s2_valores_gamma):
    s2_M, s2_mw, s2_alpha, s2_gamma = s2_config
    s2_mse, s2_epocas_conv = [], []
    for s2_semilla in s2_semillas:
        s2_r = s2_entrenar_xor(s2_M, s2_alpha, s2_gamma, s2_mw, int(s2_semilla), s2_num_epocas)
        s2_mse.append(s2_r["mse"])
        s2_epocas_conv.append(s2_epoca_convergencia(s2_r["mse"], s2_r["errores"]))
    s2_curvas[s2_config] = np.array(s2_mse)
    s2_convergidas = [e for e in s2_epocas_conv if e is not None]
    s2_filas.append({
        "M": s2_M, "max_weights": s2_mw, "alpha": s2_alpha, "gamma": s2_gamma,
        "convergen": len(s2_convergidas),
        "tasa_convergencia": len(s2_convergidas) / s2_num_semillas,
        "epoca_mediana": float(np.median(s2_convergidas)) if s2_convergidas else np.nan,
        "mse_final_mediano": float(np.median(s2_curvas[s2_config][:, -1])),
        "epocas_conv": s2_epocas_conv,
    })

# Orden de selección: mayor tasa de convergencia; a igual tasa, menos épocas (mediana) y menos neuronas.
s2_resumen = (pd.DataFrame(s2_filas)
              .sort_values(["tasa_convergencia", "epoca_mediana", "M"], ascending=[False, True, True])
              .reset_index(drop=True))
s2_elegida = s2_resumen.iloc[0]
s2_config_elegida = (int(s2_elegida["M"]), float(s2_elegida["max_weights"]),
                     float(s2_elegida["alpha"]), float(s2_elegida["gamma"]))

print(f"{len(s2_filas)} configuraciones × {s2_num_semillas} semillas × {s2_num_epocas} épocas; "
      f"criterio: 0 errores de clasificación y MSE ≤ {s2_umbral_mse} hasta la última época.")
print("Configuración elegida: M = {}, max_weights = {}, α = {}, γ = {}".format(*s2_config_elegida))
s2_resumen.drop(columns="epocas_conv").style.format(
    {"tasa_convergencia": "{:.0%}", "epoca_mediana": "{:.0f}", "mse_final_mediano": "{:.1e}",
     "max_weights": "{:g}", "alpha": "{:g}", "gamma": "{:g}"}, na_rep="—").hide(axis="index")

In [ ]:
# Curvas de aprendizaje de todas las configuraciones probadas: una fila por (M, max_weights) y una
# columna por α; en cada panel, la mediana del MSE sobre las semillas (línea) y el rango
# intercuartílico (banda) para γ = 0 y γ = 0.9. Ambos ejes en escala logarítmica.
s2_filas_panel = list(itertools.product(s2_valores_M, s2_valores_max_weights))
s2_fig, s2_ejes = plt.subplots(len(s2_filas_panel), len(s2_valores_alpha), figsize=(11, 15),
                               sharex=True, sharey=True, constrained_layout=True)
s2_epocas = np.arange(1, s2_num_epocas + 1)
for s2_i, (s2_M, s2_mw) in enumerate(s2_filas_panel):
    for s2_j, s2_alpha in enumerate(s2_valores_alpha):
        s2_ax = s2_ejes[s2_i, s2_j]
        for s2_gamma in s2_valores_gamma:
            s2_c = s2_curvas[(s2_M, s2_mw, s2_alpha, s2_gamma)]
            s2_q1, s2_med, s2_q3 = np.percentile(s2_c, [25, 50, 75], axis=0)
            s2_tasa = s2_resumen.query("M == @s2_M and max_weights == @s2_mw and alpha == @s2_alpha "
                                       "and gamma == @s2_gamma")["convergen"].iloc[0]
            s2_ax.fill_between(s2_epocas, s2_q1, s2_q3, color=s2_colores[s2_gamma], alpha=0.15, linewidth=0)
            s2_ax.plot(s2_epocas, s2_med, color=s2_colores[s2_gamma], linewidth=1.8,
                       label=f"γ = {s2_gamma:g} ({s2_tasa}/{s2_num_semillas} convergen)")
        s2_ax.axhline(s2_umbral_mse, color="#8a8984", linewidth=1, linestyle="--")
        s2_ax.set_xscale("log")
        s2_ax.set_yscale("log")
        s2_ax.grid(True, which="major", color="#e8e7e2", linewidth=0.8)
        s2_ax.spines[["top", "right"]].set_visible(False)
        s2_ax.legend(loc="lower left", fontsize=7.5, frameon=False)
        if s2_i == 0:
            s2_ax.set_title(f"α = {s2_alpha:g}", fontsize=11)
        if s2_j == 0:
            s2_ax.set_ylabel(f"M = {s2_M}, max_weights = {s2_mw:g}\nMSE de entrenamiento", fontsize=9)
        if s2_i == len(s2_filas_panel) - 1:
            s2_ax.set_xlabel("Época")
s2_fig.suptitle("XOR: MSE por época (mediana y rango intercuartílico sobre las semillas; "
                f"línea discontinua: umbral {s2_umbral_mse})", fontsize=11)
plt.show()

### 2.4 Arquitectura elegida

La red para XOR tiene:

- $D = 2$ neuronas de entrada, una por componente booleana;
- $M$ neuronas ocultas con tangente hiperbólica, con el valor de la configuración elegida en 2.3;
- $K = 2$ neuronas de salida con softmax, una por clase.

Cada capa, salvo la de salida, agrega una neurona de sesgo de valor 1 que corresponde a la fila 0 de
$W_o$ ($3 \times M$) y de $W_s$ ($(M + 1) \times 2$).

In [ ]:
s2_M_elegida = s2_config_elegida[0]
print(f"Arquitectura elegida: D = 2, M = {s2_M_elegida}, K = 2")
s2_fig, s2_ax = plt.subplots(figsize=(7, 4.5))
s2_dibujar_red(2, s2_M_elegida, 2, s2_ax)
s2_ax.set_title("Perceptrón de tres capas para XOR (los círculos «1» son las neuronas de sesgo)", fontsize=10)
plt.show()

### 2.5 Convergencia y predicciones

Con la configuración elegida se vuelve a entrenar la corrida representativa: entre las semillas que
convergen, la de época de convergencia mediana. Las gráficas muestran el MSE y el número de entradas
mal clasificadas en función de la época. La línea punteada marca la época de convergencia. La tabla
final da las salidas de la softmax y la predicción para las cuatro entradas.

In [ ]:
# Corrida representativa de la configuración elegida: entre las semillas que convergen, la de
# época de convergencia mediana (la primera en el orden de las semillas si hay empate).
s2_M, s2_mw, s2_alpha, s2_gamma = s2_config_elegida
s2_conv_elegida = s2_elegida["epocas_conv"]
s2_indices = [i for i, e in enumerate(s2_conv_elegida) if e is not None]
if not s2_indices:
    raise RuntimeError("Ninguna semilla de la configuración elegida converge")
s2_mediana = np.median([s2_conv_elegida[i] for i in s2_indices])
s2_idx = min(s2_indices, key=lambda i: abs(s2_conv_elegida[i] - s2_mediana))
s2_semilla_rep = int(s2_semillas[s2_idx])
s2_rep = s2_entrenar_xor(s2_M, s2_alpha, s2_gamma, s2_mw, s2_semilla_rep, s2_num_epocas)
s2_epoca_rep = s2_epoca_convergencia(s2_rep["mse"], s2_rep["errores"])

print(f"Semilla representativa: {s2_semilla_rep} (posición {s2_idx} de s2_semillas)")
print(f"Época de convergencia: {s2_epoca_rep} "
      f"(mediana de la configuración: {s2_mediana:g}; {len(s2_indices)}/{s2_num_semillas} semillas convergen)")
print(f"MSE en la época de convergencia: {s2_rep['mse'][s2_epoca_rep - 1]:.2e}; "
      f"MSE final (época {s2_num_epocas}): {s2_rep['mse'][-1]:.2e}")

s2_fig, (s2_ax_mse, s2_ax_err) = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
s2_epocas = np.arange(1, s2_num_epocas + 1)
s2_ax_mse.plot(s2_epocas, s2_rep["mse"], color=s2_colores.get(s2_gamma, "#2a78d6"), linewidth=2)
s2_ax_mse.axhline(s2_umbral_mse, color="#8a8984", linewidth=1, linestyle="--")
s2_ax_mse.set_yscale("log")
s2_ax_mse.set_ylabel("MSE de entrenamiento")
s2_ax_err.step(s2_epocas, s2_rep["errores"], where="post", color=s2_colores.get(s2_gamma, "#2a78d6"), linewidth=2)
s2_ax_err.set_ylim(-0.2, 4.2)
s2_ax_err.set_yticks(range(5))
s2_ax_err.set_ylabel("Observaciones mal clasificadas (de 4)")
for s2_ax in (s2_ax_mse, s2_ax_err):
    s2_ax.axvline(s2_epoca_rep, color="#0b0b0b", linewidth=1, linestyle=":")
    s2_ax.set_xscale("log")
    s2_ax.set_xlabel("Época")
    s2_ax.grid(True, color="#e8e7e2", linewidth=0.8)
    s2_ax.spines[["top", "right"]].set_visible(False)
s2_ax_mse.annotate(f"convergencia: época {s2_epoca_rep}", xy=(s2_epoca_rep, s2_umbral_mse),
                   xytext=(8, 12), textcoords="offset points", fontsize=9)
s2_fig.suptitle(f"XOR, configuración elegida (M = {s2_M}, max_weights = {s2_mw:g}, α = {s2_alpha:g}, "
                f"γ = {s2_gamma:g}); línea discontinua: umbral {s2_umbral_mse}", fontsize=10)
plt.show()

# Predicciones de la red entrenada para las cuatro entradas.
s2_salidas = s2_rep["red"].forward(s2_X)
pd.DataFrame({
    "x1": s2_X[:, 0].astype(int), "x2": s2_X[:, 1].astype(int), "XOR (t)": s2_y,
    "y_0 = P(clase 0)": s2_salidas[:, 0], "y_1 = P(clase 1)": s2_salidas[:, 1],
    "predicción": s2_rep["red"].predict(s2_X),
}).style.format({"y_0 = P(clase 0)": "{:.4f}", "y_1 = P(clase 1)": "{:.4f}"}).hide(axis="index")

### 2.6 Análisis

*Pendiente de T8.* El comentario que cite números (tasas de convergencia, épocas y MSE) se redacta con
la ejecución final del notebook (ver «Reglas de congruencia de las salidas» en `JOURNAL.md`). Debe
cubrir:

- el efecto de $M$ en la proporción de semillas que convergen;
- el de `max_weights` en el MSE alcanzable;
- el de $\alpha$ y $\gamma$ en la velocidad y la estabilidad;
- la cantidad de iteraciones necesarias para la convergencia en la configuración elegida.

## 3. Clasificador de datos en R²

*Responsable: P2 (T2a y T2b).*

### 3.1 Conjuntos de datos en R²

El enunciado pide la función `generar_datos_R2`, que no se entregó; el docente autorizó generar los
datos con scikit-learn (D4 en `JOURNAL.md`). Los dos conjuntos se representan de forma matricial: $X$ es
$n \times 2$, con una observación por fila, y $T$ es $n \times 1$, con 1 = clase positiva, que es la forma
que espera la red de §1 con $K = 1$ (contrato de T0).

| Escenario | Función | Parámetros | Motivo |
| --- | --- | --- | --- |
| Linealmente separable | `sklearn.datasets.make_blobs` | `n_samples=300`, `centers=[(-2, -2), (2, 2)]`, `cluster_std=0.7`, `random_state=SEEDS["s3_r2"]` | Dos nubes gaussianas isotrópicas de 150 puntos. La distancia entre los centros es $4\sqrt{2} \approx 5.7$, unas 8 desviaciones estándar, así que una recta puede separarlas. |
| No linealmente separable | `sklearn.datasets.make_moons` | `n_samples=300`, `noise=0.15`, `random_state=SEEDS["s3_r2"]` | Dos medias lunas entrelazadas de 150 puntos; ninguna recta las separa, ni siquiera sin ruido. El ruido gaussiano ($\sigma = 0.15$) agrega algo de traslape. |

**La separabilidad se comprueba, no se supone.** Un conjunto es linealmente separable si existen
$\vec{w} \in \mathbb{R}^2$ y $b$ tales que $s_i(\vec{w} \cdot \vec{x}_i + b) \ge 1$ para todas las
observaciones, con $s_i = +1$ en la clase 1 y $s_i = -1$ en la clase 0. Si una recta separa
estrictamente las clases, basta con escalar $(\vec{w}, b)$ para que el margen sea al menos 1, por lo que
el sistema es factible exactamente cuando los datos son separables. `s3_es_linealmente_separable` resuelve
ese programa lineal (objetivo nulo) con `scipy.optimize.linprog`.

**Partición.** `train_test_split` de scikit-learn, como pide el enunciado: 80 % entrenamiento y 20 %
validación, estratificada por clase y con `random_state=SEEDS["s3_r2"]`. Es la misma partición para las
cuatro redes de cada escenario, para que sus curvas sean comparables. Las entradas no se estandarizan:
ambos conjuntos ya están centrados cerca de 0 y en un rango de pocas unidades, donde la tangente
hiperbólica no se satura con los pesos iniciales.

#### Funciones de la sección y sus pruebas

| Función | Qué hace |
| --- | --- |
| `s3_generar_datos(tipo, seed, n_muestras=300)` | Genera el conjunto `"separable"` (`make_blobs`) o `"no_separable"` (`make_moons`) con los parámetros de la tabla anterior y devuelve `X` ($n \times 2$) y `T` ($n \times 1$, valores 0/1). |
| `s3_es_linealmente_separable(X, T)` | Plantea el programa lineal $s_i(\vec{w} \cdot \vec{x}_i + b) \ge 1$ y devuelve `True` si es factible, es decir, si una recta separa las dos clases. |
| `s3_particionar(X, T, seed, fraccion_validacion=0.2)` | Partición estratificada de scikit-learn; devuelve `(X_ent, X_val, T_ent, T_val)`. |
| `s3_entrenar_configuracion(M, gamma, max_weights, seed, particion, alpha=0.01, num_epocas=50)` | Crea una red $[2, M, 1]$ con la semilla dada, la entrena con `train_mlp` y devuelve la red, el MSE de entrenamiento y de validación por época y la tasa de aciertos final en validación. |
| `s3_ejecutar_experimento(particiones, configuraciones, semillas, max_weights, …)` | Entrena cada configuración en cada escenario con cada semilla de pesos iniciales y agrupa las curvas en matrices semillas × épocas. |
| `s3_tabla_errores_finales(resultados, configuraciones)` | Resume el MSE de la última época (mediana, media y desviación estándar muestral sobre las semillas) y la tasa de aciertos en validación, con una fila por escenario y configuración. |
| `s3_graficar_tabla_curvas(resultados, escenarios, configuraciones)` | Tabla de gráficas: una fila por configuración y una columna por escenario; en cada panel, la mediana del MSE de entrenamiento y de validación por época y su rango intercuartílico. |

Las pruebas usan datos pequeños construidos a mano o resultados calculables sin la función:

| Función | Prueba | Resultado esperado |
| --- | --- | --- |
| `s3_generar_datos` | Los dos tipos, 300 muestras | `X` de $300 \times 2$, `T` de $300 \times 1$ con 150 unos y 150 ceros |
| | Misma semilla y semilla distinta | Datos idénticos y datos distintos |
| | Tipo desconocido | `ValueError` |
| `s3_es_linealmente_separable` | Tabla de verdad del AND | `True` |
| | Tabla de verdad del XOR | `False` |
| `s3_particionar` | 100 observaciones, 60 de clase 1 | 80/20 observaciones; 48 y 12 de clase 1 |
| | Partición con una columna identificadora | Las dos partes son disjuntas, juntas reconstruyen `X` y la misma semilla repite la partición |
| `s3_entrenar_configuracion` | Frente a `Multilayer_perceptron(...).train_mlp(...)` con la misma semilla | Mismas curvas y mismos pesos finales |
| | $M = 3$, 7 épocas | Pesos $3 \times 3$ y $4 \times 1$; curvas de forma $(7,)$; tasa de aciertos en $[0, 1]$ |
| `s3_ejecutar_experimento` | 2 escenarios × 2 configuraciones × 2 semillas × 5 épocas | 4 entradas con matrices $2 \times 5$ |
| | Fila de una semilla | Igual a la corrida individual con esa semilla |
| `s3_tabla_errores_finales` | Curvas finales de entrenamiento $[0.3, 0.1]$ | Mediana 0.2, media 0.2 y desviación $\sqrt{0.02} \approx 0.141$ |
| | 2 escenarios × 2 configuraciones | 4 filas en el orden de entrada |
| `s3_graficar_tabla_curvas` | 2 escenarios × 4 configuraciones | 8 paneles con 2 curvas cada uno |
| | Curva de entrenamiento de un panel | Igual a la mediana por época de las semillas |

In [ ]:
import time

import matplotlib.pyplot as plt
import pandas as pd
from scipy.optimize import linprog
from sklearn.datasets import make_blobs, make_moons
from sklearn.model_selection import train_test_split

# Parámetros de los datos (tabla de 3.1) y del experimento (3.2 a).
S3_N_MUESTRAS = 300
S3_CENTROS_SEPARABLE = [(-2.0, -2.0), (2.0, 2.0)]
S3_DESVIACION_SEPARABLE = 0.7
S3_RUIDO_NO_SEPARABLE = 0.15
S3_FRACCION_VALIDACION = 0.2
S3_ALPHA = 0.01                 # fijado por el enunciado para las cuatro redes
S3_P = 50                       # iteraciones (épocas en lote completo) fijadas por el enunciado
S3_GAMMA_MOMENTUM = 0.9         # γ ≠ 0 «a conveniencia»
S3_CONFIGURACIONES = {          # incisos a) a d) del enunciado
    "a": {"M": 20, "gamma": S3_GAMMA_MOMENTUM},
    "b": {"M": 20, "gamma": 0.0},
    "c": {"M": 2, "gamma": S3_GAMMA_MOMENTUM},
    "d": {"M": 2, "gamma": 0.0},
}
S3_ESCENARIOS = {"separable": "Linealmente separable", "no_separable": "No linealmente separable"}
s3_colores = {"entrenamiento": "#2a78d6", "validacion": "#eb6834"}


def s3_generar_datos(tipo, seed, n_muestras=S3_N_MUESTRAS):
    """Genera uno de los dos conjuntos de §3 en forma matricial.

    Parámetros
    ----------
    tipo : "separable" (make_blobs) o "no_separable" (make_moons).
    seed : int
        `random_state` de scikit-learn.
    n_muestras : int
        Total de observaciones; scikit-learn las reparte por igual entre las dos clases.

    Retorna
    -------
    X : ndarray float de forma (n, 2), una observación por fila.
    T : ndarray float de forma (n, 1) con 1 = clase positiva (contrato de la red con K = 1).
    """
    if tipo == "separable":
        X, y = make_blobs(n_samples=n_muestras, centers=S3_CENTROS_SEPARABLE,
                          cluster_std=S3_DESVIACION_SEPARABLE, random_state=seed)
    elif tipo == "no_separable":
        X, y = make_moons(n_samples=n_muestras, noise=S3_RUIDO_NO_SEPARABLE, random_state=seed)
    else:
        raise ValueError(f"tipo debe ser 'separable' o 'no_separable'; se recibió {tipo!r}")
    return X.astype(float), y.reshape(-1, 1).astype(float)


def s3_es_linealmente_separable(X, T):
    """Indica si una recta (hiperplano) separa estrictamente las dos clases de (X, T).

    Busca (w, b) con s_i (w · x_i + b) >= 1, donde s_i = +1 si T_i = 1 y −1 si T_i = 0,
    como un programa lineal de objetivo nulo. Es factible si y solo si los datos son
    linealmente separables. Devuelve True o False.
    """
    X = np.asarray(X, dtype=float)
    t = np.asarray(T, dtype=float).reshape(-1)
    if X.ndim != 2 or t.shape[0] != X.shape[0]:
        raise ValueError("X debe ser n × D y T debe tener n etiquetas")
    s = np.where(t == 1, 1.0, -1.0)
    X_b = np.hstack([X, np.ones((X.shape[0], 1))])          # variables: w_1, …, w_D, b
    # s_i (w · x_i + b) >= 1  ⇔  −s_i [x_i, 1] · [w, b] <= −1
    r = linprog(np.zeros(X_b.shape[1]), A_ub=-s[:, None] * X_b, b_ub=-np.ones(X.shape[0]),
                bounds=[(None, None)] * X_b.shape[1], method="highs")
    if r.status not in (0, 2):                                # 0: factible; 2: infactible
        raise RuntimeError(f"linprog no resolvió el problema: {r.message}")
    return r.status == 0


def s3_particionar(X, T, seed, fraccion_validacion=S3_FRACCION_VALIDACION):
    """Partición estratificada de scikit-learn en entrenamiento y validación.

    Retorna `(X_ent, X_val, T_ent, T_val)`; validación recibe `fraccion_validacion` de las
    observaciones y conserva la proporción de clases.
    """
    return train_test_split(X, T, test_size=fraccion_validacion, stratify=np.asarray(T)[:, 0],
                            random_state=seed)


def s3_entrenar_configuracion(M, gamma, max_weights, seed, particion, alpha=S3_ALPHA, num_epocas=S3_P):
    """Entrena una red [D, M, 1] y registra el MSE de entrenamiento y de validación por época.

    `particion` es la tupla `(X_ent, X_val, T_ent, T_val)` de `s3_particionar`. La semilla fija
    los pesos iniciales. Cada valor de las curvas se mide después de la actualización de su
    época (contrato de `train_mlp`).

    Retorna
    -------
    dict con 'red', 'train_error' y 'val_error' (ndarray de forma (num_epocas,)) y
    'acierto_val' (tasa de aciertos en validación al final del entrenamiento).
    """
    X_ent, X_val, T_ent, T_val = particion
    red = Multilayer_perceptron([X_ent.shape[1], M, 1], alpha, max_weights, gamma=gamma, seed=seed)
    historial = red.train_mlp(num_epocas, X_ent, T_ent, alpha, gamma, X_val, T_val)
    return {
        "red": red,
        "train_error": np.array(historial["train_error"]),
        "val_error": np.array(historial["val_error"]),
        "acierto_val": float(np.mean(red.predict(X_val) == T_val[:, 0])),
    }


def s3_ejecutar_experimento(particiones, configuraciones, semillas, max_weights,
                            alpha=S3_ALPHA, num_epocas=S3_P):
    """Entrena cada configuración en cada escenario con cada semilla de pesos iniciales.

    Parámetros
    ----------
    particiones : dict escenario -> (X_ent, X_val, T_ent, T_val)
    configuraciones : dict clave -> {"M": int, "gamma": float}
    semillas : secuencia de enteros; las mismas para todas las combinaciones (corridas emparejadas).

    Retorna
    -------
    dict (escenario, clave) -> {"train": R × P, "val": R × P, "acierto_val": (R,)}, con R semillas
    y P = num_epocas.
    """
    resultados = {}
    for escenario, particion in particiones.items():
        for clave, config in configuraciones.items():
            corridas = [s3_entrenar_configuracion(config["M"], config["gamma"], max_weights, int(semilla),
                                                  particion, alpha, num_epocas)
                        for semilla in semillas]
            resultados[(escenario, clave)] = {
                "train": np.array([c["train_error"] for c in corridas]),
                "val": np.array([c["val_error"] for c in corridas]),
                "acierto_val": np.array([c["acierto_val"] for c in corridas]),
            }
    return resultados


def s3_tabla_errores_finales(resultados, configuraciones):
    """Tabla del error final (MSE de la última época) por escenario y configuración.

    Para el MSE de entrenamiento y de validación da la mediana, la media y la desviación
    estándar muestral (ddof = 1) sobre las semillas; agrega la mediana de la tasa de aciertos
    en validación. Una fila por clave de `resultados`, en el mismo orden.
    """
    filas = []
    for (escenario, clave), r in resultados.items():
        fila = {"escenario": escenario, "red": clave,
                "M": configuraciones[clave]["M"], "gamma": configuraciones[clave]["gamma"]}
        for nombre, curvas in (("entrenamiento", r["train"]), ("validacion", r["val"])):
            finales = curvas[:, -1]
            fila[f"mse_{nombre}_mediana"] = float(np.median(finales))
            fila[f"mse_{nombre}_media"] = float(np.mean(finales))
            fila[f"mse_{nombre}_desv"] = float(np.std(finales, ddof=1)) if finales.size > 1 else np.nan
        fila["acierto_validacion_mediana"] = float(np.median(r["acierto_val"]))
        filas.append(fila)
    return pd.DataFrame(filas)


def s3_graficar_tabla_curvas(resultados, escenarios, configuraciones):
    """Tabla de gráficas iteraciones vs. error: filas = configuraciones, columnas = escenarios.

    En cada panel se dibujan la mediana del MSE de entrenamiento y de validación por época (líneas)
    y su rango intercuartílico (bandas) sobre las semillas, con el eje vertical logarítmico y
    compartido. Devuelve la figura.
    """
    fig, ejes = plt.subplots(len(configuraciones), len(escenarios), figsize=(10, 2.6 * len(configuraciones)),
                             sharex=True, sharey=True, squeeze=False, constrained_layout=True)
    for i, (clave, config) in enumerate(configuraciones.items()):
        for j, (escenario, titulo) in enumerate(escenarios.items()):
            ax = ejes[i, j]
            r = resultados[(escenario, clave)]
            epocas = np.arange(1, r["train"].shape[1] + 1)
            for nombre, curvas in (("entrenamiento", r["train"]), ("validacion", r["val"])):
                q1, mediana, q3 = np.percentile(curvas, [25, 50, 75], axis=0)
                ax.fill_between(epocas, q1, q3, color=s3_colores[nombre], alpha=0.15, linewidth=0)
                ax.plot(epocas, mediana, color=s3_colores[nombre], linewidth=1.8,
                        label="entrenamiento (80 %)" if nombre == "entrenamiento" else "validación (20 %)")
            ax.set_yscale("log")
            ax.grid(True, which="major", color="#e8e7e2", linewidth=0.8)
            ax.spines[["top", "right"]].set_visible(False)
            if i == 0:
                ax.set_title(titulo, fontsize=11)
            if j == 0:
                ax.set_ylabel(f"{clave}) M = {config['M']}, γ = {config['gamma']:g}\nMSE", fontsize=9)
            if i == len(configuraciones) - 1:
                ax.set_xlabel("Iteración (época)")
    ejes[0, 0].legend(loc="upper right", fontsize=8, frameon=False)
    return fig

In [ ]:
import sys
import unittest

S3_AND_X = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])


class TestR2GenerarDatos(unittest.TestCase):
    def test_formas_y_clases(self):
        for tipo in ("separable", "no_separable"):
            X, T = s3_generar_datos(tipo, seed=SEEDS["s3_r2"])
            self.assertEqual((X.shape, T.shape), ((300, 2), (300, 1)))
            self.assertTrue(np.all((T == 0) | (T == 1)))
            self.assertEqual(int(T.sum()), 150)

    def test_semilla(self):
        X1, T1 = s3_generar_datos("no_separable", seed=7)
        X2, T2 = s3_generar_datos("no_separable", seed=7)
        X3, _ = s3_generar_datos("no_separable", seed=8)
        np.testing.assert_array_equal(X1, X2)
        np.testing.assert_array_equal(T1, T2)
        self.assertFalse(np.allclose(X1, X3))

    def test_tipo_desconocido(self):
        with self.assertRaises(ValueError):
            s3_generar_datos("circulos", seed=1)


class TestR2Separabilidad(unittest.TestCase):
    def test_and_es_separable(self):
        self.assertTrue(s3_es_linealmente_separable(S3_AND_X, np.array([[0], [0], [0], [1]])))

    def test_xor_no_es_separable(self):
        self.assertFalse(s3_es_linealmente_separable(S3_AND_X, np.array([[0], [1], [1], [0]])))


class TestR2Particion(unittest.TestCase):
    @staticmethod
    def _datos():
        X = np.column_stack([np.arange(100.0), np.zeros(100)])   # la primera columna identifica la fila
        T = np.zeros((100, 1))
        T[:60] = 1.0
        return X, T

    def test_tamanos_y_estratificacion(self):
        X, T = self._datos()
        X_ent, X_val, T_ent, T_val = s3_particionar(X, T, seed=3)
        self.assertEqual((X_ent.shape, X_val.shape, T_ent.shape, T_val.shape),
                         ((80, 2), (20, 2), (80, 1), (20, 1)))
        self.assertEqual((int(T_ent.sum()), int(T_val.sum())), (48, 12))

    def test_disjunta_completa_y_reproducible(self):
        X, T = self._datos()
        X_ent, X_val, T_ent, _ = s3_particionar(X, T, seed=3)
        ids = np.concatenate([X_ent[:, 0], X_val[:, 0]])
        np.testing.assert_array_equal(np.sort(ids), np.arange(100.0))
        np.testing.assert_array_equal(T_ent[:, 0], (X_ent[:, 0] < 60).astype(float))  # T sigue a su fila
        np.testing.assert_array_equal(s3_particionar(X, T, seed=3)[0], X_ent)


class TestR2Entrenamiento(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        X, T = s3_generar_datos("no_separable", seed=11, n_muestras=60)
        cls.particion = s3_particionar(X, T, seed=11)

    def test_equivale_a_train_mlp(self):
        r = s3_entrenar_configuracion(4, 0.9, 1.0, seed=5, particion=self.particion, num_epocas=10)
        X_ent, X_val, T_ent, T_val = self.particion
        red = Multilayer_perceptron([2, 4, 1], S3_ALPHA, 1.0, gamma=0.9, seed=5)
        h = red.train_mlp(10, X_ent, T_ent, S3_ALPHA, 0.9, X_val, T_val)
        np.testing.assert_allclose(r["train_error"], h["train_error"])
        np.testing.assert_allclose(r["val_error"], h["val_error"])
        np.testing.assert_allclose(r["red"].W_s, red.W_s)

    def test_formas_y_arquitectura(self):
        r = s3_entrenar_configuracion(3, 0.0, 1.0, seed=2, particion=self.particion, num_epocas=7)
        self.assertEqual((r["red"].W_o.shape, r["red"].W_s.shape), ((3, 3), (4, 1)))
        self.assertEqual((r["train_error"].shape, r["val_error"].shape), ((7,), (7,)))
        self.assertTrue(0.0 <= r["acierto_val"] <= 1.0)


class TestR2Experimento(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.particiones = {tipo: s3_particionar(*s3_generar_datos(tipo, seed=4, n_muestras=40), seed=4)
                           for tipo in ("separable", "no_separable")}
        cls.configs = {"x": {"M": 3, "gamma": 0.9}, "y": {"M": 2, "gamma": 0.0}}
        cls.resultados = s3_ejecutar_experimento(cls.particiones, cls.configs, semillas=[10, 20],
                                                 max_weights=1.0, num_epocas=5)

    def test_claves_y_formas(self):
        self.assertEqual(set(self.resultados), {(e, c) for e in self.particiones for c in self.configs})
        for r in self.resultados.values():
            self.assertEqual((r["train"].shape, r["val"].shape, r["acierto_val"].shape), ((2, 5), (2, 5), (2,)))

    def test_fila_igual_a_corrida_individual(self):
        individual = s3_entrenar_configuracion(2, 0.0, 1.0, seed=20, particion=self.particiones["separable"],
                                               num_epocas=5)
        r = self.resultados[("separable", "y")]
        np.testing.assert_allclose(r["train"][1], individual["train_error"])
        np.testing.assert_allclose(r["val"][1], individual["val_error"])


class TestR2TablaErrores(unittest.TestCase):
    configs = {"a": {"M": 20, "gamma": 0.9}, "b": {"M": 2, "gamma": 0.0}}

    def test_calculo_a_mano(self):
        resultados = {("separable", "a"): {"train": np.array([[0.4, 0.3], [0.2, 0.1]]),
                                           "val": np.array([[0.5, 0.5], [0.3, 0.3]]),
                                           "acierto_val": np.array([1.0, 0.5])}}
        fila = s3_tabla_errores_finales(resultados, self.configs).iloc[0]
        self.assertAlmostEqual(fila["mse_entrenamiento_mediana"], 0.2)
        self.assertAlmostEqual(fila["mse_entrenamiento_media"], 0.2)
        self.assertAlmostEqual(fila["mse_entrenamiento_desv"], np.sqrt(0.02))
        self.assertAlmostEqual(fila["mse_validacion_mediana"], 0.4)
        self.assertAlmostEqual(fila["acierto_validacion_mediana"], 0.75)
        self.assertEqual((fila["M"], fila["gamma"]), (20, 0.9))

    def test_una_fila_por_combinacion(self):
        r = {"train": np.ones((2, 3)), "val": np.ones((2, 3)), "acierto_val": np.ones(2)}
        claves = [("separable", "a"), ("separable", "b"), ("no_separable", "a"), ("no_separable", "b")]
        tabla = s3_tabla_errores_finales({k: r for k in claves}, self.configs)
        self.assertEqual(list(zip(tabla["escenario"], tabla["red"])), claves)


class TestR2TablaGraficas(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        rng = np.random.default_rng(0)
        cls.resultados = {(e, c): {"train": rng.uniform(0.01, 0.5, (3, 6)), "val": rng.uniform(0.01, 0.5, (3, 6)),
                                   "acierto_val": np.ones(3)}
                          for e in S3_ESCENARIOS for c in S3_CONFIGURACIONES}
        cls.fig = s3_graficar_tabla_curvas(cls.resultados, S3_ESCENARIOS, S3_CONFIGURACIONES)

    @classmethod
    def tearDownClass(cls):
        plt.close(cls.fig)

    def test_ocho_paneles_con_dos_curvas(self):
        ejes = self.fig.axes
        self.assertEqual(len(ejes), 8)
        self.assertTrue(all(len(ax.lines) == 2 for ax in ejes))

    def test_curva_es_la_mediana(self):
        ax = self.fig.axes[0]       # fila a), columna separable
        np.testing.assert_allclose(ax.lines[0].get_ydata(),
                                   np.median(self.resultados[("separable", "a")]["train"], axis=0))


suite_s3 = unittest.TestSuite()
for caso in (TestR2GenerarDatos, TestR2Separabilidad, TestR2Particion, TestR2Entrenamiento,
             TestR2Experimento, TestR2TablaErrores, TestR2TablaGraficas):
    suite_s3.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(caso))
resultado_s3 = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite_s3)
assert resultado_s3.wasSuccessful(), "Fallaron pruebas unitarias de §3"

#### Generación, verificación y partición de los datos

La celda siguiente genera los dos conjuntos, comprueba su separabilidad con el programa lineal y los
divide en entrenamiento y validación. La gráfica muestra ambos conjuntos con las dos partes.

In [ ]:
s3_datos = {tipo: s3_generar_datos(tipo, seed=SEEDS["s3_r2"]) for tipo in S3_ESCENARIOS}
s3_particiones = {tipo: s3_particionar(X, T, seed=SEEDS["s3_r2"]) for tipo, (X, T) in s3_datos.items()}

s3_filas_datos = []
for s3_tipo, (s3_X, s3_T) in s3_datos.items():
    s3_X_ent, s3_X_val, s3_T_ent, s3_T_val = s3_particiones[s3_tipo]
    s3_filas_datos.append({
        "escenario": S3_ESCENARIOS[s3_tipo], "n": len(s3_X), "clase 1": int(s3_T.sum()),
        "linealmente separable (LP)": s3_es_linealmente_separable(s3_X, s3_T),
        "entrenamiento": len(s3_X_ent), "clase 1 (ent.)": int(s3_T_ent.sum()),
        "validación": len(s3_X_val), "clase 1 (val.)": int(s3_T_val.sum()),
    })
s3_resumen_datos = pd.DataFrame(s3_filas_datos)
assert s3_resumen_datos["linealmente separable (LP)"].tolist() == [True, False], \
    "Los conjuntos no tienen la separabilidad esperada"

s3_fig_datos, s3_ejes_datos = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
s3_colores_clase = {0: "#2a78d6", 1: "#eb6834"}
for s3_ax, (s3_tipo, s3_titulo) in zip(s3_ejes_datos, S3_ESCENARIOS.items()):
    s3_X_ent, s3_X_val, s3_T_ent, s3_T_val = s3_particiones[s3_tipo]
    for s3_clase, s3_color in s3_colores_clase.items():
        s3_e = s3_T_ent[:, 0] == s3_clase
        s3_v = s3_T_val[:, 0] == s3_clase
        s3_ax.scatter(s3_X_ent[s3_e, 0], s3_X_ent[s3_e, 1], s=18, color=s3_color, alpha=0.75,
                      label=f"clase {s3_clase}, entrenamiento")
        s3_ax.scatter(s3_X_val[s3_v, 0], s3_X_val[s3_v, 1], s=38, facecolors="none", edgecolors=s3_color,
                      linewidths=1.4, label=f"clase {s3_clase}, validación")
    s3_ax.set_title(s3_titulo, fontsize=11)
    s3_ax.set_xlabel("$x_1$")
    s3_ax.set_ylabel("$x_2$")
    s3_ax.set_aspect("equal", adjustable="datalim")
    s3_ax.grid(True, color="#e8e7e2", linewidth=0.8)
    s3_ax.spines[["top", "right"]].set_visible(False)
s3_ejes_datos[0].legend(loc="upper left", fontsize=8, frameon=False)
s3_fig_datos.suptitle("Conjuntos de datos en R² (puntos: entrenamiento 80 %; círculos: validación 20 %)",
                      fontsize=11)
plt.show()
s3_resumen_datos.style.hide(axis="index")

### 3.2 a) Redes y experimento

Las cuatro redes tienen $D = 2$ entradas, $K = 1$ salida (la primera componente de
$\operatorname{softmax}([p, 0])$, ver D3 y §1) y $\alpha = 0.01$:

| Red | $M$ | $\gamma$ |
| --- | --- | --- |
| a) | 20 | 0.9 |
| b) | 20 | 0 |
| c) | 2 | 0.9 |
| d) | 2 | 0 |

**Decisiones (D10 en `JOURNAL.md`).**

- **$\gamma \ne 0$.** El enunciado lo deja «a conveniencia». Se usa $\gamma = 0.9$, un valor habitual para
  el momentum y el mismo que se probó en §2.
- **$P = 50$ iteraciones.** Son 50 épocas en lote completo: en cada una, `train_mlp` actualiza los
  pesos con todas las muestras de entrenamiento («todo el conjunto de muestras»). Después de cada época
  se registra el MSE en entrenamiento (80 %) y en validación (20 %), que es la evaluación del error que
  pide el enunciado.
- **Error final de convergencia.** Es el MSE de la última época ($P = 50$). Con $\alpha = 0.01$ y
  50 épocas algunas redes todavía no habrán convergido; la tabla reporta el valor alcanzado tal cual.
- **`max_weights`.** El enunciado no lo fija para §3. Se elige en la celda siguiente entre
  $\{0.5, 1, 2, 5\}$, con el mismo valor para las cuatro redes y los dos escenarios, usando solo el MSE
  final de **entrenamiento**: el valor con la menor media, sobre las ocho combinaciones, de la mediana
  entre semillas. La validación no interviene en la elección. Además del recorte, `max_weights` fija la
  escala de la inicialización (pesos uniformes en $[-\texttt{max\_weights}, \texttt{max\_weights}]$,
  contrato de §1), así que el valor elegido también cambia el punto de partida. La celda
  `s3-max-weights-sensibilidad` muestra qué valor elegirían otros criterios razonables y si las
  conclusiones de 3.2 b cambian con cada valor probado.
- **Repeticiones.** Los pesos iniciales son aleatorios y una sola corrida puede ser poco representativa.
  Cada combinación se entrena con $R = 20$ semillas de pesos iniciales (3 con `TP3_QUICK`), derivadas de
  `SEEDS["s3_r2"]`. Las ocho combinaciones usan las mismas semillas y la misma partición, de modo que las
  corridas quedan emparejadas. Las curvas muestran la mediana y el rango intercuartílico entre semillas;
  la tabla, la mediana, la media y la desviación estándar del error final. Como los datos y la partición
  son los mismos en todas las corridas, esa variabilidad refleja solo los pesos iniciales.
- Como complemento se reporta la tasa de aciertos en validación, que el enunciado no pide.

In [ ]:
# Elección de max_weights con el MSE final de entrenamiento (la validación no interviene).
s3_num_semillas = quick_or_full(3, 20)
s3_semillas = np.random.default_rng(SEEDS["s3_r2"]).integers(0, 2**31 - 1, size=s3_num_semillas)
s3_valores_max_weights = [0.5, 1.0, 2.0, 5.0]

s3_resultados_mw = {}   # max_weights -> resultados de s3_ejecutar_experimento
s3_filas_mw = []
for s3_mw in s3_valores_max_weights:
    s3_resultados_mw[s3_mw] = s3_ejecutar_experimento(s3_particiones, S3_CONFIGURACIONES, s3_semillas,
                                                      max_weights=s3_mw)
    s3_fila = {"max_weights": s3_mw}
    for (s3_tipo, s3_clave), s3_r in s3_resultados_mw[s3_mw].items():
        s3_fila[f"{s3_clave}) {s3_tipo}"] = float(np.median(s3_r["train"][:, -1]))
    s3_fila["media"] = float(np.mean([v for k, v in s3_fila.items() if k != "max_weights"]))
    s3_filas_mw.append(s3_fila)
s3_tabla_mw = pd.DataFrame(s3_filas_mw)
s3_max_weights = float(s3_tabla_mw.loc[s3_tabla_mw["media"].idxmin(), "max_weights"])

print(f"Mediana del MSE de entrenamiento en la época {S3_P} ({s3_num_semillas} semillas); "
      f"max_weights elegido: {s3_max_weights:g}")
s3_tabla_mw.style.format({c: ("{:g}" if c == "max_weights" else "{:.2e}") for c in s3_tabla_mw.columns}
                         ).hide(axis="index")

In [ ]:
# Sensibilidad a max_weights: qué elegirían otros criterios y si las conclusiones de 3.2 b cambian.
# Usa las medianas de s3-max-weights; no entrena redes nuevas. La elección sigue siendo la de D10.
from IPython.display import display

s3_medianas_mw = s3_tabla_mw.drop(columns="media").set_index("max_weights")
s3_criterios_mw = pd.DataFrame({
    "media (D10)": s3_medianas_mw.mean(axis=1),
    "media de log10": np.log10(s3_medianas_mw).mean(axis=1),
    "rango medio": s3_medianas_mw.rank(axis=0).mean(axis=1),   # 1 = menor MSE en esa combinación
})
for s3_criterio in s3_criterios_mw.columns:
    print(f"{s3_criterio}: elige max_weights = {s3_criterios_mw[s3_criterio].idxmin():g}")
display(s3_criterios_mw.style.format("{:.4g}").format_index("{:g}", axis=0))

# Comparaciones de 3.2 b: en cuántos pares (escenario y configuración) la primera red de cada
# par termina con menor mediana del MSE de entrenamiento que la segunda.
s3_comparaciones = {
    "M = 20 < M = 2": [("a", "c"), ("b", "d")],
    "γ = 0.9 < γ = 0": [("a", "b"), ("c", "d")],
    "b) < c)": [("b", "c")],
}
s3_conclusiones_mw = pd.DataFrame(index=s3_medianas_mw.index)
for s3_nombre, s3_pares in s3_comparaciones.items():
    s3_conclusiones_mw[s3_nombre] = [
        f"{sum(s3_fila[f'{x}) {t}'] < s3_fila[f'{y}) {t}'] for x, y in s3_pares for t in S3_ESCENARIOS)}"
        f"/{len(s3_pares) * len(S3_ESCENARIOS)}"
        for _, s3_fila in s3_medianas_mw.iterrows()
    ]
s3_conclusiones_mw.style.format_index("{:g}", axis=0).format_index(escape="html", axis=1)


In [ ]:
# Experimento de 3.2 a): 2 escenarios × 4 redes × R semillas × P = 50 épocas.
s3_inicio = time.perf_counter()
s3_resultados = s3_ejecutar_experimento(s3_particiones, S3_CONFIGURACIONES, s3_semillas,
                                        max_weights=s3_max_weights)
print(f"{len(s3_resultados)} combinaciones × {s3_num_semillas} semillas × {S3_P} épocas "
      f"(α = {S3_ALPHA}, max_weights = {s3_max_weights:g}) en {time.perf_counter() - s3_inicio:.1f} s")

#### a′) Tabla de gráficas: iteraciones vs. error

Una fila por red y una columna por escenario. Las líneas son la mediana del MSE entre las semillas y las
bandas, el rango intercuartílico; el eje vertical es logarítmico y compartido para poder comparar los
paneles.

In [ ]:
s3_fig_curvas = s3_graficar_tabla_curvas(s3_resultados, S3_ESCENARIOS, S3_CONFIGURACIONES)
s3_fig_curvas.suptitle(f"MSE por iteración, α = {S3_ALPHA}, P = {S3_P} "
                       f"(mediana y rango intercuartílico sobre {s3_num_semillas} semillas)", fontsize=11)
plt.show()

#### b′) Tabla de errores finales de convergencia

MSE en la última época ($P = 50$) para cada red y escenario: mediana, media y desviación estándar
muestral entre semillas, en entrenamiento y en validación, y la mediana de la tasa de aciertos en
validación.

In [ ]:
s3_tabla_errores = s3_tabla_errores_finales(s3_resultados, S3_CONFIGURACIONES)
s3_tabla_errores["escenario"] = s3_tabla_errores["escenario"].map(S3_ESCENARIOS)
s3_formatos = {c: "{:.2e}" for c in s3_tabla_errores.columns if c.startswith("mse_")}
s3_formatos.update({"gamma": "{:g}", "acierto_validacion_mediana": "{:.1%}"})
s3_tabla_errores.style.format(s3_formatos).hide(axis="index")

#### Experimento complementario: tasa de aprendizaje según $M$

Sirve para responder con datos cómo ajustar $\alpha$ según el número de neuronas ocultas (3.2 b). Se
entrenan las redes con $M = 2$ y $M = 20$, sin momentum ($\gamma = 0$), con
$\alpha \in \{0.003, 0.01, 0.03, 0.1, 0.3\}$. Se usan las mismas $P = 50$ épocas, la misma partición, las
mismas semillas y el mismo `max_weights` que en 3.2 a. Se reporta la mediana entre semillas del MSE final
de entrenamiento.

In [ ]:
s3_valores_alpha = [0.003, 0.01, 0.03, 0.1, 0.3]
s3_configs_alpha = {"M2": {"M": 2, "gamma": 0.0}, "M20": {"M": 20, "gamma": 0.0}}
s3_filas_alpha = []
for s3_a in s3_valores_alpha:
    s3_r_a = s3_ejecutar_experimento(s3_particiones, s3_configs_alpha, s3_semillas,
                                     max_weights=s3_max_weights, alpha=s3_a)
    for (s3_tipo, s3_clave), s3_r in s3_r_a.items():
        s3_filas_alpha.append({"escenario": S3_ESCENARIOS[s3_tipo], "M": s3_configs_alpha[s3_clave]["M"],
                               "alpha": s3_a, "mse_entrenamiento_mediana": float(np.median(s3_r["train"][:, -1]))})
s3_tabla_alpha = pd.DataFrame(s3_filas_alpha)

s3_fig_alpha, s3_ejes_alpha = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True, constrained_layout=True)
for s3_ax, s3_titulo in zip(s3_ejes_alpha, S3_ESCENARIOS.values()):
    for s3_M, s3_color in ((20, "#2a78d6"), (2, "#eb6834")):
        s3_sub = s3_tabla_alpha.query("escenario == @s3_titulo and M == @s3_M")
        s3_ax.plot(s3_sub["alpha"], s3_sub["mse_entrenamiento_mediana"], marker="o", color=s3_color,
                   linewidth=1.8, label=f"M = {s3_M}")
    s3_ax.set_xscale("log")
    s3_ax.set_yscale("log")
    s3_ax.set_title(s3_titulo, fontsize=11)
    s3_ax.set_xlabel("α")
    s3_ax.grid(True, which="major", color="#e8e7e2", linewidth=0.8)
    s3_ax.spines[["top", "right"]].set_visible(False)
s3_ejes_alpha[0].set_ylabel(f"MSE de entrenamiento, época {S3_P}\n(mediana, γ = 0)")
s3_ejes_alpha[0].legend(frameon=False)
plt.show()
(s3_tabla_alpha.pivot_table(index="alpha", columns=["escenario", "M"], values="mse_entrenamiento_mediana")
 .style.format("{:.2e}").format_index("{:g}", axis=0))

### 3.2 b) Comparación de los resultados

*Los números citados son de la ejecución del 2026-11-26 en Linux x86_64 (20 semillas) y se revisan con
la ejecución final de T8. Salvo que se indique otra cosa, cada valor es la mediana entre semillas del MSE
en la época 50.*

**Comparación de las gráficas.**

- **Conjunto separable.** Solo la red a) ($M = 20$, $\gamma = 0.9$) converge con claridad en las
  50 iteraciones. Su MSE de entrenamiento baja de 0.24 en la primera época a menos de 0.01 en la sexta
  y termina en $3.1 \cdot 10^{-5}$. Las redes b) ($M = 20$, $\gamma = 0$) y c) ($M = 2$, $\gamma = 0.9$)
  descienden de forma sostenida pero lenta y terminan en $1.7 \cdot 10^{-2}$ y $3.8 \cdot 10^{-2}$;
  aun así, ya clasifican bien todo el conjunto de validación (mediana de 100 % de aciertos). La red d)
  ($M = 2$, $\gamma = 0$) casi no se mueve: pasa de 0.28 a 0.18, y sus aciertos en validación van de 0 %
  a 100 % según los pesos iniciales (mediana de 67 %). En 50 épocas sigue cerca de su estado inicial.
- **Conjunto no separable.** Ninguna red baja de 0.05. Los MSE finales de entrenamiento son 0.079 (a),
  0.13 (b), 0.13 (c) y 0.25 (d), y las curvas son más planas: la frontera es curva y el ruido hace que
  las clases se traslapen. La mediana de aciertos en validación es 85.0 % (a), 85.8 % (b), 87.5 % (c) y 50.0 % (d);
  la red d) no supera al azar.
- **Entrenamiento frente a validación.** En los ocho paneles las dos curvas casi coinciden. La
  diferencia media entre el MSE final de validación y el de entrenamiento va de −0.007 a +0.016. Con
  $\alpha = 0.01$ y 50 iteraciones no hay sobreajuste: el problema es de subajuste, por falta de
  entrenamiento, y no de generalización.
- **El MSE y la tasa de aciertos no ordenan igual.** En el conjunto no separable, c) tiene un MSE mayor
  que a) y, sin embargo, una mediana de aciertos algo mayor (87.5 % frente a 85.0 %). El MSE también
  penaliza las salidas poco seguras que quedan del lado correcto del umbral 0.5.
- **Momentum.** Con el mismo $M$ y el mismo escenario, $\gamma = 0.9$ reduce el MSE final en los cuatro
  pares. En el separable baja de $1.7 \cdot 10^{-2}$ a $3.1 \cdot 10^{-5}$ con $M = 20$ y de 0.18 a 0.038
  con $M = 2$; en el no separable, de 0.13 a 0.079 y de 0.25 a 0.13. LeCun et al. [7] explican que el
  momentum produce una tasa de aprendizaje efectiva mayor. En nuestra regla de actualización, si el
  gradiente mantiene su dirección, la velocidad $v$ tiende a $\text{paso}/(1-\gamma)$, es decir, diez
  veces el paso de AdaGrad con $\gamma = 0.9$.
- **Sensibilidad a `max_weights`.** El criterio de D10 (media de las ocho medianas) elige 1 por poco:
  0.102 frente a 0.115 con 0.5 y 0.116 con 2. La media aritmética la dominan las combinaciones con más
  error: con la media de los $\log_{10}$ se elegiría 2, y con el rango medio, 1, también por poco (2 frente
  a 2.125 de `max_weights` = 2).
  Con los cuatro valores probados, $M = 20$ termina con menos error que $M = 2$, y $\gamma = 0.9$ con
  menos que $\gamma = 0$, en los cuatro pares, así que las conclusiones de este apartado no dependen de
  la elección. Lo único que cambia es el orden entre b) y c): con `max_weights` igual a 0.5 o 1, b)
  termina con menos error en los dos escenarios; con 2 o 5, c).

**¿Cómo se puede ajustar $\alpha$ en función de las neuronas de la capa oculta?**

Con AdaGrad [8], el paso de cada peso es $\alpha\, g / (\sqrt{G} + \varepsilon)$ (D5). Como $G$ incluye
$g^2$, se cumple $|g| / \sqrt{G} \le 1$: sin momentum, ningún peso se mueve más de $\alpha$ por época,
sin importar la magnitud de su gradiente. El peso neto de la salida es
$a^s = w_0 + \sum_{m=1}^{M} w_m\, y^o_m$, con $|y^o_m| \le 1$ por la tangente hiperbólica. Por eso, los
pesos de la capa de salida pueden cambiarlo a lo sumo $(M + 1)\,\alpha$ por época. Con $M = 2$ y
$\alpha = 0.01$ eso es 0.03 por época y, como mucho, 1.5 en 50 épocas; con $M = 20$, 0.21 por época y
10.5 en total. Con el mismo $\alpha$, la red pequeña está obligada a aprender más despacio. Para que la
salida cambie a una velocidad parecida, $\alpha$ debe crecer cuando $M$ disminuye, aproximadamente en
proporción inversa a $M$.

El experimento complementario lo confirma. En los dos tamaños el MSE final baja al aumentar $\alpha$
dentro del rango probado. Además, $M = 2$ necesita $\alpha = 0.1$ para llegar a lo que $M = 20$ logra
con $\alpha = 0.01$: $2.3 \cdot 10^{-2}$ frente a $1.7 \cdot 10^{-2}$ en el separable y 0.12 frente a
0.13 en el no separable. Es decir, una capa oculta diez veces menor necesita un $\alpha$ unas diez veces
mayor. En sentido inverso, con más neuronas el mismo $\alpha$ produce cambios agregados mayores en la
salida, de modo que una red grande admite, o necesita, un $\alpha$ menor para no dar pasos excesivos.
Hasta $\alpha = 0.3$ no apareció inestabilidad en 50 épocas, así que ese límite no se observó aquí.
LeCun et al. [7] recomiendan igualar la velocidad de aprendizaje de los pesos («Equalize the Learning
Speeds»), con tasas que dependan del número de entradas de cada unidad. Su regla concreta es para
descenso del gradiente sin la normalización por peso de AdaGrad. Por eso aquí se toma solo el principio
y la dependencia concreta se mide.

**¿Qué relación hay entre la cantidad de neuronas ocultas y el error de entrenamiento?**

En los cuatro pares comparables (mismo escenario y mismo $\gamma$), $M = 20$ termina con menos error de
entrenamiento que $M = 2$. En el separable: $3.1 \cdot 10^{-5}$ frente a $3.8 \cdot 10^{-2}$ con
$\gamma = 0.9$, y $1.7 \cdot 10^{-2}$ frente a 0.18 con $\gamma = 0$. En el no separable: 0.079 frente a
0.13, y 0.13 frente a 0.25. Hay dos causas:

1. **Velocidad.** Con AdaGrad, la salida cambia más rápido cuanto más neuronas hay, por el argumento
   anterior.
2. **Capacidad.** Con $M = 2$ la frontera de decisión combina solo dos transiciones suaves, una por
   neurona tanh. Eso basta para separar con una recta, pero limita la frontera curva de las medias
   lunas. Con más neuronas, una capa oculta puede aproximar fronteras más complejas [1].

El recorte de los pesos agrega un límite. Con $|w| \le$ `max_weights` y $|y^o_m| \le 1$, el peso neto
de la salida cumple $|a^s| \le (M + 1)\,\texttt{max\_weights}$. Con $M = 2$ y `max_weights` = 1,
$|a^s| \le 3$ y la salida queda entre $\sigma(-3) \approx 0.047$ y $\sigma(3) \approx 0.953$: aun
clasificando todo bien, el MSE no puede bajar de $(1 - \sigma(3))^2 \approx 2.2 \cdot 10^{-3}$. Con
$M = 20$ el límite es $|a^s| \le 21$, que en la práctica no restringe. En la época 50 ese piso todavía
no se alcanza (el menor MSE de $M = 2$ con `max_weights` = 1 es $1.5 \cdot 10^{-2}$, con $\alpha = 0.3$
en el experimento complementario), pero con más épocas limitaría a la red pequeña.

Además, $M = 20$ da resultados más estables entre semillas. En el no separable con $\gamma = 0.9$, la
desviación estándar del MSE final de entrenamiento es 0.004 con $M = 20$ y 0.035 con $M = 2$. Reducir el
error de entrenamiento no garantiza generalizar mejor (Barron [4], en la investigación de abajo). Aquí la
validación sigue al entrenamiento porque 50 épocas con $\alpha = 0.01$ no alcanzan para sobreajustar las
240 muestras de entrenamiento.

**Referencias de este apartado** (las [1] a [6] están en la investigación siguiente)

7. Y. LeCun, L. Bottou, G. B. Orr y K.-R. Müller, “Efficient BackProp”, en G. B. Orr y K.-R. Müller
   (eds.), *Neural Networks: Tricks of the Trade*, LNCS 1524, Springer, 1998, pp. 9–50, sección 4.7.
   doi:10.1007/3-540-49430-8_2
8. J. Duchi, E. Hazan e Y. Singer, “Adaptive subgradient methods for online learning and stochastic
   optimization”, *Journal of Machine Learning Research*, vol. 12, pp. 2121–2159, 2011.

### 3.2 b) Investigación: neuronas en la capa oculta frente a neuronas de entrada

*La comparación de las gráficas, el ajuste de $\alpha$ según $M$ y la relación entre $M$ y
el error de entrenamiento están en el apartado anterior.*

**Pregunta.** ¿Puede ajustarse la cantidad de neuronas en la capa oculta $M$ en función
de las neuronas en la capa de entrada $D$?

**Lo que dice la literatura consultada.**

- **Existe una red suficiente, pero el teorema no dice de qué tamaño.** Hornik,
  Stinchcombe y White [1] demostraron que una red con una sola capa oculta puede
  aproximar cualquier función con la precisión deseada si dispone de suficientes
  neuronas ocultas. El resultado garantiza existencia; no da una fórmula para $M$.
- **Las cotas para memorizar los datos dependen de $N$, no de $D$.** Sartori y
  Antsaklis [2] muestran que una capa oculta con $p-1$ neuronas basta para ajustar
  exactamente $p$ patrones de entrenamiento. Huang [3] obtiene, para redes con dos
  capas ocultas, que unas $2\sqrt{(m+2)N}$ neuronas ocultas aprenden $N$ muestras
  distintas, con $m$ salidas. En ambos casos el tamaño queda determinado por el número
  de muestras y de salidas; la dimensión de entrada no aparece en la cota. Además,
  ajustar exactamente el entrenamiento no garantiza generalizar.
- **Cuando se busca generalizar, $D$ sí aparece.** Barron [4] acota el error cuadrático
  medio integrado de una red de una capa oculta con $n$ nodos, entrenada con $N$
  observaciones de dimensión $d$, por $O(C_f^2/n) + O\big(\tfrac{nd}{N}\log N\big)$,
  donde $C_f$ mide la complejidad de la función objetivo. El primer término
  (aproximación) baja al agregar neuronas; el segundo (estimación) crece con $n \cdot d$,
  que es del orden del número de pesos de la capa oculta. El balance se alcanza con
  $n \sim C_f \, \big(N/(d \log N)\big)^{1/2}$: con el mismo número de muestras, más
  entradas justifican **menos** neuronas ocultas, no más.
- **En la práctica no hay una regla exacta.** Sheela y Deepa [5] revisan unos 20 años
  de métodos para fijar el número de neuronas ocultas y comparan 101 criterios;
  advierten que una elección arbitraria produce sobreajuste o subajuste. Stathakis [6]
  concluye que no existe una solución exacta y busca la arquitectura con un algoritmo
  genético que favorece soluciones precisas y compactas.

**Conclusión.** Sí puede usarse $D$ para orientar la elección de $M$, pero no como única
variable: el tamaño adecuado depende también del número de muestras $N$, de las salidas
$K$ y de la complejidad de la frontera de decisión. Las reglas basadas solo en $D$ son
heurísticas de partida; $M$ debe calibrarse con un conjunto de validación.

**Referencias**

1. K. Hornik, M. Stinchcombe y H. White, “Multilayer feedforward networks are universal
   approximators”, *Neural Networks*, vol. 2, n.º 5, pp. 359–366, 1989.
   doi:10.1016/0893-6080(89)90020-8
2. M. A. Sartori y P. J. Antsaklis, “A simple method to derive bounds on the size and
   to train multilayer neural networks”, *IEEE Transactions on Neural Networks*,
   vol. 2, n.º 4, pp. 467–471, 1991. doi:10.1109/72.88168
3. G.-B. Huang, “Learning capability and storage capacity of two-hidden-layer
   feedforward networks”, *IEEE Transactions on Neural Networks*, vol. 14, n.º 2,
   pp. 274–281, 2003. doi:10.1109/TNN.2003.809401
4. A. R. Barron, “Approximation and estimation bounds for artificial neural networks”,
   *Machine Learning*, vol. 14, pp. 115–133, 1994. doi:10.1023/A:1022650905902
5. K. G. Sheela y S. N. Deepa, “Review on methods to fix number of hidden neurons in
   neural networks”, *Mathematical Problems in Engineering*, vol. 2013, art. 425740,
   2013. doi:10.1155/2013/425740
6. D. Stathakis, “How many hidden layers and nodes?”, *International Journal of Remote
   Sensing*, vol. 30, n.º 8, pp. 2133–2147, 2009. doi:10.1080/01431160802549278

## 4. Detector de ataques con UNSW-NB15

### Preparación de datos UNSW-NB15

*Responsable: P3 (T3).*

#### Origen de los datos

Se usan los dos CSV particionados oficiales de UNSW-NB15 (Moustafa y Slay, 2015), descargados de la
[página del conjunto de datos](https://research.unsw.edu.au/projects/unsw-nb15-dataset) de UNSW Canberra.
No se usan los cuatro CSV crudos (`UNSW-NB15_1.csv` a `UNSW-NB15_4.csv`, unos 2.5 millones de filas). Como
pide el enunciado, los dos archivos se unen y se vuelven a particionar: 80/20 en §4.1 y 70/30 en §4.4.

| Archivo | Filas de datos | SHA-256 |
| --- | --- | --- |
| `UNSW_NB15_training-set.csv` | 175 341 | `bec7dd5ec88dc2a0ccc7a07879d338395ed7421750f675fd0339e07dfe0648fa` |
| `UNSW_NB15_testing-set.csv` | 82 332 | `734fe6642edf758f7c94d7d9149426b49d202fe8e7bf0bef47392489c3c0a559` |

**Dónde están:** los dos archivos están versionados en el repositorio, en `data/raw/unsw_nb15/`, así que
llegan con `git pull`. `.gitattributes` los trata como binarios para que Git no cambie sus fines de línea
(CRLF) ni, por lo tanto, sus sumas. La celda `s4-datos-carga` comprueba las sumas y los conteos antes de
usarlos. Para volver a obtenerlos de la fuente: la página enlaza a una carpeta de SharePoint que no se
puede descargar con `curl`; desde el navegador, entrar a la carpeta de los CSV, luego a la de *training
and testing sets*, y descargar los dos archivos con esos nombres.

**Lo que se comprobó en los datos (T3):**

- Los dos archivos tienen las mismas 45 columnas y empiezan con un BOM de UTF-8. `pandas.read_csv` ya
  lo descarta por su cuenta, pero se lee con `encoding="utf-8-sig"` para dejarlo explícito; con el
  módulo `csv` de Python, por ejemplo, la primera columna se llamaría `"﻿id"`.
- **Clase:** `label` (1 = ataque). Hay 164 673 ataques y 93 000 tráficos normales (63.9 % ataques).
  `attack_cat` vale `Normal` exactamente cuando `label = 0`.
- **Columnas excluidas de los atributos:** `id`, porque es un número de fila que se repite entre archivos
  (cada uno empieza en 1); `label`, porque es la clase; y `attack_cat`, porque la revela.
- **Atributos:** 42 en total. Son 39 numéricos y 3 categóricos: `proto` (133 valores), `service` (13) y
  `state` (11). `service = "-"` (sin servicio identificado, 141 321 filas) se trata como una categoría más.
  `is_ftp_login` toma los valores 0, 1, 2 y 4, no solo 0/1; se deja como numérico, igual que las demás.
- No hay valores faltantes ni columnas numéricas con valores negativos, así que log1p se aplica a las 39.
- `state` toma los valores `ACC` y `CLO` solo en el archivo de prueba original. Después de volver a
  particionar, si una categoría cae solo en prueba, el one-hot la ignora (fila en ceros en ese bloque).

**Duplicados (D9 en `JOURNAL.md`).** 103 575 filas (40 %) repiten exactamente los atributos y la etiqueta
de otra fila; 96 297 de ellas son ataques, sobre todo de la categoría *Generic*. Además, 414 grupos de filas
tienen atributos idénticos con etiquetas distintas. Se decidió **conservar todas las filas**, porque así lo
fija el contrato de datos y el enunciado pide usar los datos provistos. La consecuencia es que una parte de
las filas de prueba tiene una copia exacta en entrenamiento, y para esas filas el modelo no generaliza, solo
reconoce lo que ya vio: la tasa de aciertos, el F1 y el ECE resultan optimistas. `s4-datos-carga` mide esa
superposición en todas las particiones. `load_unsw_nb15(..., drop_duplicates=True)` permite repetir el
análisis sin duplicados.

**Referencia**

N. Moustafa y J. Slay, «UNSW-NB15: a comprehensive data set for network intrusion detection systems
(UNSW-NB15 network data set)», en *2015 Military Communications and Information Systems Conference
(MilCIS)*, 2015, pp. 1–6. doi: [10.1109/MilCIS.2015.7348942](https://doi.org/10.1109/MilCIS.2015.7348942).

#### Funciones de preparación

Son las funciones del contrato de datos acordado en T0 (`JOURNAL.md`). Las usan §4.1, §4.2 y §4.4.

| Función | Qué hace |
| --- | --- |
| `load_unsw_nb15(data_dir, drop_duplicates=False)` | Lee los dos CSV, comprueba que tengan las mismas columnas y los une en un `DataFrame` con un índice nuevo de 0 a n − 1. Con `drop_duplicates=True` quita las filas repetidas en atributos y etiqueta (D9). |
| `split_unsw(df, test_size, seed)` | Parte el `DataFrame` en entrenamiento y prueba con `train_test_split` de scikit-learn, **estratificando por `label`**. Así cada partición conserva la proporción de ataques del total y las corridas varían menos por puro azar. Los `DataFrame` resultantes conservan el índice original. |
| `preprocess_unsw(train_df, test_df, l2=False)` | Convierte las dos particiones en `X_train`, `X_test` (float64, n × D) y `y_train`, `y_test` (enteros 0/1, forma (n,)), con el orden fijo que se describe abajo. |
| `s4_columnas_atributos(df)` | Devuelve las listas de atributos numéricos y categóricos, sin `id`, `label` ni `attack_cat`. |
| `s4_nombres_atributos(train_df)` | Devuelve los nombres de las D columnas de `X` en su orden (`dur`, …, `proto=tcp`, …), para interpretar la matriz. |
| `s4_normalizar_l2(X)` | Divide cada fila entre su norma euclídea. Una fila de norma 0 se deja igual. |
| `s4_etiquetas(df)` | Extrae `label` como enteros de forma (n,) y rechaza valores distintos de 0 y 1. |
| `s4_superposicion(train_df, test_df)` | Fracción de filas de prueba cuyos atributos aparecen idénticos en entrenamiento; mide el efecto de los duplicados (D9). |
| `s4_sha256(ruta)` y `s4_verificar_archivos(data_dir)` | Calculan la suma SHA-256 de un archivo y comparan los CSV descargados con las sumas y los conteos registrados arriba. |

**Orden del preprocesamiento.** Todo lo que se ajusta a los datos (las categorías, las medias y las
desviaciones) sale **solo de entrenamiento** y después se aplica a prueba, para no filtrar información de
prueba:

1. **One-hot** de `proto`, `service` y `state`: una columna 0/1 por cada categoría vista en entrenamiento,
   en orden alfabético. Una categoría que solo aparece en prueba deja ese bloque en ceros. Las columnas
   one-hot no reciben log1p ni z-score.
2. **log1p**: $x \mapsto \ln(1 + x)$ en las columnas numéricas cuyo mínimo en entrenamiento es $\geq 0$
   (en UNSW-NB15, las 39). Varios atributos van de 0 hasta unos $10^9$ (`sload`, `stcpb`, `dtcpb`); el
   logaritmo comprime esas colas para que unos pocos valores enormes no dominen el z-score. Una columna con
   negativos en entrenamiento no recibe log1p. Un valor negativo en prueba en una columna con log1p da
   `ValueError`, en lugar de producir un `NaN` en silencio.
3. **z-score**: $z = (x - \mu) / \sigma$, con $\mu$ y $\sigma$ (desviación poblacional) de entrenamiento.
   Si $\sigma = 0$, se usa $\sigma = 1$, de modo que la columna queda en ceros y no en `NaN`. Sin este
   paso, la tangente hiperbólica de la red se saturaría (D8).
4. **L2 por fila** (solo si `l2=True`, D8): $\mathbf{x} \mapsto \mathbf{x} / \lVert \mathbf{x} \rVert_2$
   sobre la fila completa (numéricas y one-hot). No aprende nada de los datos.

`X` lleva primero las 39 numéricas, en el orden del CSV, y luego los bloques one-hot de `proto`, `service`
y `state`.

**Particiones y semillas** (acordadas en T3). `split_unsw` recibe la semilla de cada partición:

- §4.1, 30 particiones 80/20: `S4_SEMILLAS_80_20` = 4000, 4001, …, 4029 (`SEEDS["s4_datos"] + i`).
- §4.4, 10 particiones 70/30: `S4_SEMILLAS_70_30` = 4500, 4501, …, 4509 (`SEEDS["s4_datos"] + 500 + i`).

El `DataFrame` cargado queda en `s4_unsw` (celda `s4-datos-carga`). Las demás secciones lo reutilizan sin
modificarlo, por ejemplo: `train_df, test_df = split_unsw(s4_unsw, 0.2, S4_SEMILLAS_80_20[i])` y luego
`preprocess_unsw(train_df, test_df, l2=...)`.

**Uso con la red de §1:** `y_train` tiene forma (n,). Para la red con K = 1 se pasa `T = y_train.reshape(-1, 1)`,
porque `train_mlp` exige T de n × K y un vector (n,) da `ValueError` (contrato de la red). Con K = 2 se
construye el one-hot de la etiqueta.

In [ ]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

S4_ARCHIVOS = ("UNSW_NB15_training-set.csv", "UNSW_NB15_testing-set.csv")
# Suma SHA-256 y filas de datos (sin el encabezado) de los archivos descargados en T3 (2026-11-25).
S4_REGISTRO_ARCHIVOS = {
    "UNSW_NB15_training-set.csv": ("bec7dd5ec88dc2a0ccc7a07879d338395ed7421750f675fd0339e07dfe0648fa", 175341),
    "UNSW_NB15_testing-set.csv": ("734fe6642edf758f7c94d7d9149426b49d202fe8e7bf0bef47392489c3c0a559", 82332),
}
S4_ETIQUETA = "label"
S4_EXCLUIDAS = ("id", "label", "attack_cat")
S4_CATEGORICAS = ("proto", "service", "state")

# Semillas de las particiones (acordadas en T3): 30 de 80/20 para §4.1 y 10 de 70/30 para §4.4.
S4_SEMILLAS_80_20 = [SEEDS["s4_datos"] + i for i in range(30)]
S4_SEMILLAS_70_30 = [SEEDS["s4_datos"] + 500 + i for i in range(10)]


def s4_sha256(ruta, bloque=1 << 20):
    """Devuelve la suma SHA-256 (hexadecimal) del archivo `ruta`, leído por bloques."""
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for parte in iter(lambda: f.read(bloque), b""):
            h.update(parte)
    return h.hexdigest()


def s4_verificar_archivos(data_dir):
    """Compara los CSV de `data_dir` con las sumas y los conteos de `S4_REGISTRO_ARCHIVOS`.

    Devuelve un `DataFrame` con una fila por archivo: filas de datos, SHA-256 y si ambos coinciden."""
    data_dir = Path(data_dir)
    filas = []
    for nombre, (sha_esperado, n_esperado) in S4_REGISTRO_ARCHIVOS.items():
        ruta = data_dir / nombre
        with open(ruta, "rb") as f:
            n = sum(1 for _ in f) - 1  # sin el encabezado
        sha = s4_sha256(ruta)
        filas.append({"archivo": nombre, "filas": n, "sha256": sha,
                      "coincide": sha == sha_esperado and n == n_esperado})
    return pd.DataFrame(filas)


def s4_columnas_atributos(df):
    """Devuelve (numéricas, categóricas): los atributos de `df` sin `id`, `label` ni `attack_cat`.

    Las numéricas conservan el orden de las columnas de `df`. Da `ValueError` si falta `label` o
    alguna de las columnas categóricas."""
    faltan = [c for c in (S4_ETIQUETA, *S4_CATEGORICAS) if c not in df.columns]
    if faltan:
        raise ValueError(f"Faltan columnas: {faltan}")
    numericas = [c for c in df.columns if c not in S4_EXCLUIDAS and c not in S4_CATEGORICAS]
    return numericas, list(S4_CATEGORICAS)


def load_unsw_nb15(data_dir, drop_duplicates=False):
    """Carga y une los dos CSV particionados oficiales de UNSW-NB15.

    Lee `UNSW_NB15_training-set.csv` y `UNSW_NB15_testing-set.csv` de `data_dir` con `utf-8-sig`
    (los archivos empiezan con un BOM), comprueba que tengan las mismas columnas y los concatena con
    un índice nuevo de 0 a n − 1. Con `drop_duplicates=True` elimina las filas idénticas en atributos
    y etiqueta y conserva la primera aparición (D9); por omisión se conservan todas.
    """
    data_dir = Path(data_dir)
    faltan = [n for n in S4_ARCHIVOS if not (data_dir / n).is_file()]
    if faltan:
        raise FileNotFoundError(
            f"Faltan {faltan} en {data_dir}. Ver «Origen de los datos» en §4 para descargarlos.")
    partes = [pd.read_csv(data_dir / n, encoding="utf-8-sig") for n in S4_ARCHIVOS]
    if list(partes[0].columns) != list(partes[1].columns):
        raise ValueError("Los dos CSV no tienen las mismas columnas.")
    df = pd.concat(partes, ignore_index=True)
    numericas, categoricas = s4_columnas_atributos(df)
    if drop_duplicates:
        df = df.drop_duplicates(subset=[*numericas, *categoricas, S4_ETIQUETA]).reset_index(drop=True)
    return df


def split_unsw(df, test_size, seed):
    """Parte `df` en (entrenamiento, prueba), estratificando por `label`.

    `test_size` es la fracción de prueba (0.2 en §4.1 y 0.3 en §4.4) y `seed` fija la partición.
    Los dos `DataFrame` conservan el índice de `df`, de modo que cada fila se puede rastrear."""
    if not 0.0 < test_size < 1.0:
        raise ValueError("test_size debe estar en (0, 1).")
    train_df, test_df = train_test_split(
        df, test_size=test_size, random_state=seed, stratify=df[S4_ETIQUETA])
    return train_df, test_df


def s4_categorias(train_df):
    """Categorías vistas en entrenamiento por cada columna categórica, en orden alfabético."""
    return {c: np.sort(train_df[c].astype(str).unique()) for c in S4_CATEGORICAS}


def s4_nombres_atributos(train_df):
    """Nombres de las columnas de `X` en su orden: numéricas y luego `columna=categoría`."""
    numericas, _ = s4_columnas_atributos(train_df)
    one_hot = [f"{c}={v}" for c, cats in s4_categorias(train_df).items() for v in cats]
    return numericas + one_hot


def s4_one_hot(valores, categorias):
    """Matriz n × len(categorias) de 0/1; un valor que no está en `categorias` deja su fila en ceros."""
    codigos = pd.Index(categorias).get_indexer(np.asarray(valores, dtype=str))  # −1 si no se vio
    M = np.zeros((len(codigos), len(categorias)), dtype=np.float64)
    vistos = codigos >= 0
    M[np.flatnonzero(vistos), codigos[vistos]] = 1.0
    return M


def s4_normalizar_l2(X):
    """Divide cada fila de `X` entre su norma euclídea; una fila de norma 0 se deja igual."""
    normas = np.linalg.norm(X, axis=1, keepdims=True)
    normas[normas == 0] = 1.0
    return X / normas


def s4_etiquetas(df):
    """Devuelve `label` como enteros de forma (n,); da `ValueError` si hay valores distintos de 0 y 1."""
    y = df[S4_ETIQUETA].to_numpy()
    if not np.isin(y, (0, 1)).all():
        raise ValueError("label debe contener solo 0 y 1.")
    return y.astype(np.int64)


def preprocess_unsw(train_df, test_df, l2=False):
    """Convierte las particiones en matrices para la red: one-hot, log1p, z-score y L2 opcional.

    Las categorías, las medias y las desviaciones se ajustan solo con `train_df`:
    1. One-hot de `proto`, `service` y `state` con las categorías de entrenamiento.
    2. log1p de las numéricas cuyo mínimo en entrenamiento es ≥ 0; da `ValueError` si prueba
       tiene negativos en esas columnas.
    3. z-score con la media y la desviación poblacional de entrenamiento (σ = 0 → 1).
    4. Si `l2`, cada fila completa se divide entre su norma euclídea (D8).

    Devuelve `X_train`, `X_test` (float64, n × D: numéricas y luego one-hot) e `y_train`, `y_test`
    (enteros 0/1, forma (n,)). `id`, `label` y `attack_cat` no entran en `X`.
    """
    if list(test_df.columns) != list(train_df.columns):
        raise ValueError("train_df y test_df deben tener las mismas columnas.")
    numericas, _ = s4_columnas_atributos(train_df)

    num_train = train_df[numericas].to_numpy(dtype=np.float64)
    num_test = test_df[numericas].to_numpy(dtype=np.float64)
    con_log = num_train.min(axis=0) >= 0
    if np.any(num_test[:, con_log] < 0):
        raise ValueError("Prueba tiene valores negativos en columnas que en entrenamiento son no negativas.")
    num_train[:, con_log] = np.log1p(num_train[:, con_log])
    num_test[:, con_log] = np.log1p(num_test[:, con_log])

    media = num_train.mean(axis=0)
    desv = num_train.std(axis=0)
    desv[desv == 0] = 1.0
    num_train = (num_train - media) / desv
    num_test = (num_test - media) / desv

    categorias = s4_categorias(train_df)
    X_train = np.hstack([num_train, *(s4_one_hot(train_df[c], cats) for c, cats in categorias.items())])
    X_test = np.hstack([num_test, *(s4_one_hot(test_df[c], cats) for c, cats in categorias.items())])
    if l2:
        X_train, X_test = s4_normalizar_l2(X_train), s4_normalizar_l2(X_test)
    return X_train, X_test, s4_etiquetas(train_df), s4_etiquetas(test_df)


def s4_superposicion(train_df, test_df):
    """Fracción de filas de `test_df` cuyos atributos (sin `id`, `label` ni `attack_cat`) aparecen
    idénticos en alguna fila de `train_df` (D9)."""
    numericas, categoricas = s4_columnas_atributos(train_df)
    columnas = [*numericas, *categoricas]
    h_train = pd.util.hash_pandas_object(train_df[columnas], index=False)
    h_test = pd.util.hash_pandas_object(test_df[columnas], index=False)
    return float(h_test.isin(set(h_train)).mean())

#### Pruebas unitarias de la preparación de datos

Las pruebas no usan los CSV reales: trabajan con `DataFrame` sintéticos de pocas filas, cuyos resultados
se calculan a mano, y con archivos temporales. Así se ejecutan aunque no se hayan descargado los datos, y
cada una comprueba una regla del contrato de datos. Hay 21 pruebas, de dos a cuatro por función:

| Clase | Qué comprueba |
| --- | --- |
| `TestUnswCarga` | `load_unsw_nb15` une los dos archivos con un índice nuevo y lee bien un CSV con BOM; `drop_duplicates=True` quita solo las copias exactas (atributos y etiqueta) y conserva las filas con etiqueta contradictoria; un archivo faltante da `FileNotFoundError` y columnas distintas dan `ValueError`. |
| `TestUnswParticion` | `split_unsw` produce particiones disjuntas que cubren todo, con los tamaños pedidos; conserva la proporción de clases (estratificación) en 80/20 y 70/30; es reproducible con la misma semilla y cambia con otra; rechaza `test_size` fuera de (0, 1). Las listas de semillas son las acordadas. |
| `TestUnswPreprocesamiento` | log1p y z-score coinciden con un cálculo a mano, con media y desviación de entrenamiento aplicadas a prueba; el one-hot usa solo categorías de entrenamiento (una categoría nueva en prueba deja el bloque en ceros) y queda en 0/1; formas y tipos del contrato; `id`, `attack_cat` y `label` no influyen en `X`; σ = 0 no produce `NaN`; una columna con negativos no recibe log1p; negativos en prueba y columnas distintas dan `ValueError`; con `l2=True`, cada fila es la fila sin L2 dividida entre su norma. |
| `TestUnswAuxiliares` | `s4_sha256` reproduce dos sumas conocidas (cadena vacía y `"abc"`) y no depende del tamaño de bloque; `s4_verificar_archivos` detecta archivos distintos de los registrados; `s4_columnas_atributos`, `s4_one_hot`, `s4_normalizar_l2`, `s4_etiquetas` y `s4_superposicion` dan los resultados esperados en casos pequeños. |

Para comprobar que las pruebas detectan errores, se introdujeron a propósito siete errores en el código,
uno por vez: partición sin estratificar, z-score ajustado con prueba, σ = 0 sin reemplazar, log1p en una
columna con negativos, categorías tomadas también de prueba, L2 solo sobre las numéricas y duplicados
comparados incluyendo `id`. Cada uno hizo fallar al menos una prueba.

In [ ]:
import sys
import tempfile
import unittest
from pathlib import Path


def s4_df_sintetico():
    """Seis filas con la estructura de UNSW-NB15: `id`, dos numéricas, las tres categóricas,
    `attack_cat` y `label`. La última fila tiene `proto = "ospf"`, que no aparece en las cuatro primeras."""
    return pd.DataFrame({
        "id": [1, 2, 3, 4, 5, 6],
        "dur": [0.0, 1.0, 3.0, 7.0, 1.0, 0.5],
        "sbytes": [10, 20, 30, 40, 50, 60],
        "proto": ["tcp", "udp", "tcp", "arp", "udp", "ospf"],
        "service": ["-", "http", "-", "dns", "http", "-"],
        "state": ["FIN", "INT", "FIN", "CON", "INT", "FIN"],
        "attack_cat": ["Normal", "Generic", "Normal", "DoS", "Generic", "Normal"],
        "label": [0, 1, 0, 1, 1, 0],
    })


class TestUnswCarga(unittest.TestCase):
    @staticmethod
    def _escribir(carpeta, a, b):
        a.to_csv(Path(carpeta) / S4_ARCHIVOS[0], index=False, encoding="utf-8-sig")
        b.to_csv(Path(carpeta) / S4_ARCHIVOS[1], index=False, encoding="utf-8-sig")

    def test_une_archivos_y_quita_bom(self):
        df = s4_df_sintetico()
        with tempfile.TemporaryDirectory() as d:
            self._escribir(d, df.iloc[:4], df.iloc[4:])
            self.assertTrue((Path(d) / S4_ARCHIVOS[0]).read_bytes().startswith(b"\xef\xbb\xbf"))
            cargado = load_unsw_nb15(d)
        self.assertEqual(cargado.columns[0], "id")  # sin el BOM
        self.assertEqual(list(cargado.index), list(range(6)))  # índice nuevo de 0 a n − 1
        pd.testing.assert_frame_equal(cargado, df, check_dtype=False)

    def test_drop_duplicates_quita_solo_copias_exactas(self):
        df = s4_df_sintetico()
        copia = df.iloc[[0]].assign(id=99)  # mismos atributos y etiqueta, otro id
        contradictoria = df.iloc[[1]].assign(id=98, label=0, attack_cat="Normal")  # otra etiqueta
        with tempfile.TemporaryDirectory() as d:
            self._escribir(d, df, pd.concat([copia, contradictoria]))
            todas = load_unsw_nb15(d)
            sin_duplicados = load_unsw_nb15(d, drop_duplicates=True)
        self.assertEqual(len(todas), 8)
        self.assertEqual(len(sin_duplicados), 7)
        self.assertNotIn(99, sin_duplicados["id"].tolist())
        self.assertIn(98, sin_duplicados["id"].tolist())
        self.assertEqual(list(sin_duplicados.index), list(range(7)))

    def test_archivo_faltante(self):
        with tempfile.TemporaryDirectory() as d:
            s4_df_sintetico().to_csv(Path(d) / S4_ARCHIVOS[0], index=False)
            with self.assertRaises(FileNotFoundError):
                load_unsw_nb15(d)

    def test_columnas_distintas(self):
        df = s4_df_sintetico()
        with tempfile.TemporaryDirectory() as d:
            self._escribir(d, df, df.drop(columns="dur"))
            with self.assertRaises(ValueError):
                load_unsw_nb15(d)


class TestUnswParticion(unittest.TestCase):
    @staticmethod
    def _df(n=100, positivos=30):
        return pd.DataFrame({
            "id": range(n), "dur": np.arange(n, dtype=float), "proto": "tcp", "service": "-",
            "state": "FIN", "attack_cat": "x", "label": [1] * positivos + [0] * (n - positivos)})

    def test_tamanos_disjuntas_y_completas(self):
        df = self._df()
        train, test = split_unsw(df, 0.2, seed=4000)
        self.assertEqual((len(train), len(test)), (80, 20))
        self.assertEqual(set(train.index) & set(test.index), set())
        self.assertEqual(set(train.index) | set(test.index), set(df.index))

    def test_estratifica_por_label(self):
        # 30 % de positivos en el total → 30 % en cada lado.
        train, test = split_unsw(self._df(), 0.2, seed=4000)
        self.assertEqual((int(train["label"].sum()), int(test["label"].sum())), (24, 6))
        train, test = split_unsw(self._df(), 0.3, seed=4500)
        self.assertEqual((int(train["label"].sum()), int(test["label"].sum())), (21, 9))

    def test_reproducible_por_semilla(self):
        df = self._df()
        a, _ = split_unsw(df, 0.2, 4000)
        b, _ = split_unsw(df, 0.2, 4000)
        c, _ = split_unsw(df, 0.2, 4001)
        self.assertEqual(list(a.index), list(b.index))
        self.assertNotEqual(sorted(a.index), sorted(c.index))

    def test_rechaza_test_size_invalido(self):
        for test_size in (0.0, 1.0, 1.5):
            with self.assertRaises(ValueError):
                split_unsw(self._df(), test_size, 4000)

    def test_semillas_acordadas(self):
        self.assertEqual(S4_SEMILLAS_80_20, list(range(4000, 4030)))
        self.assertEqual(S4_SEMILLAS_70_30, list(range(4500, 4510)))


class TestUnswPreprocesamiento(unittest.TestCase):
    def setUp(self):
        df = s4_df_sintetico()
        self.train, self.test = df.iloc[:4], df.iloc[4:]

    def test_log1p_y_zscore_a_mano(self):
        X_train, X_test, _, _ = preprocess_unsw(self.train, self.test)
        # dur = 0, 1, 3, 7 → log1p(dur) = ln 2 · (0, 1, 2, 3): media 1.5 · ln 2, desviación √1.25 · ln 2.
        np.testing.assert_allclose(X_train[:, 0], (np.array([0, 1, 2, 3]) - 1.5) / np.sqrt(1.25))
        # Prueba usa la media y la desviación de entrenamiento: dur = 1 y 0.5 → log₂(2) y log₂(1.5).
        np.testing.assert_allclose(X_test[:, 0], (np.log2([2.0, 1.5]) - 1.5) / np.sqrt(1.25))
        np.testing.assert_allclose(X_train[:, :2].mean(axis=0), 0.0, atol=1e-12)
        np.testing.assert_allclose(X_train[:, :2].std(axis=0), 1.0)

    def test_one_hot_con_categorias_de_entrenamiento(self):
        nombres = s4_nombres_atributos(self.train)
        self.assertEqual(nombres, [
            "dur", "sbytes", "proto=arp", "proto=tcp", "proto=udp", "service=-", "service=dns",
            "service=http", "state=CON", "state=FIN", "state=INT"])
        X_train, X_test, _, _ = preprocess_unsw(self.train, self.test)
        self.assertEqual(X_train.shape[1], len(nombres))
        self.assertTrue(np.isin(X_train[:, 2:], (0.0, 1.0)).all())  # one-hot sin log1p ni z-score
        np.testing.assert_array_equal(X_train[:, 2:5].sum(axis=1), 1.0)  # un 1 por bloque
        np.testing.assert_array_equal(X_test[0, 2:5], [0, 0, 1])  # udp
        np.testing.assert_array_equal(X_test[1, 2:5], [0, 0, 0])  # ospf: solo en prueba

    def test_formas_tipos_y_columnas_excluidas(self):
        X_train, X_test, y_train, y_test = preprocess_unsw(self.train, self.test)
        self.assertEqual((X_train.shape, X_test.shape), ((4, 11), (2, 11)))
        self.assertEqual((X_train.dtype, y_train.dtype), (np.float64, np.int64))
        self.assertEqual(y_train.shape, (4,))
        np.testing.assert_array_equal(y_test, [1, 0])
        # Cambiar id, attack_cat o label no cambia X: no son atributos.
        otro = self.train.assign(id=[7, 8, 9, 10], attack_cat="x", label=[1, 1, 0, 0])
        X_train2, X_test2, y_train2, _ = preprocess_unsw(otro, self.test)
        np.testing.assert_array_equal(X_train, X_train2)
        np.testing.assert_array_equal(X_test, X_test2)
        np.testing.assert_array_equal(y_train2, [1, 1, 0, 0])

    def test_desviacion_cero_y_columna_con_negativos(self):
        train = self.train.assign(sbytes=5, dur=[-1.0, 0.0, 1.0, 2.0])
        test = self.test.assign(sbytes=5, dur=[-3.0, 0.5])
        X_train, X_test, _, _ = preprocess_unsw(train, test)
        # dur tiene negativos en entrenamiento: solo z-score (media 0.5, desviación √1.25).
        np.testing.assert_allclose(X_train[:, 0], (np.array([-1, 0, 1, 2]) - 0.5) / np.sqrt(1.25))
        np.testing.assert_allclose(X_test[:, 0], (np.array([-3.0, 0.5]) - 0.5) / np.sqrt(1.25))
        # sbytes es constante: σ = 0 → 1, la columna queda en ceros y no en NaN.
        np.testing.assert_array_equal(X_train[:, 1], 0.0)
        np.testing.assert_array_equal(X_test[:, 1], 0.0)

    def test_rechaza_negativos_en_prueba_y_columnas_distintas(self):
        with self.assertRaises(ValueError):
            preprocess_unsw(self.train, self.test.assign(sbytes=[50, -1]))
        with self.assertRaises(ValueError):
            preprocess_unsw(self.train, self.test.drop(columns="dur"))

    def test_l2_divide_cada_fila_entre_su_norma(self):
        X_train, X_test, _, _ = preprocess_unsw(self.train, self.test, l2=False)
        X_train_l2, X_test_l2, _, _ = preprocess_unsw(self.train, self.test, l2=True)
        np.testing.assert_allclose(np.linalg.norm(X_train_l2, axis=1), 1.0)
        np.testing.assert_allclose(X_train_l2, X_train / np.linalg.norm(X_train, axis=1, keepdims=True))
        np.testing.assert_allclose(X_test_l2, X_test / np.linalg.norm(X_test, axis=1, keepdims=True))


class TestUnswAuxiliares(unittest.TestCase):
    def test_sha256_vectores_conocidos(self):
        with tempfile.TemporaryDirectory() as d:
            vacio, abc = Path(d) / "vacio", Path(d) / "abc"
            vacio.write_bytes(b"")
            abc.write_bytes(b"abc")
            self.assertEqual(s4_sha256(vacio),
                             "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855")
            esperado = "ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad"
            self.assertEqual(s4_sha256(abc), esperado)
            self.assertEqual(s4_sha256(abc, bloque=2), esperado)  # leer por bloques no cambia la suma

    def test_verificar_archivos_detecta_diferencias(self):
        with tempfile.TemporaryDirectory() as d:
            for nombre in S4_ARCHIVOS:
                (Path(d) / nombre).write_text("id,label\n1,0\n2,1\n")
            tabla = s4_verificar_archivos(d)
        self.assertEqual(tabla["filas"].tolist(), [2, 2])
        self.assertFalse(tabla["coincide"].any())

    def test_columnas_atributos(self):
        numericas, categoricas = s4_columnas_atributos(s4_df_sintetico())
        self.assertEqual((numericas, categoricas), (["dur", "sbytes"], ["proto", "service", "state"]))
        with self.assertRaises(ValueError):
            s4_columnas_atributos(s4_df_sintetico().drop(columns="state"))

    def test_one_hot_y_l2(self):
        np.testing.assert_array_equal(
            s4_one_hot(["b", "z", "a"], np.array(["a", "b"])), [[0, 1], [0, 0], [1, 0]])
        np.testing.assert_allclose(s4_normalizar_l2(np.array([[3.0, 4.0], [0.0, 0.0]])),
                                   [[0.6, 0.8], [0.0, 0.0]])

    def test_etiquetas(self):
        y = s4_etiquetas(pd.DataFrame({"label": [0, 1, 1]}))
        self.assertEqual((y.dtype, y.shape), (np.int64, (3,)))
        with self.assertRaises(ValueError):
            s4_etiquetas(pd.DataFrame({"label": [0, 2]}))

    def test_superposicion(self):
        df = s4_df_sintetico()
        train = df.iloc[:4]
        # Fila 0 repetida con otro id, otra etiqueta y otra categoría de ataque: cuenta, porque la red
        # solo ve los atributos. La fila 5 no aparece en entrenamiento.
        test = pd.concat([df.iloc[[0]].assign(id=50, label=1, attack_cat="DoS"), df.iloc[[5]]])
        self.assertAlmostEqual(s4_superposicion(train, test), 0.5)
        # La fila 4 coincide con la 1 salvo en sbytes (50 frente a 20): no cuenta.
        self.assertAlmostEqual(s4_superposicion(train, df.iloc[4:]), 0.0)


suite_s4_datos = unittest.TestSuite()
for caso in (TestUnswCarga, TestUnswParticion, TestUnswPreprocesamiento, TestUnswAuxiliares):
    suite_s4_datos.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(caso))
resultado_s4_datos = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite_s4_datos)
assert resultado_s4_datos.wasSuccessful(), "Fallaron pruebas unitarias de la preparación de datos (§4)"

#### Carga y verificación de los datos

La celda siguiente:

1. comprueba que los CSV de `data/raw/unsw_nb15/` tengan las sumas SHA-256 y los conteos registrados, y se
   detiene si no coinciden;
2. carga los datos en `s4_unsw`, el `DataFrame` que reutilizan §4.1, §4.2 y §4.4, y resume las clases, las
   categorías, los valores faltantes y los negativos;
3. cuenta los duplicados y mide, en las 30 particiones 80/20 y las 10 particiones 70/30, qué fracción de las
   filas de prueba tiene una copia exacta de sus atributos en entrenamiento (D9);
4. preprocesa la primera partición 80/20 con y sin L2 y comprueba las formas, que no haya `NaN`, que las
   numéricas de entrenamiento queden con media 0 y desviación 1 y que con L2 las filas tengan norma 1.

In [ ]:
import time

# 1. Archivos descargados frente a lo registrado en T3.
s4_verificacion = s4_verificar_archivos(UNSW_DIR)
print(s4_verificacion.to_string(index=False))
assert s4_verificacion["coincide"].all(), "Los CSV no coinciden con las sumas o los conteos registrados en T3."

# 2. Carga y resumen.
s4_unsw = load_unsw_nb15(UNSW_DIR)
s4_numericas, s4_categoricas = s4_columnas_atributos(s4_unsw)
print(f"\n{len(s4_unsw)} filas · {len(s4_numericas)} atributos numéricos + {len(s4_categoricas)} categóricos")
s4_clases = s4_unsw["label"].value_counts().sort_index()
print(pd.DataFrame({"filas": s4_clases, "proporción": (s4_clases / len(s4_unsw)).round(4)})
      .rename(index={0: "normal (0)", 1: "ataque (1)"}).to_string())
print("Categorías por columna:", {c: s4_unsw[c].nunique() for c in s4_categoricas})
print("Valores faltantes:", int(s4_unsw.isna().sum().sum()),
      "· numéricas con negativos:", [c for c in s4_numericas if (s4_unsw[c] < 0).any()])

# 3. Duplicados y superposición entre entrenamiento y prueba (D9).
s4_n_duplicados = int(s4_unsw.duplicated(subset=[*s4_numericas, *s4_categoricas, "label"]).sum())
s4_n_contradictorios = int(
    (s4_unsw.groupby([*s4_numericas, *s4_categoricas])["label"].nunique() > 1).sum())
print(f"\nFilas duplicadas (atributos y etiqueta): {s4_n_duplicados} ({s4_n_duplicados / len(s4_unsw):.1%})"
      f" · grupos de atributos con etiquetas distintas: {s4_n_contradictorios}")
s4_superposiciones = pd.DataFrame(
    [{"partición": "80/20 (§4.1)", "semilla": s, "superposición": s4_superposicion(*split_unsw(s4_unsw, 0.2, s))}
     for s in S4_SEMILLAS_80_20]
    + [{"partición": "70/30 (§4.4)", "semilla": s, "superposición": s4_superposicion(*split_unsw(s4_unsw, 0.3, s))}
       for s in S4_SEMILLAS_70_30])
print("Fracción de filas de prueba con copia exacta de sus atributos en entrenamiento:")
print(s4_superposiciones.groupby("partición")["superposición"]
      .agg(["count", "mean", "std", "min", "max"]).round(4).to_string())

# 4. Preprocesamiento de la primera partición 80/20, con y sin L2.
s4_train_df, s4_test_df = split_unsw(s4_unsw, 0.2, S4_SEMILLAS_80_20[0])
print(f"\nPartición 80/20 con semilla {S4_SEMILLAS_80_20[0]}: {len(s4_train_df)} de entrenamiento y "
      f"{len(s4_test_df)} de prueba; ataques {s4_train_df['label'].mean():.4f} y {s4_test_df['label'].mean():.4f}")
for s4_l2 in (False, True):
    s4_inicio = time.perf_counter()
    s4_X_train, s4_X_test, s4_y_train, s4_y_test = preprocess_unsw(s4_train_df, s4_test_df, l2=s4_l2)
    s4_segundos = time.perf_counter() - s4_inicio
    s4_n_num = len(s4_numericas)
    assert s4_X_train.shape[1] == len(s4_nombres_atributos(s4_train_df))
    assert not (np.isnan(s4_X_train).any() or np.isnan(s4_X_test).any())
    assert s4_y_train.shape == (len(s4_train_df),) and s4_y_train.dtype == np.int64
    if s4_l2:
        assert np.allclose(np.linalg.norm(s4_X_train, axis=1), 1.0)
        assert np.allclose(np.linalg.norm(s4_X_test, axis=1), 1.0)
    else:
        assert np.allclose(s4_X_train[:, :s4_n_num].mean(axis=0), 0.0, atol=1e-9)
        assert np.allclose(s4_X_train[:, :s4_n_num].std(axis=0), 1.0)
    print(f"l2={s4_l2}: X_train {s4_X_train.shape}, X_test {s4_X_test.shape} "
          f"({s4_n_num} numéricas + {s4_X_train.shape[1] - s4_n_num} one-hot), {s4_segundos:.2f} s")
del s4_X_train, s4_X_test, s4_y_train, s4_y_test  # ~0.4 GB; cada sección genera sus propias matrices

### 4.1 Clasificador UNSW-NB15 y normalización L2

*Responsable: P1 (T4).*

#### Planteamiento y protocolo

El detector de ataques es la red de §1 entrenada sobre UNSW-NB15 con la preparación de datos de T3
(`split_unsw` y `preprocess_unsw`): todos los atributos, con one-hot de `proto`, `service` y `state`,
log1p y z-score ajustados con entrenamiento. Se comparan dos tratamientos que difieren solo en el
último paso del preprocesamiento:

- a) **con** normalización L2 por fila (cada observación se divide entre su norma euclídea, D8);
- b) **sin** normalización L2.

El protocolo de la comparación se fijó en T0, antes de ver resultados («Comparación con y sin L2» en
`JOURNAL.md`). T4 acordó además los detalles de calibración, épocas y salida:

| Aspecto | Decisión |
| --- | --- |
| Salida | $K = 1$: softmax$([p, 0])$ = sigmoide$(p)$ (D3), con `T = y.reshape(-1, 1)`; predicción ataque si $y \ge 0.5$ |
| Calibración | Una vez por tratamiento, con la **misma rejilla** para ambos: $M \in \{10, 20\}$, $\alpha \in \{0.01, 0.1, 0.5\}$, $\gamma \in \{0, 0.9\}$, `max_weights` = 1, 100 épocas (12 configuraciones). Se entrena con el 80 % del entrenamiento de la primera partición 80/20 de T3 y se valida con el 20 % restante; el 20 % de prueba no se usa. Gana el mayor F1 de validación de la clase ataque; a igual F1, la de menos neuronas ocultas |
| Corridas | 30, con las particiones 80/20 estratificadas de T3 (`S4_SEMILLAS_80_20`); 100 épocas en lote completo sobre el 80 % de entrenamiento |
| Emparejamiento | La corrida $i$ usa en los dos tratamientos la misma partición y la misma semilla de pesos iniciales (`SEEDS["s4_1_l2"] + 100 + i`). Los pesos iniciales son idénticos si ambos tratamientos eligen el mismo $M$; si no, comparten la semilla |
| Métricas | F1 de la clase ataque (principal) y tasa de aciertos, sobre el 20 % de prueba; también precisión y *recall* |
| Prueba | Wilcoxon de rangos con signo, bilateral, sobre las 30 diferencias emparejadas de F1 (principal); prueba t pareada como complemento, con Shapiro-Wilk sobre las diferencias para su supuesto de normalidad; $\alpha = 0.05$ |
| Decisión | Se usa L2 solo si Wilcoxon da $p < 0.05$ **y** la mediana de las diferencias (con L2 − sin L2) es positiva. Si no hay diferencia significativa, se usa la opción más simple, sin L2. La decisión se registra en `USE_L2` (`s0-config`) y se aplica a §4.2 y §4.4 |

**Limitaciones que se tienen en cuenta al analizar.**

- *Calibración compartida.* La calibración usa filas que en otras corridas caen en la partición de
  prueba. El sesgo optimista que eso introduce afecta por igual a los dos tratamientos, así que no
  favorece a ninguno en la comparación. Calibrar dentro de cada corrida costaría unas 30 veces más.
- *Duplicados (D9).* Casi la mitad de las filas de prueba tienen una copia exacta de sus atributos en
  entrenamiento, por lo que la tasa de aciertos y el F1 absolutos son optimistas. El efecto es el mismo
  en ambos tratamientos, porque comparten las particiones.

**Costo.** Una época sobre unas 206 000 filas y 195 atributos tarda alrededor de 0.1 s en un Apple M2
Pro. La sección completa, calibración incluida, tarda del orden de 15 a 20 minutos. Con `TP3_QUICK=1` se
usan 10 épocas y 3 corridas, solo para comprobar que el código se ejecuta.

#### Funciones auxiliares y sus pruebas

| Función | Qué hace |
| --- | --- |
| `s4_1_metricas(y, pred)` | Tasa de aciertos, precisión, *recall* y F1 de la clase ataque (1) a partir de etiquetas 0/1. |
| `s4_1_entrenar_y_evaluar(X_train, y_train, X_eval, y_eval, M, alpha, gamma, max_weights, num_epochs, seed, curva_eval)` | Entrena una red $[D, M, 1]$ con `train_mlp` y devuelve la red, el MSE por época (también el de evaluación si `curva_eval`), las métricas sobre la evaluación y el tiempo. |
| `s4_1_comparar(f1_con_l2, f1_sin_l2, alpha)` | Calcula las diferencias emparejadas y aplica Wilcoxon, la prueba t pareada, Shapiro-Wilk y la regla de decisión de L2. |

| Función | Prueba | Resultado esperado |
| --- | --- | --- |
| `s4_1_metricas` | $y = [1,1,1,0,0]$, predicción $[1,1,0,1,0]$ | VP = 2, FP = 1, FN = 1: precisión, *recall* y F1 = 2/3; aciertos 0.6 |
| | 200 etiquetas aleatorias | Igual a `f1_score` y `accuracy_score` de scikit-learn |
| | Ninguna predicción de ataque | Precisión, *recall* y F1 = 0, sin división entre cero |
| | Longitudes distintas o etiqueta 2 | `ValueError` |
| `s4_1_entrenar_y_evaluar` | Dos nubes separables en $\mathbb R^3$, 40 épocas, con curva | Curvas de 40 valores, F1 > 0.95, red $[3, 4, 1]$ |
| | Dos llamadas con la misma semilla, sin curva | Pesos, errores y F1 idénticos; curva de evaluación vacía |
| `s4_1_comparar` | F1 con L2 = F1 sin L2 + 0.01 + ruido pequeño | Wilcoxon y t con $p < 0.05$; `usar_l2` verdadero |
| | F1 con L2 = F1 sin L2 − 0.01 + ruido | $p < 0.05$, pero `usar_l2` falso (L2 es peor) |
| | Solo ruido simétrico, y tratamientos idénticos | $p > 0.05$ y `usar_l2` falso; con diferencias nulas, $p = 1$ |
| | Longitudes distintas | `ValueError` |

In [ ]:
import itertools
import time

import matplotlib.pyplot as plt
import matplotlib.ticker
from scipy import stats

s4_1_colores = {False: "#2a78d6", True: "#eb6834"}   # sin L2 y con L2 (§4.1)
s4_1_colores_gamma = {0.0: "#2a78d6", 0.9: "#eb6834"}


def s4_1_metricas(y, pred):
    """Tasa de aciertos y métricas de la clase ataque (1) a partir de etiquetas reales y predichas.

    Devuelve un dict con 'tasa_aciertos', 'precision', 'recall' y 'f1_ataque'. Si un cociente
    tiene denominador 0 (por ejemplo, ninguna predicción de ataque), esa métrica vale 0.
    """
    y = np.asarray(y).ravel()
    pred = np.asarray(pred).ravel()
    if y.shape != pred.shape or y.size == 0:
        raise ValueError("y y pred deben ser vectores no vacíos de la misma longitud")
    if not (np.isin(y, (0, 1)).all() and np.isin(pred, (0, 1)).all()):
        raise ValueError("y y pred deben contener solo 0 y 1")
    vp = int(np.sum((pred == 1) & (y == 1)))
    fp = int(np.sum((pred == 1) & (y == 0)))
    fn = int(np.sum((pred == 0) & (y == 1)))
    cociente = lambda a, b: a / b if b > 0 else 0.0
    return {
        "tasa_aciertos": float(np.mean(pred == y)),
        "precision": cociente(vp, vp + fp),
        "recall": cociente(vp, vp + fn),
        "f1_ataque": cociente(2 * vp, 2 * vp + fp + fn),
    }


def s4_1_entrenar_y_evaluar(X_train, y_train, X_eval, y_eval, M, alpha, gamma, max_weights,
                            num_epochs, seed, curva_eval=False):
    """Entrena una red [D, M, 1] (salida softmax([p, 0]), D3) y la evalúa en (X_eval, y_eval).

    Entrena `num_epochs` épocas en lote completo con `train_mlp` y AdaGrad con momentum. Si
    `curva_eval`, `train_mlp` mide además el MSE de (X_eval, y_eval) en cada época (solo para
    graficar: no influye en el entrenamiento).

    Retorna
    -------
    dict con 'red', 'train_error' y 'eval_error' (MSE por época; 'eval_error' vacía sin
    `curva_eval`), las métricas de `s4_1_metricas` sobre (X_eval, y_eval) y 'segundos'
    (tiempo de entrenamiento).
    """
    T_train = np.asarray(y_train).reshape(-1, 1).astype(float)
    T_eval = np.asarray(y_eval).reshape(-1, 1).astype(float)
    red = Multilayer_perceptron([X_train.shape[1], M, 1], alpha, max_weights, gamma=gamma, seed=seed)
    inicio = time.perf_counter()
    if curva_eval:
        historial = red.train_mlp(num_epochs, X_train, T_train, alpha, gamma, X_val=X_eval, T_val=T_eval)
    else:
        historial = red.train_mlp(num_epochs, X_train, T_train, alpha, gamma)
    segundos = time.perf_counter() - inicio
    return {"red": red, "train_error": historial["train_error"], "eval_error": historial["val_error"],
            **s4_1_metricas(y_eval, red.predict(X_eval)), "segundos": segundos}


def s4_1_comparar(f1_con_l2, f1_sin_l2, alpha=0.05):
    """Compara dos tratamientos emparejados y aplica la regla de decisión de L2 acordada en T0.

    Con las diferencias d = F1 con L2 − F1 sin L2 de cada corrida, aplica la prueba de Wilcoxon
    de rangos con signo (bilateral, prueba principal), la prueba t pareada (bilateral,
    complemento) y la de Shapiro-Wilk sobre d (supuesto de normalidad de la prueba t). Se usa L2
    solo si Wilcoxon da p < `alpha` y la mediana de d es positiva (L2 significativamente mejor).

    Retorna un dict con 'diferencias', 'media', 'mediana', 'wilcoxon_estadistico', 'wilcoxon_p',
    't_estadistico', 't_p', 'shapiro_p' y 'usar_l2'.
    """
    a = np.asarray(f1_con_l2, dtype=float).ravel()
    b = np.asarray(f1_sin_l2, dtype=float).ravel()
    if a.shape != b.shape or a.size < 2:
        raise ValueError("Se necesitan al menos dos corridas emparejadas por tratamiento")
    d = a - b
    if np.all(d == 0):  # tratamientos idénticos: no hay diferencia que probar
        w_est, w_p, t_est, t_p = 0.0, 1.0, 0.0, 1.0
    else:
        w = stats.wilcoxon(a, b, alternative="two-sided")
        t = stats.ttest_rel(a, b, alternative="two-sided")
        w_est, w_p, t_est, t_p = float(w.statistic), float(w.pvalue), float(t.statistic), float(t.pvalue)
    shapiro_p = float(stats.shapiro(d).pvalue) if d.size >= 3 and np.ptp(d) > 0 else np.nan
    return {
        "diferencias": d, "media": float(d.mean()), "mediana": float(np.median(d)),
        "wilcoxon_estadistico": w_est, "wilcoxon_p": w_p, "t_estadistico": t_est, "t_p": t_p,
        "shapiro_p": shapiro_p, "usar_l2": bool(w_p < alpha and np.median(d) > 0),
    }

In [ ]:
import sys
import unittest

from sklearn.metrics import accuracy_score, f1_score


class TestL2Metricas(unittest.TestCase):
    def test_calculo_a_mano(self):
        # VP = 2, FP = 1, FN = 1, VN = 1: precisión 2/3, recall 2/3, F1 = 4/6, aciertos 3/5.
        m = s4_1_metricas([1, 1, 1, 0, 0], [1, 1, 0, 1, 0])
        self.assertAlmostEqual(m["tasa_aciertos"], 0.6)
        self.assertAlmostEqual(m["precision"], 2 / 3)
        self.assertAlmostEqual(m["recall"], 2 / 3)
        self.assertAlmostEqual(m["f1_ataque"], 2 / 3)

    def test_coincide_con_sklearn(self):
        rng = np.random.default_rng(SEEDS["s4_1_l2"])
        y, pred = rng.integers(0, 2, 200), rng.integers(0, 2, 200)
        m = s4_1_metricas(y, pred)
        self.assertAlmostEqual(m["f1_ataque"], f1_score(y, pred, pos_label=1))
        self.assertAlmostEqual(m["tasa_aciertos"], accuracy_score(y, pred))

    def test_sin_predicciones_de_ataque(self):
        m = s4_1_metricas([1, 0, 1], [0, 0, 0])
        self.assertEqual((m["precision"], m["recall"], m["f1_ataque"]), (0.0, 0.0, 0.0))

    def test_rechaza_entradas_invalidas(self):
        with self.assertRaises(ValueError):
            s4_1_metricas([0, 1], [0, 1, 1])
        with self.assertRaises(ValueError):
            s4_1_metricas([0, 2], [0, 1])


class TestL2EntrenarYEvaluar(unittest.TestCase):
    @staticmethod
    def _datos(rng, n=60):
        """Dos nubes separables en R³ con etiquetas 0/1."""
        X = np.vstack([rng.normal(-1.0, 0.5, (n, 3)), rng.normal(1.0, 0.5, (n, 3))])
        return X, np.repeat([0, 1], n)

    def test_aprende_problema_separable(self):
        rng = np.random.default_rng(SEEDS["s4_1_l2"])
        X, y = self._datos(rng)
        X_ev, y_ev = self._datos(rng, 30)
        r = s4_1_entrenar_y_evaluar(X, y, X_ev, y_ev, M=4, alpha=0.1, gamma=0.9, max_weights=1.0,
                                    num_epochs=40, seed=1, curva_eval=True)
        self.assertEqual((len(r["train_error"]), len(r["eval_error"])), (40, 40))
        self.assertGreater(r["f1_ataque"], 0.95)
        self.assertEqual(r["red"].neurons_per_layer, [3, 4, 1])

    def test_reproducible_y_sin_curva(self):
        rng = np.random.default_rng(SEEDS["s4_1_l2"])
        X, y = self._datos(rng, 20)
        a = s4_1_entrenar_y_evaluar(X, y, X, y, 3, 0.1, 0.0, 1.0, 5, seed=7)
        b = s4_1_entrenar_y_evaluar(X, y, X, y, 3, 0.1, 0.0, 1.0, 5, seed=7)
        np.testing.assert_array_equal(a["red"].W_o, b["red"].W_o)
        self.assertEqual(a["train_error"], b["train_error"])
        self.assertEqual(a["eval_error"], [])
        self.assertEqual(a["f1_ataque"], b["f1_ataque"])


class TestL2Comparar(unittest.TestCase):
    def setUp(self):
        self.base = np.random.default_rng(SEEDS["s4_1_l2"]).uniform(0.90, 0.95, 30)
        self.ruido = np.random.default_rng(SEEDS["s4_1_l2"] + 1).normal(0, 0.001, 30)

    def test_l2_claramente_mejor(self):
        r = s4_1_comparar(self.base + 0.01 + self.ruido, self.base)
        self.assertLess(r["wilcoxon_p"], 0.05)
        self.assertLess(r["t_p"], 0.05)
        self.assertTrue(r["usar_l2"])
        np.testing.assert_allclose(r["diferencias"], 0.01 + self.ruido)

    def test_l2_claramente_peor(self):
        r = s4_1_comparar(self.base - 0.01 + self.ruido, self.base)
        self.assertLess(r["wilcoxon_p"], 0.05)
        self.assertFalse(r["usar_l2"])  # significativo, pero a favor de no usar L2

    def test_sin_diferencia(self):
        r = s4_1_comparar(self.base + self.ruido, self.base)
        self.assertGreater(r["wilcoxon_p"], 0.05)
        self.assertFalse(r["usar_l2"])
        iguales = s4_1_comparar(self.base, self.base)
        self.assertEqual((iguales["wilcoxon_p"], iguales["usar_l2"]), (1.0, False))

    def test_rechaza_longitudes_distintas(self):
        with self.assertRaises(ValueError):
            s4_1_comparar([0.9, 0.8, 0.7], [0.9, 0.8])


suite_s4_1 = unittest.TestSuite()
for caso in (TestL2Metricas, TestL2EntrenarYEvaluar, TestL2Comparar):
    suite_s4_1.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(caso))
resultado_s4_1 = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite_s4_1)
assert resultado_s4_1.wasSuccessful(), "Fallaron pruebas unitarias de §4.1"

#### Calibración de hiperparámetros

Cada tratamiento entrena las 12 configuraciones de la rejilla con la misma partición de calibración y
la misma semilla de pesos (`SEEDS["s4_1_l2"]`). El preprocesamiento se ajusta solo con la parte de
entrenamiento de la calibración. La tabla da, por configuración:

- el F1 de la clase ataque y la tasa de aciertos en la validación;
- el MSE de validación y de entrenamiento en la última época;
- el tiempo.

La figura siguiente muestra el MSE de validación por época de cada configuración. La estrella marca la
elegida en cada tratamiento.

In [ ]:
import pandas as pd

# Calibración (una vez por tratamiento): validación estratificada 80/20 dentro del entrenamiento de la
# primera partición 80/20 de T3. El 20 % de prueba de esa partición no se usa.
s4_1_valores_M = [10, 20]
s4_1_valores_alpha = [0.01, 0.1, 0.5]
s4_1_valores_gamma = [0.0, 0.9]
s4_1_max_weights = 1.0
s4_1_epocas_calibracion = quick_or_full(10, 100)
s4_1_semilla_pesos_calibracion = SEEDS["s4_1_l2"]

s4_1_train_df, _ = split_unsw(s4_unsw, 0.2, S4_SEMILLAS_80_20[0])
s4_1_cal_train_df, s4_1_cal_val_df = split_unsw(s4_1_train_df, 0.2, SEEDS["s4_1_l2"])
print(f"Calibración: partición {S4_SEMILLAS_80_20[0]}; {len(s4_1_cal_train_df)} filas para entrenar y "
      f"{len(s4_1_cal_val_df)} de validación; {s4_1_epocas_calibracion} épocas por configuración")

s4_1_curvas_cal = {}  # (l2, M, alpha, gamma) -> MSE de validación por época
s4_1_filas_cal = []
for s4_1_l2 in (False, True):
    s4_1_Xa, s4_1_Xv, s4_1_ya, s4_1_yv = preprocess_unsw(s4_1_cal_train_df, s4_1_cal_val_df, l2=s4_1_l2)
    for s4_1_M, s4_1_alpha, s4_1_gamma in itertools.product(s4_1_valores_M, s4_1_valores_alpha, s4_1_valores_gamma):
        s4_1_r = s4_1_entrenar_y_evaluar(s4_1_Xa, s4_1_ya, s4_1_Xv, s4_1_yv, s4_1_M, s4_1_alpha, s4_1_gamma,
                                         s4_1_max_weights, s4_1_epocas_calibracion,
                                         seed=s4_1_semilla_pesos_calibracion, curva_eval=True)
        s4_1_curvas_cal[(s4_1_l2, s4_1_M, s4_1_alpha, s4_1_gamma)] = np.array(s4_1_r["eval_error"])
        s4_1_filas_cal.append({
            "l2": s4_1_l2, "M": s4_1_M, "alpha": s4_1_alpha, "gamma": s4_1_gamma,
            "f1_ataque_val": s4_1_r["f1_ataque"], "tasa_aciertos_val": s4_1_r["tasa_aciertos"],
            "mse_val": s4_1_r["eval_error"][-1], "mse_train": s4_1_r["train_error"][-1],
            "segundos": s4_1_r["segundos"],
        })
    del s4_1_Xa, s4_1_Xv

# Selección por tratamiento: mayor F1 de validación; a igualdad, menos neuronas ocultas.
s4_1_calibracion = pd.DataFrame(s4_1_filas_cal)
s4_1_elegidas = {}
for s4_1_l2, s4_1_grupo in s4_1_calibracion.groupby("l2"):
    s4_1_mejor = s4_1_grupo.sort_values(["f1_ataque_val", "M"], ascending=[False, True]).iloc[0]
    s4_1_elegidas[bool(s4_1_l2)] = {"M": int(s4_1_mejor["M"]), "alpha": float(s4_1_mejor["alpha"]),
                                    "gamma": float(s4_1_mejor["gamma"]), "max_weights": s4_1_max_weights}
for s4_1_l2, s4_1_hp in s4_1_elegidas.items():
    print(f"{'Con' if s4_1_l2 else 'Sin'} L2: M = {s4_1_hp['M']}, α = {s4_1_hp['alpha']:g}, "
          f"γ = {s4_1_hp['gamma']:g}, max_weights = {s4_1_hp['max_weights']:g}")
print(f"Tiempo total de calibración: {s4_1_calibracion['segundos'].sum():.0f} s")

(s4_1_calibracion.assign(l2=s4_1_calibracion["l2"].map({False: "sin L2", True: "con L2"}))
 .sort_values(["l2", "f1_ataque_val"], ascending=[False, False])
 .style.format({"alpha": "{:g}", "gamma": "{:g}", "f1_ataque_val": "{:.4f}", "tasa_aciertos_val": "{:.4f}",
                "mse_val": "{:.4f}", "mse_train": "{:.4f}", "segundos": "{:.1f}"})
 .hide(axis="index"))

In [ ]:
# Curvas de calibración: MSE de validación por época de cada configuración probada. Una fila por
# tratamiento y M, una columna por α; en cada panel, γ = 0 y γ = 0.9.
s4_1_filas_panel = list(itertools.product((False, True), s4_1_valores_M))
s4_1_fig, s4_1_ejes = plt.subplots(len(s4_1_filas_panel), len(s4_1_valores_alpha), figsize=(11, 10),
                                   sharex=True, sharey=True, constrained_layout=True)
s4_1_eje_epocas = np.arange(1, s4_1_epocas_calibracion + 1)
for s4_1_i, (s4_1_l2, s4_1_M) in enumerate(s4_1_filas_panel):
    for s4_1_j, s4_1_alpha in enumerate(s4_1_valores_alpha):
        s4_1_ax = s4_1_ejes[s4_1_i, s4_1_j]
        for s4_1_gamma in s4_1_valores_gamma:
            s4_1_f1 = s4_1_calibracion.query("l2 == @s4_1_l2 and M == @s4_1_M and alpha == @s4_1_alpha "
                                             "and gamma == @s4_1_gamma")["f1_ataque_val"].iloc[0]
            s4_1_ax.plot(s4_1_eje_epocas, s4_1_curvas_cal[(s4_1_l2, s4_1_M, s4_1_alpha, s4_1_gamma)],
                         color=s4_1_colores_gamma[s4_1_gamma], linewidth=1.8,
                         label=f"γ = {s4_1_gamma:g} (F1 val. {s4_1_f1:.4f})")
        if s4_1_elegidas[s4_1_l2]["M"] == s4_1_M and s4_1_elegidas[s4_1_l2]["alpha"] == s4_1_alpha:
            s4_1_ax.set_title(("" if s4_1_i else f"α = {s4_1_alpha:g}\n") + "★ elegida: "
                              f"γ = {s4_1_elegidas[s4_1_l2]['gamma']:g}", fontsize=10)
        elif s4_1_i == 0:
            s4_1_ax.set_title(f"α = {s4_1_alpha:g}", fontsize=11)
        s4_1_ax.set_yscale("log")
        s4_1_ax.grid(True, which="major", color="#e8e7e2", linewidth=0.8)
        s4_1_ax.spines[["top", "right"]].set_visible(False)
        s4_1_ax.legend(loc="upper right", fontsize=8, frameon=False)
        if s4_1_j == 0:
            s4_1_ax.set_ylabel(f"{'Con' if s4_1_l2 else 'Sin'} L2, M = {s4_1_M}\nMSE de validación", fontsize=9)
        if s4_1_i == len(s4_1_filas_panel) - 1:
            s4_1_ax.set_xlabel("Época")
s4_1_fig.suptitle("Calibración en UNSW-NB15: MSE de validación por época "
                  "(la leyenda da el F1 de la clase ataque al final)", fontsize=11)
plt.show()

#### Treinta corridas emparejadas

Cada tratamiento se entrena con sus hiperparámetros calibrados sobre el 80 % de entrenamiento de cada
partición y se evalúa en su 20 % de prueba. La tabla resume la media y la desviación estándar de las
métricas en las corridas; `s4_1_resultados` guarda los valores de cada corrida.
`W_o_en_limite` y `W_s_en_limite` dan la fracción de pesos que terminan en $\pm$`max_weights`, donde el
recorte con momentum puede retenerlos (ver «Recorte y momentum» en 1.2).

In [ ]:
# 30 corridas emparejadas: la corrida i usa, en ambos tratamientos, la partición 80/20 de semilla
# S4_SEMILLAS_80_20[i] y la misma semilla de pesos iniciales. Cada tratamiento usa sus hiperparámetros.
s4_1_num_corridas = quick_or_full(3, 30)
s4_1_epocas = quick_or_full(10, 100)
s4_1_filas = []
s4_1_inicio = time.perf_counter()
for s4_1_i, s4_1_semilla_part in enumerate(S4_SEMILLAS_80_20[:s4_1_num_corridas]):
    s4_1_tr_df, s4_1_te_df = split_unsw(s4_unsw, 0.2, s4_1_semilla_part)
    s4_1_semilla_pesos = SEEDS["s4_1_l2"] + 100 + s4_1_i
    for s4_1_l2 in (False, True):
        s4_1_Xtr, s4_1_Xte, s4_1_ytr, s4_1_yte = preprocess_unsw(s4_1_tr_df, s4_1_te_df, l2=s4_1_l2)
        s4_1_hp = s4_1_elegidas[s4_1_l2]
        s4_1_r = s4_1_entrenar_y_evaluar(s4_1_Xtr, s4_1_ytr, s4_1_Xte, s4_1_yte, s4_1_hp["M"], s4_1_hp["alpha"],
                                         s4_1_hp["gamma"], s4_1_hp["max_weights"], s4_1_epocas,
                                         seed=s4_1_semilla_pesos)
        s4_1_filas.append({"corrida": s4_1_i + 1, "semilla_particion": s4_1_semilla_part,
                           "semilla_pesos": s4_1_semilla_pesos, "l2": s4_1_l2,
                           **{k: s4_1_r[k] for k in ("tasa_aciertos", "precision", "recall", "f1_ataque",
                                                     "segundos")},
                           "mse_train_final": s4_1_r["train_error"][-1],
                           # Fracción de pesos en ±max_weights al final (recorte con momentum, s1-derivacion).
                           **{f"W_{capa}_en_limite": float(np.mean(np.isclose(np.abs(W), s4_1_hp["max_weights"])))
                              for capa, W in (("o", s4_1_r["red"].W_o), ("s", s4_1_r["red"].W_s))}})
        del s4_1_Xtr, s4_1_Xte
s4_1_resultados = pd.DataFrame(s4_1_filas)
print(f"{s4_1_num_corridas} corridas × 2 tratamientos × {s4_1_epocas} épocas en "
      f"{time.perf_counter() - s4_1_inicio:.0f} s")

# Resumen por tratamiento (media y desviación estándar sobre las corridas, prueba de 20 %).
(s4_1_resultados.assign(tratamiento=s4_1_resultados["l2"].map({False: "sin L2", True: "con L2"}))
 .groupby("tratamiento", sort=False)[["f1_ataque", "tasa_aciertos", "precision", "recall",
                                      "W_o_en_limite", "W_s_en_limite"]]
 .agg(["mean", "std"]).style.format("{:.4f}"))

#### Prueba estadística y decisión sobre L2

La prueba principal es Wilcoxon de rangos con signo sobre las diferencias de F1 por corrida, con L2
menos sin L2. No supone normalidad y aprovecha el emparejamiento, porque cada par comparte la
partición y la semilla de pesos. La prueba t pareada se reporta como complemento, junto con
Shapiro-Wilk sobre las diferencias, que indica si su supuesto de normalidad es razonable. La misma
comparación se repite con la tasa de aciertos como referencia. La decisión usa solo el F1, con la regla
fijada en T0.

In [ ]:
# Prueba estadística y decisión (regla fijada en T0, antes de ver resultados).
s4_1_por_corrida = s4_1_resultados.pivot(index="corrida", columns="l2", values=["f1_ataque", "tasa_aciertos"])
s4_1_f1_sin, s4_1_f1_con = s4_1_por_corrida[("f1_ataque", False)], s4_1_por_corrida[("f1_ataque", True)]
s4_1_cmp = s4_1_comparar(s4_1_f1_con, s4_1_f1_sin, alpha=0.05)
s4_1_cmp_aciertos = s4_1_comparar(s4_1_por_corrida[("tasa_aciertos", True)],
                                  s4_1_por_corrida[("tasa_aciertos", False)], alpha=0.05)
for s4_1_nombre, s4_1_c in (("F1 de la clase ataque (principal)", s4_1_cmp),
                            ("Tasa de aciertos (complemento)", s4_1_cmp_aciertos)):
    print(f"{s4_1_nombre}: diferencia con L2 − sin L2, media {s4_1_c['media']:+.4f} y mediana "
          f"{s4_1_c['mediana']:+.4f}; Wilcoxon W = {s4_1_c['wilcoxon_estadistico']:.1f}, "
          f"p = {s4_1_c['wilcoxon_p']:.3g}; t pareada t = {s4_1_c['t_estadistico']:.2f}, "
          f"p = {s4_1_c['t_p']:.3g}; Shapiro-Wilk de las diferencias p = {s4_1_c['shapiro_p']:.3g}")
s4_1_decision_l2 = s4_1_cmp["usar_l2"]
print(f"\nDecisión (L2 solo si es significativamente mejor en F1, α = 0.05): usar L2 = {s4_1_decision_l2}")
if USE_L2 is None:
    print("Aviso: USE_L2 sigue en None en s0-config; registrar allí la decisión (T4).")
elif USE_L2 != s4_1_decision_l2:
    s4_1_mensaje = f"USE_L2 = {USE_L2} en s0-config no coincide con la decisión de esta ejecución"
    if QUICK:
        print(f"Aviso (ejecución reducida, {s4_1_num_corridas} corridas): {s4_1_mensaje}.")
    else:
        raise AssertionError(s4_1_mensaje)

# F1 por corrida (líneas entre pares) y diferencias emparejadas.
s4_1_fig, (s4_1_ax_f1, s4_1_ax_d) = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True,
                                                 gridspec_kw={"width_ratios": [1.2, 1]})
for s4_1_k in s4_1_por_corrida.index:
    s4_1_ax_f1.plot([0, 1], [s4_1_f1_sin[s4_1_k], s4_1_f1_con[s4_1_k]], color="#c3c2b7", linewidth=0.8, zorder=1)
for s4_1_x, s4_1_l2 in ((0, False), (1, True)):
    s4_1_ax_f1.scatter(np.full(len(s4_1_por_corrida), s4_1_x), s4_1_por_corrida[("f1_ataque", s4_1_l2)],
                       s=36, color=s4_1_colores[s4_1_l2], edgecolor="white", linewidth=1, zorder=2)
s4_1_ax_f1.set_xticks([0, 1], ["Sin L2", "Con L2"])
s4_1_ax_f1.set_xlim(-0.4, 1.4)
s4_1_ax_f1.set_ylabel("F1 de la clase ataque (prueba de 20 %)")
s4_1_ax_f1.set_title("F1 por corrida (líneas: misma partición y pesos iniciales)", fontsize=10)
s4_1_ax_d.axvline(0, color="#8a8984", linewidth=1, linestyle="--")
s4_1_ax_d.hist(s4_1_cmp["diferencias"], bins=12, color="#6250d6", edgecolor="white", linewidth=2)
s4_1_ax_d.axvline(s4_1_cmp["mediana"], color="#0b0b0b", linewidth=1.5)
s4_1_ax_d.set_xlabel("Diferencia de F1 (con L2 − sin L2)")
s4_1_ax_d.set_ylabel("Corridas")
s4_1_ax_d.yaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
s4_1_ax_d.set_title(f"Mediana {s4_1_cmp['mediana']:+.4f} (línea negra); Wilcoxon p = {s4_1_cmp['wilcoxon_p']:.3g}",
                    fontsize=10)
for s4_1_ax in (s4_1_ax_f1, s4_1_ax_d):
    s4_1_ax.grid(True, axis="y", color="#e8e7e2", linewidth=0.8)
    s4_1_ax.spines[["top", "right"]].set_visible(False)
plt.show()

#### Análisis

*Pendiente de T8.* El comentario que cite números se redacta con la ejecución final del notebook (ver
«Reglas de congruencia de las salidas» en `JOURNAL.md`). Debe cubrir:

- las configuraciones elegidas en la calibración y cómo se comparan las curvas;
- la magnitud y la significancia de la diferencia en F1 y en tasa de aciertos;
- el resultado de Shapiro-Wilk para la prueba t;
- la fracción de pesos en el límite (`max_weights` = 1 fijo en la rejilla) y lo que implica el recorte con
  momentum;
- la decisión sobre L2 con la regla acordada;
- las limitaciones: calibración compartida y duplicados (D9).

### 4.2 Ensamble y cuantificación de incertidumbre

*Responsable: P3 (T5).*

#### Método y decisiones adoptadas

**Método del material de apoyo** (*Estimación de incertidumbre para modelos de aprendizaje automático*,
§2, en `docs/assignment/EstimacionIncertidumbre.pdf`, que cita el artículo *Uncertainty Quantification and
Deep Ensembles*). El conjunto de entrenamiento $D$ se divide en $N$ particiones $D_1, \dots, D_N$ y se entrena
un modelo independiente por partición, $f_{W_m} = T(D_m)$. El ensamble $f_{W_1}, \dots, f_{W_N}$ aproxima
la distribución $p(W \mid D)$, y la incertidumbre de una entrada $\vec{x}^*$ se mide con la varianza de las
$N$ predicciones:

$$u(\vec{x}^*) = \frac{1}{N - 1} \sum_{m=1}^{N} \left( f_{W_m}(\vec{x}^*) - \bar{f}(\vec{x}^*) \right)^2,
\qquad \bar{f}(\vec{x}^*) = \frac{1}{N} \sum_{m=1}^{N} f_{W_m}(\vec{x}^*).$$

**Miembros: clasificadores de mínimos cuadrados** (Bishop, 2006, §4.1.3). Cada miembro es un vector de
pesos $\vec{w}_m \in \mathbb{R}^{D+1}$, con el sesgo en la posición 0 ($x_0 = 1$, como en la representación
del curso). Con objetivos $t_i = 2y_i - 1 \in \{-1, +1\}$, el miembro $m$ minimiza el error cuadrático sobre
su partición:

$$\vec{w}_m = \arg\min_{\vec{w}} \sum_{i \in D_m} \left( \vec{w}^{\top} \tilde{\vec{x}}_i - t_i \right)^2,
\qquad \tilde{\vec{x}}_i = [1, \vec{x}_i].$$

Como pide el enunciado, el puntaje de cada miembro es **la salida del producto punto**,
$f_{W_m}(\vec{x}) = \vec{w}_m^{\top} \tilde{\vec{x}}$, sin pasar por la función de activación del modelo (el
signo, que da la clase).

**Decisiones (D11 en `JOURNAL.md`).**

1. **Una salida con objetivos ±1.** La clase de UNSW-NB15 es binaria. Con una sola salida, la varianza de
   los puntajes es un escalar por observación, que es lo que recibe el ECE de §4.3 (contrato D6). Con dos
   salidas one-hot habría que acordar además cómo combinar dos varianzas.
2. **Solución de norma mínima.** Los bloques one-hot de la preparación de datos suman 1, igual que la
   columna del sesgo, así que $\tilde{X}^{\top}\tilde{X}$ es singular y la ecuación normal
   $(\tilde{X}^{\top}\tilde{X})^{-1}\tilde{X}^{\top}\vec{t}$ no se puede usar. Además, con $N = 1000$ (§4.4),
   cada miembro recibe unas 180 filas para cerca de 196 pesos y el sistema tiene infinitas soluciones. Se
   usa `numpy.linalg.lstsq` (basado en la SVD), que devuelve la solución de mínimos cuadrados de **norma
   mínima**, $\vec{w}_m = \tilde{X}_m^{+} \vec{t}_m$ con $\tilde{X}_m^{+}$ la pseudoinversa. No agrega
   hiperparámetros, y cuando la solución es única coincide con la ecuación normal. Con pocas filas por
   miembro, los miembros pueden discrepar mucho lejos de sus datos y aparecen varianzas atípicas grandes: es
   el comportamiento esperado de un estimador de incertidumbre epistémica con poca información, y §4.4 lo
   tiene en cuenta al normalizar la varianza.
3. **Particiones disjuntas y aleatorias.** Las filas de entrenamiento se reordenan al azar con una semilla
   y se reparten en $N$ partes disjuntas cuyos tamaños difieren en una fila como máximo
   (`numpy.array_split`). Cada fila entrena a un único miembro, como describe el material. La semilla por
   omisión es `SEEDS["s4_2_ensamble"]`.
4. **Firma de `quantify_uncertainty_ensemble(x, model, N=10)`.** El enunciado no define `model`. Aquí es el
   ensamble ya entrenado que devuelve `train_ensemble`, que se reutiliza para cualquier entrada. `N` debe
   coincidir con su número de miembros; si no, la función da `ValueError` en lugar de usar en silencio un
   ensamble de otro tamaño. `x` puede ser una observación $\vec{x} \in \mathbb{R}^D$ (devuelve un número) o
   una matriz $n \times D$ (devuelve un vector de forma $(n,)$).
5. **Predicción del ensamble.** La etiqueta es el signo del puntaje medio: ataque (1) si
   $\bar{f}(\vec{x}) \geq 0$ y normal (0) si no. Usa los mismos puntajes previos a la activación que la
   varianza, como pide D6 para el parámetro `predictions` del ECE.

**Funciones.**

| Función | Qué hace |
| --- | --- |
| `train_ensemble(X, y, N=10, seed=None)` | Reparte las filas de `X` (n × D) en N particiones disjuntas y ajusta un clasificador de mínimos cuadrados en cada una. Devuelve el ensamble: un `dict` con `W` (N × (D + 1), una fila de pesos por miembro, sesgo en la columna 0), `N`, `indices` (las filas de cada partición) y `seed`. |
| `run_ensemble_uq(X, model, return_predictions=False, batch_size=10000)` | Calcula los puntajes de los N miembros para cada fila de `X` y devuelve su varianza muestral (con $N - 1$), forma (n,). Con `return_predictions=True` devuelve además las etiquetas 0/1 del signo del puntaje medio, forma (n,). Procesa `X` por bloques de filas para no guardar la matriz completa n × N (con N = 1000 y el 30 % de prueba ocuparía unos 0.6 GB). |
| `quantify_uncertainty_ensemble(x, model, N=10)` | Incertidumbre del ensamble `model` para `x`: comprueba que tenga N miembros y devuelve la varianza de `run_ensemble_uq`. |
| `s4_2_minimos_cuadrados(X, y)` | Ajusta un miembro: el vector $\vec{w}$ de norma mínima que minimiza $\lVert \tilde{X}\vec{w} - \vec{t} \rVert^2$. |
| `s4_2_particiones(n, N, seed)` | Índices de N particiones disjuntas de `range(n)`, en orden aleatorio, con tamaños que difieren en 1 como máximo. |
| `s4_2_puntajes(X, model)` | Matriz n × N de puntajes $\tilde{X} W^{\top}$ (producto punto, sin activación). |
| `s4_2_con_sesgo(X)` y `s4_2_objetivos(y)` | Agregan la columna de unos del sesgo y convierten las etiquetas 0/1 en objetivos −1/+1. |

**Uso con §4.3 y §4.4.** Para un conjunto de prueba:
`u, pred = run_ensemble_uq(X_test, ensamble, return_predictions=True)` y luego
`calculate_expected_calibration_error(X_test, y_test, u, pred)`.

**Referencia**

C. M. Bishop, *Pattern Recognition and Machine Learning*. Nueva York: Springer, 2006, §4.1.3
(«Least squares for classification»).

In [ ]:
import numpy as np


def s4_2_con_sesgo(X):
    """Devuelve [1, X]: agrega a la izquierda la columna de unos del sesgo (x₀ = 1)."""
    X = np.asarray(X, dtype=np.float64)
    if X.ndim != 2:
        raise ValueError("X debe ser una matriz n × D.")
    return np.hstack([np.ones((X.shape[0], 1)), X])


def s4_2_objetivos(y):
    """Convierte etiquetas 0/1 de forma (n,) en objetivos −1/+1 para mínimos cuadrados."""
    y = np.asarray(y).ravel()
    if not np.isin(y, (0, 1)).all():
        raise ValueError("y debe contener solo 0 y 1.")
    return 2.0 * y.astype(np.float64) - 1.0


def s4_2_particiones(n, N, seed):
    """Índices de N particiones disjuntas de range(n), en orden aleatorio.

    Las filas se reordenan con `numpy.random.default_rng(seed)` y se reparten con
    `numpy.array_split`, así que los tamaños difieren en 1 como máximo. Se exige 2 ≤ N ≤ n:
    la varianza necesita al menos dos miembros y cada miembro, al menos una fila."""
    if int(N) != N or not 2 <= N <= n:
        raise ValueError(f"N debe ser un entero entre 2 y n = {n}.")
    rng = np.random.default_rng(seed)
    return np.array_split(rng.permutation(n), int(N))


def s4_2_minimos_cuadrados(X, y):
    """Ajusta un clasificador de mínimos cuadrados y devuelve sus pesos w de forma (D + 1,).

    Minimiza ||[1, X] w − t||² con t = 2y − 1. Usa `numpy.linalg.lstsq` (SVD): si la solución no
    es única (columnas colineales o menos filas que pesos), devuelve la de norma mínima, w = X⁺ t."""
    return np.linalg.lstsq(s4_2_con_sesgo(X), s4_2_objetivos(y), rcond=None)[0]


def train_ensemble(X, y, N=10, seed=None):
    """Entrena un ensamble de N clasificadores de mínimos cuadrados (material de apoyo, §2).

    Las filas de `X` (n × D) se reparten en N particiones disjuntas (`s4_2_particiones`) y el
    miembro m se ajusta solo con la partición m. Si `seed` es None se usa `SEEDS["s4_2_ensamble"]`.

    Devuelve un dict con:
      W        pesos de los miembros, forma (N, D + 1); la columna 0 es el sesgo.
      N        número de miembros.
      indices  lista con las filas de `X` de cada partición.
      seed     semilla usada para las particiones.
    """
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y).ravel()
    if X.ndim != 2 or X.shape[0] != y.shape[0]:
        raise ValueError("X debe ser n × D e y debe tener n etiquetas.")
    seed = SEEDS["s4_2_ensamble"] if seed is None else seed
    indices = s4_2_particiones(X.shape[0], N, seed)
    W = np.vstack([s4_2_minimos_cuadrados(X[idx], y[idx]) for idx in indices])
    return {"W": W, "N": int(N), "indices": indices, "seed": seed}


def s4_2_puntajes(X, model):
    """Puntajes de los miembros: [1, X] Wᵀ, forma (n, N). Es el producto punto, sin activación."""
    A = s4_2_con_sesgo(X)
    if A.shape[1] != model["W"].shape[1]:
        raise ValueError(f"X tiene {A.shape[1] - 1} atributos; el ensamble espera {model['W'].shape[1] - 1}.")
    return A @ model["W"].T


def run_ensemble_uq(X, model, return_predictions=False, batch_size=10_000):
    """Evalúa el ensamble `model` sobre las filas de `X` (n × D).

    Para cada observación calcula los N puntajes de los miembros y devuelve su varianza muestral,
    u = Σ (f_m − f̄)² / (N − 1), de forma (n,). Con `return_predictions=True` devuelve
    `(u, predicciones)`, donde predicciones = 1 si f̄ ≥ 0 y 0 si no (enteros, forma (n,)).
    `X` se procesa por bloques de `batch_size` filas; el resultado no depende de ese tamaño."""
    X = np.asarray(X, dtype=np.float64)
    if X.ndim != 2:
        raise ValueError("X debe ser una matriz n × D.")
    varianza = np.empty(X.shape[0])
    media = np.empty(X.shape[0])
    for inicio in range(0, X.shape[0], batch_size):
        S = s4_2_puntajes(X[inicio:inicio + batch_size], model)
        varianza[inicio:inicio + batch_size] = S.var(axis=1, ddof=1)
        media[inicio:inicio + batch_size] = S.mean(axis=1)
    if return_predictions:
        return varianza, (media >= 0).astype(np.int64)
    return varianza


def quantify_uncertainty_ensemble(x, model, N=10):
    """Cuantifica la incertidumbre del ensamble `model` para la entrada `x`.

    `model` es el ensamble que devuelve `train_ensemble` y debe tener exactamente N miembros.
    Si `x` es una observación (forma (D,)), devuelve su varianza como float; si es una matriz
    n × D, devuelve las n varianzas, forma (n,)."""
    if model["W"].shape[0] != N:
        raise ValueError(f"El ensamble tiene {model['W'].shape[0]} miembros, no N = {N}.")
    x = np.asarray(x, dtype=np.float64)
    if x.ndim == 1:
        return float(run_ensemble_uq(x[np.newaxis, :], model)[0])
    return run_ensemble_uq(x, model)

#### Pruebas unitarias de §4.2

Las pruebas usan datos sintéticos pequeños y ensambles armados a mano, cuyos resultados se pueden calcular
sin el código. No dependen de UNSW-NB15. Hay 16 pruebas, al menos dos por función:

| Clase | Qué comprueba |
| --- | --- |
| `TestEnsambleMinimosCuadrados` | Un miembro recupera sin error un separador exacto ($t = x$); con rango completo coincide con la ecuación normal $(\tilde{X}^{\top}\tilde{X})^{-1}\tilde{X}^{\top}\vec{t}$; con columnas colineales reparte el peso por igual (norma mínima) y con menos filas que pesos interpola los objetivos y coincide con la pseudoinversa. |
| `TestEnsambleAuxiliares` | La columna del sesgo va en la posición 0 y las etiquetas 0/1 pasan a −1/+1 (se rechazan otros valores y los vectores en lugar de matrices). Las particiones son disjuntas, cubren todas las filas, tienen tamaños que difieren en 1 como máximo, son reproducibles con la semilla y rechazan N < 2, N > n y N no entero. |
| `TestTrainEnsemble` | El miembro $m$ es exactamente el clasificador de mínimos cuadrados de la partición $m$; el ensamble tiene la forma documentada, es reproducible y usa `SEEDS["s4_2_ensamble"]` por omisión. |
| `TestRunEnsembleUq` | Con tres miembros conocidos ($f_1 = x$, $f_2 = 3x$, $f_3 = 1 + 2x$), la varianza con $N - 1$ y la predicción por el signo del puntaje medio coinciden con un cálculo a mano; miembros idénticos dan varianza 0; procesar por bloques no cambia el resultado; un número de atributos distinto da `ValueError`. |
| `TestQuantifyUncertaintyEnsemble` | Devuelve un número para una observación y un vector para una matriz, con los valores calculados a mano; exige que `N` coincida con el número de miembros. **Comportamiento como estimador de incertidumbre:** con entrenamiento concentrado en $[-1, 1]$, la varianza en $x = 50$ supera más de 100 veces la de $x = 0$, porque las rectas de los miembros casi coinciden cerca de los datos y se separan lejos de ellos. Las salidas tienen el formato del contrato con §4.3 (varianza no negativa y predicciones 0/1 de forma $(n,)$). |

Para comprobar que las pruebas detectan errores, se introdujeron a propósito diez errores, uno por vez:
varianza dividida entre $N$, objetivos 0/1 en lugar de ±1, puntajes después de la activación (signo), sesgo
en la última columna, particiones con reemplazo, miembros entrenados con todos los datos, ridge en lugar de
norma mínima, predicción constante, `N` sin validar y bloques mal indexados. Cada uno hizo fallar al menos
una prueba.

In [ ]:
import sys
import unittest


class TestEnsambleMinimosCuadrados(unittest.TestCase):
    def test_recupera_separador_exacto(self):
        # x = −1 → t = −1 y x = 1 → t = +1: la recta t = x (w = [0, 1]) ajusta sin error.
        w = s4_2_minimos_cuadrados(np.array([[-1.0], [1.0], [-1.0], [1.0]]), [0, 1, 0, 1])
        np.testing.assert_allclose(w, [0.0, 1.0], atol=1e-12)

    def test_coincide_con_ecuacion_normal_si_hay_rango_completo(self):
        rng = np.random.default_rng(0)
        X, y = rng.normal(size=(50, 3)), rng.integers(0, 2, size=50)
        A, t = s4_2_con_sesgo(X), s4_2_objetivos(y)
        np.testing.assert_allclose(s4_2_minimos_cuadrados(X, y), np.linalg.solve(A.T @ A, A.T @ t))

    def test_norma_minima_con_columnas_colineales_y_pocas_filas(self):
        rng = np.random.default_rng(1)
        # Columna repetida: hay infinitas soluciones; la de norma mínima reparte el peso por igual.
        x = rng.normal(size=(20, 1))
        w = s4_2_minimos_cuadrados(np.hstack([x, x]), (x[:, 0] > 0).astype(int))
        self.assertAlmostEqual(w[1], w[2])
        # Menos filas que pesos (3 filas, 5 pesos): interpola los objetivos con la pseudoinversa.
        X, y = rng.normal(size=(3, 4)), np.array([0, 1, 1])
        w = s4_2_minimos_cuadrados(X, y)
        np.testing.assert_allclose(s4_2_con_sesgo(X) @ w, s4_2_objetivos(y), atol=1e-10)
        np.testing.assert_allclose(w, np.linalg.pinv(s4_2_con_sesgo(X)) @ s4_2_objetivos(y))


class TestEnsambleAuxiliares(unittest.TestCase):
    def test_con_sesgo_y_objetivos(self):
        np.testing.assert_array_equal(s4_2_con_sesgo([[2.0, 3.0]]), [[1.0, 2.0, 3.0]])
        np.testing.assert_array_equal(s4_2_objetivos([0, 1, 1]), [-1.0, 1.0, 1.0])
        with self.assertRaises(ValueError):
            s4_2_objetivos([0, 2])
        with self.assertRaises(ValueError):
            s4_2_con_sesgo([1.0, 2.0])  # un vector, no una matriz

    def test_particiones_disjuntas_completas_y_equilibradas(self):
        partes = s4_2_particiones(103, 10, seed=4200)
        todas = np.concatenate(partes)
        self.assertEqual(len(partes), 10)
        self.assertEqual(sorted(todas.tolist()), list(range(103)))  # cubren todo y sin repetir
        self.assertLessEqual(max(map(len, partes)) - min(map(len, partes)), 1)

    def test_particiones_reproducibles_y_validas(self):
        a, b, c = (s4_2_particiones(50, 5, s) for s in (7, 7, 8))
        self.assertTrue(all(np.array_equal(p, q) for p, q in zip(a, b)))
        self.assertFalse(all(np.array_equal(p, q) for p, q in zip(a, c)))
        for N in (1, 51, 2.5):
            with self.assertRaises(ValueError):
                s4_2_particiones(50, N, 7)


class TestTrainEnsemble(unittest.TestCase):
    def setUp(self):
        rng = np.random.default_rng(2)
        self.X = rng.normal(size=(60, 2))
        self.y = (self.X[:, 0] + 0.5 * rng.normal(size=60) > 0).astype(int)

    def test_cada_miembro_se_ajusta_con_su_particion(self):
        modelo = train_ensemble(self.X, self.y, N=4, seed=3)
        self.assertEqual((modelo["W"].shape, modelo["N"], modelo["seed"]), ((4, 3), 4, 3))
        for m, idx in enumerate(modelo["indices"]):
            np.testing.assert_allclose(modelo["W"][m], s4_2_minimos_cuadrados(self.X[idx], self.y[idx]))

    def test_reproducible_y_semilla_por_omision(self):
        a = train_ensemble(self.X, self.y, N=3, seed=5)
        b = train_ensemble(self.X, self.y, N=3, seed=5)
        np.testing.assert_array_equal(a["W"], b["W"])
        self.assertEqual(train_ensemble(self.X, self.y, N=3)["seed"], SEEDS["s4_2_ensamble"])
        with self.assertRaises(ValueError):
            train_ensemble(self.X, self.y[:-1], N=3)


class TestRunEnsembleUq(unittest.TestCase):
    # Tres miembros a mano (D = 1): f₁ = x, f₂ = 3x y f₃ = 1 + 2x.
    modelo = {"W": np.array([[0.0, 1.0], [0.0, 3.0], [1.0, 2.0]]), "N": 3}

    def test_varianza_y_prediccion_calculadas_a_mano(self):
        X = np.array([[1.0], [0.0], [-2.0]])
        # Puntajes: x = 1 → (1, 3, 3); x = 0 → (0, 0, 1); x = −2 → (−2, −6, −3).
        u, pred = run_ensemble_uq(X, self.modelo, return_predictions=True)
        np.testing.assert_allclose(u, [4 / 3, 1 / 3, 13 / 3])  # Σ (f − f̄)² / (N − 1)
        np.testing.assert_array_equal(pred, [1, 1, 0])  # signo del puntaje medio: 7/3, 1/3 y −11/3
        self.assertEqual(pred.dtype, np.int64)

    def test_miembros_identicos_dan_varianza_cero(self):
        iguales = {"W": np.tile([[0.5, -1.0, 2.0]], (4, 1)), "N": 4}
        u = run_ensemble_uq(np.random.default_rng(4).normal(size=(10, 2)), iguales)
        np.testing.assert_allclose(u, 0.0, atol=1e-24)

    def test_bloques_no_cambian_el_resultado(self):
        X = np.linspace(-3, 3, 25)[:, np.newaxis]
        u, pred = run_ensemble_uq(X, self.modelo, return_predictions=True)
        u1, pred1 = run_ensemble_uq(X, self.modelo, return_predictions=True, batch_size=1)
        u7 = run_ensemble_uq(X, self.modelo, batch_size=7)
        np.testing.assert_allclose(u1, u)
        np.testing.assert_allclose(u7, u)
        np.testing.assert_array_equal(pred1, pred)

    def test_rechaza_dimension_distinta(self):
        with self.assertRaises(ValueError):
            run_ensemble_uq(np.zeros((2, 3)), self.modelo)


class TestQuantifyUncertaintyEnsemble(unittest.TestCase):
    def test_observacion_y_matriz(self):
        modelo = TestRunEnsembleUq.modelo
        u = quantify_uncertainty_ensemble(np.array([1.0]), modelo, N=3)
        self.assertIsInstance(u, float)
        self.assertAlmostEqual(u, 4 / 3)
        np.testing.assert_allclose(
            quantify_uncertainty_ensemble(np.array([[1.0], [0.0]]), modelo, N=3), [4 / 3, 1 / 3])

    def test_exige_el_mismo_numero_de_miembros(self):
        with self.assertRaises(ValueError):
            quantify_uncertainty_ensemble(np.array([1.0]), TestRunEnsembleUq.modelo)  # N = 10 por omisión

    def test_incertidumbre_crece_lejos_de_los_datos(self):
        # Entrenamiento concentrado en [−1, 1] con etiquetas ruidosas. Cada miembro ajusta una recta
        # distinta; las rectas casi coinciden cerca de los datos y se separan lejos de ellos, así que
        # la varianza en x = 50 debe superar con creces la de x = 0.
        rng = np.random.default_rng(6)
        x = rng.uniform(-1, 1, size=(400, 1))
        y = (x[:, 0] + rng.normal(scale=0.8, size=400) > 0).astype(int)
        modelo = train_ensemble(x, y, N=10, seed=SEEDS["s4_2_ensamble"])
        u_cerca, u_lejos = quantify_uncertainty_ensemble(np.array([[0.0], [50.0]]), modelo, N=10)
        self.assertGreater(u_lejos, 100 * u_cerca)

    def test_salidas_con_el_formato_del_contrato(self):
        # Contrato con §4.3 (D6): varianza no negativa y predicciones 0/1, ambas de forma (n,).
        # (El ECE se define después, en §4.3; §4.4 los usa juntos.)
        rng = np.random.default_rng(7)
        X = rng.normal(size=(200, 3))
        y = (X[:, 0] - X[:, 1] > 0).astype(int)
        modelo = train_ensemble(X[:150], y[:150], N=5, seed=8)
        u, pred = run_ensemble_uq(X[150:], modelo, return_predictions=True)
        self.assertEqual((u.shape, pred.shape), ((50,), (50,)))
        self.assertTrue(np.all(u >= 0) and np.isin(pred, (0, 1)).all())
        np.testing.assert_allclose(u, quantify_uncertainty_ensemble(X[150:], modelo, N=5))
        self.assertGreater(np.mean(pred == y[150:]), 0.9)  # problema lineal: el ensamble acierta


suite_s4_2 = unittest.TestSuite()
for caso in (TestEnsambleMinimosCuadrados, TestEnsambleAuxiliares, TestTrainEnsemble,
             TestRunEnsembleUq, TestQuantifyUncertaintyEnsemble):
    suite_s4_2.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(caso))
resultado_s4_2 = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite_s4_2)
assert resultado_s4_2.wasSuccessful(), "Fallaron pruebas unitarias de §4.2"

#### Demostración con UNSW-NB15

La celda siguiente aplica el ensamble a la primera partición 70/30 de la preparación de datos
(`S4_SEMILLAS_70_30[0]`), con $N = 10$, el preprocesamiento base y la decisión de §4.1 sobre L2
(`USE_L2 = False`: sin L2). Muestra el tamaño de cada partición del ensamble, los tiempos
de entrenamiento y evaluación, la tasa de aciertos del ensamble y de sus miembros, la distribución de la
varianza y la varianza media de las predicciones correctas e incorrectas. Si el estimador es útil, las
incorrectas deberían tener más incertidumbre. La evaluación con $N \in \{10, 100, 1000\}$, el ECE, los
tiempos y la prueba estadística corresponden a §4.4.

In [ ]:
import time

s4_2_l2 = USE_L2  # decisión de §4.1 (T4): sin L2
s4_2_train_df, s4_2_test_df = split_unsw(s4_unsw, 0.3, S4_SEMILLAS_70_30[0])
s4_2_X_train, s4_2_X_test, s4_2_y_train, s4_2_y_test = preprocess_unsw(s4_2_train_df, s4_2_test_df, l2=s4_2_l2)
print(f"Partición 70/30 con semilla {S4_SEMILLAS_70_30[0]} (l2={s4_2_l2}): X_train {s4_2_X_train.shape}, "
      f"X_test {s4_2_X_test.shape}")

s4_2_inicio = time.perf_counter()
s4_2_ensamble = train_ensemble(s4_2_X_train, s4_2_y_train, N=10)
s4_2_t_entrenar = time.perf_counter() - s4_2_inicio
s4_2_tamanos = [len(idx) for idx in s4_2_ensamble["indices"]]
print(f"Ensamble de N = {s4_2_ensamble['N']} (semilla {s4_2_ensamble['seed']}): "
      f"{min(s4_2_tamanos)}–{max(s4_2_tamanos)} filas por miembro, W {s4_2_ensamble['W'].shape}, "
      f"entrenado en {s4_2_t_entrenar:.2f} s")

s4_2_inicio = time.perf_counter()
s4_2_u, s4_2_pred = run_ensemble_uq(s4_2_X_test, s4_2_ensamble, return_predictions=True)
s4_2_t_evaluar = time.perf_counter() - s4_2_inicio
s4_2_aciertos = s4_2_pred == s4_2_y_test
s4_2_aciertos_miembros = ((s4_2_puntajes(s4_2_X_test, s4_2_ensamble) >= 0) == s4_2_y_test[:, None]).mean(axis=0)
print(f"Evaluado sobre {len(s4_2_y_test)} observaciones de prueba en {s4_2_t_evaluar:.2f} s")
print(f"Tasa de aciertos del ensamble: {s4_2_aciertos.mean():.4f} · miembros: "
      f"{s4_2_aciertos_miembros.min():.4f}–{s4_2_aciertos_miembros.max():.4f}")
print("Varianza (cuantiles 0, 25, 50, 75, 99 y 100 %):",
      np.round(np.quantile(s4_2_u, [0, 0.25, 0.5, 0.75, 0.99, 1]), 6))
print(f"Varianza media · predicciones correctas: {s4_2_u[s4_2_aciertos].mean():.6f} · "
      f"incorrectas: {s4_2_u[~s4_2_aciertos].mean():.6f}")
print(f"quantify_uncertainty_ensemble para la primera observación de prueba: "
      f"{quantify_uncertainty_ensemble(s4_2_X_test[0], s4_2_ensamble, N=10):.6f} "
      f"(etiqueta real {s4_2_y_test[0]}, predicha {s4_2_pred[0]})")
del s4_2_X_train, s4_2_X_test  # ~0.4 GB; §4.4 genera sus propias matrices

### 4.3 Expected calibration error

*Responsable: P2 (T6).*

#### Definición y decisiones adoptadas

**Definición del material de apoyo** (*Estimación de incertidumbre para modelos de
aprendizaje automático*, §4, en `docs/assignment/EstimacionIncertidumbre.pdf`). Dado un
estimador de incertidumbre $s(\vec{x}_i)$ y un conjunto de prueba con $N$ observaciones
etiquetadas, los puntajes se reparten en cubetas $B_1, \dots, B_{n_b}$. Para cada cubeta,
$\overline{\mathrm{acc}}(B_m)$ es la tasa de aciertos y $\overline{s}(B_m)$ el puntaje
promedio de sus observaciones:

$$\mathrm{ECE} = \sum_{m=1}^{n_b} \frac{|B_m|}{N} \left| \overline{\mathrm{acc}}(B_m) - \overline{s}(B_m) \right|$$

Las cubetas vacías se ignoran. El material supone puntajes en $[0, 1]$; si no lo están,
propone medir la correlación de Pearson entre incertidumbre y tasa de aciertos.

**Decisiones (D6 en `JOURNAL.md`).**

1. **Puntaje.** La incertidumbre del ensamble (§4.2) es una varianza: no está acotada y
   crece cuando el modelo es *menos* confiable, mientras que $\overline{\mathrm{acc}}$ mide
   aciertos. Por eso se usa la confianza $s = 1 - u_{\text{norm}}$, con $u_{\text{norm}}$ la
   incertidumbre normalizada min-max al rango del conjunto evaluado (o a un rango dado).
2. **Complemento.** Se reporta además la correlación de Pearson entre incertidumbre y
   acierto (1 o 0), la alternativa que ofrece el material. Un buen estimador da un valor
   negativo: más incertidumbre, menos aciertos.
3. **Cubetas.** 10 cubetas de igual ancho en $[0, 1]$ por defecto, como en la figura del
   material, configurables con `n_bins`. La cubeta $m$ cubre $[m/n_b, (m+1)/n_b)$; la
   última incluye el 1.
4. **Firma y relación con la última capa.** La tasa de aciertos necesita las predicciones,
   que la firma `calculate_expected_calibration_error(X, y, uncertainties)` no recibe. Se
   agrega el parámetro `predictions`: las etiquetas que el ensamble predice a partir de la
   salida de su última capa **antes** de la activación (los puntajes del producto punto),
   como pide el enunciado. Ese cálculo pertenece a §4.2; esta función recibe las etiquetas
   ya calculadas. `X` se conserva por la firma pedida y solo se usa para verificar que el
   número de observaciones coincida.

In [ ]:
import numpy as np


def uncertainty_to_confidence(uncertainties, u_range=None):
    """Convierte incertidumbres no negativas en puntajes de confianza en [0, 1].

    Calcula s = 1 - u_norm, donde u_norm es la incertidumbre normalizada
    min-max. La menor incertidumbre recibe confianza 1 y la mayor, 0.

    Parámetros
    ----------
    uncertainties : array_like, forma (n,)
        Incertidumbre por observación (por ejemplo, la varianza del ensamble).
        Debe tener valores finitos y no negativos.
    u_range : tupla (u_min, u_max) o None
        Rango de normalización. Si es None, se usan el mínimo y el máximo de
        `uncertainties`. Los valores fuera del rango se recortan a [0, 1].

    Retorna
    -------
    numpy.ndarray, forma (n,)
        Confianzas en [0, 1]. Si el rango es degenerado (u_max == u_min),
        todas valen 1, porque no hay diferencia de incertidumbre que ordenar.
    """
    u = np.asarray(uncertainties, dtype=float).ravel()
    if u.size == 0:
        raise ValueError("uncertainties no puede estar vacío")
    if not np.all(np.isfinite(u)) or np.any(u < 0):
        raise ValueError("uncertainties debe tener valores finitos y no negativos")
    if u_range is None:
        u_min, u_max = float(u.min()), float(u.max())
    else:
        u_min, u_max = (float(v) for v in u_range)
        if u_max < u_min:
            raise ValueError("u_range debe cumplir u_min <= u_max")
    if u_max == u_min:
        return np.ones_like(u)
    u_norm = np.clip((u - u_min) / (u_max - u_min), 0.0, 1.0)
    return 1.0 - u_norm


def calibration_bins(confidences, correct, n_bins=10):
    """Agrupa las observaciones en cubetas de igual ancho sobre [0, 1].

    La cubeta m cubre [m / n_bins, (m + 1) / n_bins); la última incluye 1.

    Parámetros
    ----------
    confidences : array_like, forma (n,)
        Puntajes de confianza en [0, 1].
    correct : array_like de bool o 0/1, forma (n,)
        Indica si la predicción de cada observación fue correcta.
    n_bins : int
        Número de cubetas (al menos 1).

    Retorna
    -------
    dict con
        'edges' : bordes de las cubetas, forma (n_bins + 1,).
        'count' : |B_m|, observaciones por cubeta, forma (n_bins,).
        'accuracy' : tasa de aciertos por cubeta; NaN si está vacía.
        'confidence' : confianza promedio por cubeta; NaN si está vacía.
    """
    c = np.asarray(confidences, dtype=float).ravel()
    ok = np.asarray(correct).ravel().astype(float)
    if c.size == 0:
        raise ValueError("confidences no puede estar vacío")
    if c.shape != ok.shape:
        raise ValueError("confidences y correct deben tener la misma longitud")
    if not np.all(np.isfinite(c)) or np.any((c < 0) | (c > 1)):
        raise ValueError("confidences debe estar en [0, 1]")
    if not np.all((ok == 0) | (ok == 1)):
        raise ValueError("correct debe contener solo valores booleanos o 0/1")
    if int(n_bins) != n_bins or n_bins < 1:
        raise ValueError("n_bins debe ser un entero positivo")
    n_bins = int(n_bins)

    idx = np.minimum(np.floor(c * n_bins).astype(int), n_bins - 1)
    count = np.bincount(idx, minlength=n_bins)
    hits = np.bincount(idx, weights=ok, minlength=n_bins)
    conf_sum = np.bincount(idx, weights=c, minlength=n_bins)
    filled = count > 0
    accuracy = np.full(n_bins, np.nan)
    confidence = np.full(n_bins, np.nan)
    accuracy[filled] = hits[filled] / count[filled]
    confidence[filled] = conf_sum[filled] / count[filled]
    return {
        "edges": np.linspace(0.0, 1.0, n_bins + 1),
        "count": count,
        "accuracy": accuracy,
        "confidence": confidence,
    }


def calculate_expected_calibration_error(X, y, uncertainties, predictions,
                                         n_bins=10, u_range=None):
    """Calcula el expected calibration error (ECE) de un estimador de incertidumbre.

    Sigue la definición del material de apoyo:
    ECE = sum_m |B_m| / N * |acc(B_m) - s(B_m)|, ignorando las cubetas vacías,
    con s = 1 - incertidumbre normalizada (ver `uncertainty_to_confidence`).

    Parámetros
    ----------
    X : array_like, forma (n, D)
        Observaciones evaluadas. Se conserva por la firma del enunciado y solo
        se usa para verificar que haya n observaciones.
    y : array_like, forma (n,)
        Etiquetas correctas (ground truth).
    uncertainties : array_like, forma (n,)
        Incertidumbre de cada observación (por ejemplo, varianza del ensamble).
    predictions : array_like, forma (n,)
        Etiquetas predichas a partir de la salida de la última capa antes de
        la activación (contrato con §4.2).
    n_bins : int
        Número de cubetas de igual ancho en [0, 1].
    u_range : tupla (u_min, u_max) o None
        Rango para normalizar la incertidumbre; None usa el del conjunto.

    Retorna
    -------
    float
        ECE en [0, 1]; 0 indica calibración perfecta.
    """
    X = np.asarray(X)
    if X.ndim == 0:
        raise ValueError("X debe tener al menos una dimensión")
    n = X.shape[0]
    y = np.asarray(y).ravel()
    predictions = np.asarray(predictions).ravel()
    confidences = uncertainty_to_confidence(uncertainties, u_range)
    if not (len(y) == len(predictions) == len(confidences) == n):
        raise ValueError("X, y, uncertainties y predictions deben tener n observaciones")
    bins = calibration_bins(confidences, predictions == y, n_bins)
    filled = bins["count"] > 0
    weights = bins["count"][filled] / n
    gaps = np.abs(bins["accuracy"][filled] - bins["confidence"][filled])
    return float(np.sum(weights * gaps))


def uncertainty_accuracy_correlation(uncertainties, y, predictions):
    """Correlación de Pearson entre la incertidumbre y el acierto (1 o 0).

    Es la alternativa del material de apoyo para puntajes fuera de [0, 1].
    Un estimador útil da un valor negativo: más incertidumbre, menos aciertos.

    Parámetros
    ----------
    uncertainties : array_like, forma (n,)
    y : array_like, forma (n,)
        Etiquetas correctas.
    predictions : array_like, forma (n,)
        Etiquetas predichas.

    Retorna
    -------
    float
        Coeficiente en [-1, 1], o NaN si la incertidumbre o los aciertos son
        constantes (la correlación no está definida).
    """
    u = np.asarray(uncertainties, dtype=float).ravel()
    correct = (np.asarray(predictions).ravel() == np.asarray(y).ravel()).astype(float)
    if u.shape != correct.shape or u.size < 2:
        raise ValueError("se necesitan al menos dos observaciones de igual longitud")
    if np.std(u) == 0 or np.std(correct) == 0:
        return float("nan")
    return float(np.corrcoef(u, correct)[0, 1])

#### Pruebas unitarias de §4.3

Las pruebas usan datos construidos a mano, con resultados que se pueden calcular sin la
función, y no dependen de otras secciones:

| Función | Prueba | Resultado esperado |
| --- | --- | --- |
| `uncertainty_to_confidence` | Incertidumbres $[0, 2, 1]$ | Confianzas $[1, 0, 0.5]$ |
| | Rango fijo $(0, 1)$ con $[0.25, 1.5]$ | $[0.75, 0]$ (se recorta) |
| | Incertidumbre constante | Todas las confianzas valen 1 |
| | Incertidumbre negativa | `ValueError` |
| `calibration_bins` | Confianzas $[0.05, 0.15, 0.15, 1]$, aciertos $[1, 0, 1, 1]$ | Conteos 1, 2 y 1 en las cubetas 0, 1 y 9; tasa 0.5 en la cubeta 1 |
| | Confianza 1.2 | `ValueError` |
| `calculate_expected_calibration_error` | 4 obs. con confianza 0.75 y 3 aciertos; 4 con 0.25 y 1 acierto | ECE = 0 |
| | 10 obs. con confianza 0.95 y 8 aciertos; 10 con 0.55 y 5 aciertos | ECE = 0.5·0.15 + 0.5·0.05 = 0.10 |
| | Confianza 1 y todas incorrectas | ECE = 1 |
| | Incertidumbres $[0, 10]$ sin rango, primera correcta | ECE = 0 (normaliza y orienta) |
| | Longitudes distintas | `ValueError` |
| `uncertainty_accuracy_correlation` | Las dos incertidumbres altas fallan | $r = -1$ |
| | Todas correctas | NaN |

In [ ]:
import sys
import unittest


class TestUncertaintyToConfidence(unittest.TestCase):
    def test_extremos_y_orden(self):
        np.testing.assert_allclose(uncertainty_to_confidence([0.0, 2.0, 1.0]), [1.0, 0.0, 0.5])

    def test_rango_fijo_y_recorte(self):
        np.testing.assert_allclose(
            uncertainty_to_confidence([0.25, 1.5], u_range=(0.0, 1.0)), [0.75, 0.0])

    def test_incertidumbre_constante(self):
        np.testing.assert_allclose(uncertainty_to_confidence([3.0, 3.0, 3.0]), [1.0, 1.0, 1.0])

    def test_rechaza_negativos(self):
        with self.assertRaises(ValueError):
            uncertainty_to_confidence([0.1, -0.2])


class TestCalibrationBins(unittest.TestCase):
    def test_conteos_y_promedios(self):
        bins = calibration_bins([0.05, 0.15, 0.15, 1.0], [1, 0, 1, 1], n_bins=10)
        self.assertEqual(bins["count"].sum(), 4)
        self.assertEqual((bins["count"][0], bins["count"][1], bins["count"][9]), (1, 2, 1))
        self.assertAlmostEqual(bins["accuracy"][1], 0.5)
        self.assertAlmostEqual(bins["confidence"][1], 0.15)
        self.assertTrue(np.isnan(bins["accuracy"][5]))

    def test_rechaza_confianza_fuera_de_rango(self):
        with self.assertRaises(ValueError):
            calibration_bins([0.5, 1.2], [1, 0])


class TestExpectedCalibrationError(unittest.TestCase):
    @staticmethod
    def _caso(confianzas, aciertos):
        """Arma X, y, incertidumbres y predicciones con rango fijo (0, 1)."""
        n = len(confianzas)
        y = np.zeros(n, dtype=int)
        predictions = np.where(np.asarray(aciertos, dtype=bool), 0, 1)
        uncertainties = 1.0 - np.asarray(confianzas, dtype=float)
        return np.zeros((n, 2)), y, uncertainties, predictions

    def test_calibracion_perfecta(self):
        X, y, u, p = self._caso([0.75] * 4 + [0.25] * 4, [1, 1, 1, 0] + [1, 0, 0, 0])
        ece = calculate_expected_calibration_error(X, y, u, p, u_range=(0.0, 1.0))
        self.assertAlmostEqual(ece, 0.0)

    def test_ejemplo_calculado_a_mano(self):
        X, y, u, p = self._caso([0.95] * 10 + [0.55] * 10, [1] * 8 + [0] * 2 + [1] * 5 + [0] * 5)
        ece = calculate_expected_calibration_error(X, y, u, p, u_range=(0.0, 1.0))
        self.assertAlmostEqual(ece, 0.10)

    def test_peor_caso(self):
        X, y, u, p = self._caso([1.0] * 5, [0] * 5)
        ece = calculate_expected_calibration_error(X, y, u, p, u_range=(0.0, 1.0))
        self.assertAlmostEqual(ece, 1.0)

    def test_normaliza_y_orienta_la_incertidumbre(self):
        ece = calculate_expected_calibration_error(
            np.zeros((2, 2)), y=[0, 0], uncertainties=[0.0, 10.0], predictions=[0, 1])
        self.assertAlmostEqual(ece, 0.0)

    def test_rechaza_longitudes_distintas(self):
        with self.assertRaises(ValueError):
            calculate_expected_calibration_error(np.zeros((3, 2)), [0, 1, 0], [0.1, 0.2], [0, 1, 0])


class TestUncertaintyAccuracyCorrelation(unittest.TestCase):
    def test_correlacion_negativa_perfecta(self):
        r = uncertainty_accuracy_correlation([0.0, 0.0, 1.0, 1.0], [0, 0, 0, 0], [0, 0, 1, 1])
        self.assertAlmostEqual(r, -1.0)

    def test_aciertos_constantes(self):
        r = uncertainty_accuracy_correlation([0.1, 0.5, 0.9], [1, 0, 1], [1, 0, 1])
        self.assertTrue(np.isnan(r))


suite_s4_3 = unittest.TestSuite()
for caso in (TestUncertaintyToConfidence, TestCalibrationBins,
             TestExpectedCalibrationError, TestUncertaintyAccuracyCorrelation):
    suite_s4_3.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(caso))
resultado_s4_3 = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite_s4_3)
assert resultado_s4_3.wasSuccessful(), "Fallaron pruebas unitarias de §4.3"

### 4.4 Evaluación de ensambles: ECE y tiempo de ejecución

*Responsable: P3 (T7).*

#### Planteamiento y protocolo

El enunciado pide medir el ECE (§4.3) y el tiempo de ejecución de ensambles de $N \in \{10, 100, 1000\}$
miembros (§4.2), entrenados con el 70 % de los datos y evaluados con el 30 % restante, en 10 particiones;
reportar la media y la desviación estándar del ECE de cada configuración, aplicar una prueba estadística y
graficar y comentar los resultados. Las decisiones que el enunciado no fija están en D12 de `JOURNAL.md`, y
la lectura de la frase incompleta sobre la prueba estadística, en D7 (propuestas por P3 en T7).

| Aspecto | Decisión |
| --- | --- |
| Particiones | Las 10 particiones 70/30 estratificadas de T3 (`S4_SEMILLAS_70_30` = 4500–4509), con el preprocesamiento base y `l2=USE_L2` (sin L2, decisión de §4.1) |
| Configuraciones | $N \in \{10, 100, 1000\}$ miembros de mínimos cuadrados (`train_ensemble`, D11). En la partición $i$, las tres usan los mismos datos y la misma semilla para repartir las filas entre los miembros (`SEEDS["s4_4_evaluacion"] + i`): 30 combinaciones, emparejadas por partición |
| ECE (principal) | `calculate_expected_calibration_error` como se acordó en D6: confianza $s = 1 - u_{\text{norm}}$, con la varianza normalizada min-max al rango del 30 % de prueba, 10 cubetas y las predicciones del signo del puntaje medio |
| Sensibilidad del ECE | El mismo ECE con el rango de normalización recortado al percentil 99 de la varianza de prueba (`u_range` = (mínimo, percentil 99)): el 1 % más incierto recibe confianza 0 y el resto se reparte en $[0, 1]$. No usa las etiquetas. Se agrega porque unas pocas varianzas extremas concentran casi todas las confianzas en la última cubeta (ver «Limitaciones») |
| Complementos | Correlación de Pearson entre varianza y acierto (D6), tasa de aciertos del ensamble y varianza media de las predicciones correctas e incorrectas |
| Tiempo | Reloj de pared (`time.perf_counter`) de tres etapas: **entrenar** (`train_ensemble` sobre el 70 %: reparto de filas y $N$ ajustes de mínimos cuadrados), **evaluar** (`run_ensemble_uq` sobre el 30 %: puntajes, varianzas y predicciones) y **ECE** (`calculate_expected_calibration_error`). El total es la suma de las tres. No incluye la carga ni el preprocesamiento, que son iguales para los tres $N$ y se reportan aparte |
| Medición | Antes de la primera medición se entrena y evalúa un ensamble pequeño, para que la inicialización de BLAS no recaiga en la primera configuración. En cada partición, el orden de los tres $N$ se sortea con un generador creado con `SEEDS["s4_4_evaluacion"]`, para que la deriva de la máquina (temperatura, otros procesos) no favorezca siempre al mismo $N$. La celda imprime el procesador, los núcleos y la biblioteca BLAS con sus hilos |
| Prueba (D7) | Friedman sobre los 10 bloques (particiones) × 3 configuraciones, con $\alpha = 0.05$: ¿existe una diferencia significativa de ECE entre las configuraciones? Las comparaciones por pares usan Wilcoxon de rangos con signo bilateral con la corrección de Holm, y la $W$ de Kendall mide el tamaño del efecto. La misma prueba se aplica, como complemento, al ECE con rango recortado, a la correlación de Pearson y al tiempo total |

**Por qué Friedman.** Las tres configuraciones se evalúan sobre las mismas 10 particiones, así que las
mediciones están emparejadas por partición. Friedman (1937) es la prueba no paramétrica para comparar
$k \geq 3$ tratamientos medidos en los mismos bloques, y Demšar (2006) la recomienda para comparar
clasificadores sobre varios conjuntos de datos. No supone normalidad, que no se puede comprobar con 10 valores
por grupo, y extiende a tres grupos la prueba de Wilcoxon de §4.1. Con 10 bloques, la menor $p$ exacta de
Wilcoxon es $2/2^{10} \approx 0.002$, así que una comparación por pares puede ser significativa aun después de
la corrección de Holm (1979), que en el peor caso multiplica esa $p$ por 3. La $W$ de Kendall,
$W = \chi^2_F / (n(k - 1))$ (Kendall y Babington Smith, 1939), vale 0 si los bloques no coinciden en el orden
de las configuraciones y 1 si todos las ordenan igual.

**Limitaciones que se tienen en cuenta al analizar.**

- *Normalización min-max (D6).* La varianza tiene una cola larga, sobre todo con $N = 1000$, donde cada
  miembro tiene menos filas que pesos (D11). La normalización min-max manda entonces casi todas las
  observaciones a la cubeta de confianza más alta, y el ECE queda cerca de la tasa de error. Por eso se
  reportan la sensibilidad con el rango recortado y la correlación de Pearson. La celda del experimento
  imprime los cuantiles de la varianza.
- *Duplicados (D9).* Cerca del 44.5 % de las filas de prueba de las particiones 70/30 tienen una copia exacta
  de sus atributos en entrenamiento, así que la tasa de aciertos y el ECE son optimistas. El efecto es el
  mismo para los tres $N$, porque comparten las particiones.
- *Tiempos.* Dependen del procesador, de la biblioteca BLAS y de la carga de la máquina. Se comparan
  configuraciones medidas en la misma máquina y en la misma ejecución; los valores absolutos cambian de una
  máquina a otra.

**Costo.** En un Intel Core i3-1115G4 (Linux, OpenBLAS con 4 hilos), las 30 combinaciones tardaron unos
70 s, más 0.5 s de preprocesamiento por partición. Con `TP3_QUICK=1` se usan 3 particiones, solo para
comprobar que el código se ejecuta.

**Referencias**

- J. Demšar, «Statistical comparisons of classifiers over multiple data sets», *Journal of Machine
  Learning Research*, vol. 7, pp. 1–30, 2006.
- M. Friedman, «The use of ranks to avoid the assumption of normality implicit in the analysis of
  variance», *Journal of the American Statistical Association*, vol. 32, n.º 200, pp. 675–701, 1937.
- S. Holm, «A simple sequentially rejective multiple test procedure», *Scandinavian Journal of
  Statistics*, vol. 6, pp. 65–70, 1979.
- M. G. Kendall y B. Babington Smith, «The problem of m rankings», *The Annals of Mathematical
  Statistics*, vol. 10, n.º 3, pp. 275–287, 1939.

#### Funciones auxiliares y sus pruebas

| Función | Qué hace |
| --- | --- |
| `s4_4_evaluar_ensamble(X_train, y_train, X_test, y_test, N, seed, n_bins=10, cuantil_rango=0.99)` | Entrena un ensamble de N miembros con `train_ensemble`, lo evalúa con `run_ensemble_uq` y calcula su ECE con `calculate_expected_calibration_error`, midiendo el tiempo de cada etapa. Devuelve un `dict` con el ECE (rango min-max y rango recortado al cuantil `cuantil_rango` de la varianza), la correlación de Pearson, la tasa de aciertos, resúmenes de la varianza, las cubetas de los dos ECE y los tiempos. |
| `s4_4_holm(p_valores)` | Corrección de Holm para comparaciones múltiples: ordena las $m$ $p$ de menor a mayor, multiplica la $j$-ésima por $m - j + 1$, impone que no decrezcan y las recorta a 1. Devuelve las $p$ ajustadas en el orden original. |
| `s4_4_comparar(valores, alpha=0.05)` | Recibe una tabla de $n$ bloques × $k \geq 3$ configuraciones (un `DataFrame` con una columna por configuración) y aplica Friedman, la $W$ de Kendall y, por pares, Wilcoxon de rangos con signo con la corrección de Holm. |

| Función | Prueba | Resultado esperado |
| --- | --- | --- |
| `s4_4_evaluar_ensamble` | Dos nubes gaussianas que se solapan un poco, en $\mathbb{R}^3$, con $N = 5$ | ECE, Pearson, tasa de aciertos, mediana de la varianza y cubetas iguales a llamar a mano a `train_ensemble`, `run_ensemble_uq` y las funciones de §4.3; 60 filas por miembro; tiempos no negativos y total igual a la suma |
| | `cuantil_rango` = 1 y 0.9 | Con 1, el ECE recortado es igual al min-max (el cuantil 100 % es el máximo); con 0.9, es igual al ECE calculado a mano con `u_range` = (mínimo, percentil 90), y el 10 % más incierto cae en la cubeta de confianza 0 |
| | Misma semilla dos veces | Mismas métricas (salvo los tiempos) |
| | `cuantil_rango` fuera de $(0, 1]$ o $N$ mayor que el número de filas | `ValueError` |
| `s4_4_holm` | $p = [0.01, 0.04, 0.03]$ | $[0.03, 0.06, 0.06]$: $3 \cdot 0.01$ y $\max(2 \cdot 0.03,\ 1 \cdot 0.04)$ para las otras dos |
| | $p = [0.6, 0.9]$ y una sola $p$ | $[1, 1]$ ($0.6 \cdot 2 = 1.2$ se recorta a 1) y la misma $p$ |
| | $p$ fuera de $[0, 1]$ o lista vacía | `ValueError` |
| `s4_4_comparar` | 10 bloques en los que las tres configuraciones quedan siempre en el mismo orden | $\chi^2_F = 2n = 20$ (cálculo a mano con rangos 1, 2 y 3), $W = 1$ y $p < 0.05$; en cada par, $p$ exacta de Wilcoxon $= 2/2^{10}$ y $p$ de Holm $= 3 \cdot 2/2^{10}$, significativas |
| | Bloques muy dispersos ($\sigma = 1$) con diferencias pequeñas (+0.05) y constantes | Friedman y los tres pares significativos, aunque Mann-Whitney, sin emparejar, da $p > 0.05$ |
| | Dos configuraciones iguales y la tercera siempre menor | Friedman significativa; solo los dos pares con la tercera son significativos |
| | Tres configuraciones idénticas | $\chi^2_F = 0$, $p = 1$, $W = 0$ y ningún par significativo |
| | Ruido sin diferencias (semilla fija) | $p > 0.05$ |
| | Menos de 3 configuraciones o de 2 bloques, o un `NaN` | `ValueError` |

In [ ]:
import itertools
import os
import platform
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import threadpoolctl
from scipy import stats

S4_4_TAMANOS = (10, 100, 1000)
s4_4_colores = {10: "#2a78d6", 100: "#eb6834", 1000: "#1baf7a"}  # N = 10, 100 y 1000 (§4.4)


def s4_4_evaluar_ensamble(X_train, y_train, X_test, y_test, N, seed, n_bins=10, cuantil_rango=0.99):
    """Entrena un ensamble de N miembros, lo evalúa en prueba y calcula su ECE, midiendo cada etapa.

    Etapas cronometradas con `time.perf_counter` (reloj de pared):
      entrenar  `train_ensemble(X_train, y_train, N, seed)`: reparto de filas y N ajustes.
      evaluar   `run_ensemble_uq(X_test, ensamble, return_predictions=True)`: varianzas y predicciones.
      ece       `calculate_expected_calibration_error` con la normalización min-max de D6.
    El ECE con el rango recortado, u_range = (mínimo, cuantil `cuantil_rango` de la varianza de
    prueba), la correlación de Pearson y los resúmenes se calculan después y no se cronometran.

    Retorna un dict con 'ece', 'ece_recortado', 'pearson', 'tasa_aciertos', 'u_media_correctas',
    'u_media_incorrectas', 'u_mediana', 'u_maxima', 'filas_por_miembro' (mínimo y máximo),
    'cubetas' y 'cubetas_recortado' (salidas de `calibration_bins`) y los tiempos 't_entrenar',
    't_evaluar', 't_ece' y 't_total' (segundos).
    """
    if not 0 < cuantil_rango <= 1:
        raise ValueError("cuantil_rango debe estar en (0, 1]")
    y_test = np.asarray(y_test).ravel()
    inicio = time.perf_counter()
    ensamble = train_ensemble(X_train, y_train, N=N, seed=seed)
    t_entrenar = time.perf_counter() - inicio
    inicio = time.perf_counter()
    u, pred = run_ensemble_uq(X_test, ensamble, return_predictions=True)
    t_evaluar = time.perf_counter() - inicio
    inicio = time.perf_counter()
    ece = calculate_expected_calibration_error(X_test, y_test, u, pred, n_bins=n_bins)
    t_ece = time.perf_counter() - inicio

    rango_recortado = (float(u.min()), float(np.quantile(u, cuantil_rango)))
    aciertos = pred == y_test
    tamanos = [len(idx) for idx in ensamble["indices"]]
    return {
        "ece": ece,
        "ece_recortado": calculate_expected_calibration_error(X_test, y_test, u, pred, n_bins=n_bins,
                                                              u_range=rango_recortado),
        "pearson": uncertainty_accuracy_correlation(u, y_test, pred),
        "tasa_aciertos": float(aciertos.mean()),
        "u_media_correctas": float(u[aciertos].mean()) if aciertos.any() else np.nan,
        "u_media_incorrectas": float(u[~aciertos].mean()) if (~aciertos).any() else np.nan,
        "u_mediana": float(np.median(u)),
        "u_maxima": float(u.max()),
        "filas_por_miembro": (min(tamanos), max(tamanos)),
        "cubetas": calibration_bins(uncertainty_to_confidence(u), aciertos, n_bins),
        "cubetas_recortado": calibration_bins(uncertainty_to_confidence(u, rango_recortado), aciertos, n_bins),
        "t_entrenar": t_entrenar,
        "t_evaluar": t_evaluar,
        "t_ece": t_ece,
        "t_total": t_entrenar + t_evaluar + t_ece,
    }


def s4_4_holm(p_valores):
    """Ajusta p-valores por comparaciones múltiples con el método de Holm (1979).

    Con las m p ordenadas de menor a mayor, la j-ésima (j = 1, …, m) se multiplica por m − j + 1;
    luego se impone que las ajustadas no decrezcan en ese orden y se recortan a 1. Devuelve las p
    ajustadas en el orden original, forma (m,). Rechazar las hipótesis con p ajustada < α mantiene
    en α la probabilidad de cometer algún falso positivo entre las m comparaciones.
    """
    p = np.asarray(p_valores, dtype=float).ravel()
    if p.size == 0 or not np.all((p >= 0) & (p <= 1)):
        raise ValueError("p_valores debe ser no vacío y estar en [0, 1]")
    m = p.size
    orden = np.argsort(p, kind="stable")
    ajustadas = np.empty(m)
    ajustadas[orden] = np.minimum(np.maximum.accumulate(p[orden] * (m - np.arange(m))), 1.0)
    return ajustadas


def s4_4_comparar(valores, alpha=0.05):
    """Compara k ≥ 3 configuraciones medidas sobre los mismos n bloques (particiones).

    `valores` es una tabla n × k (un DataFrame con una fila por bloque y una columna por
    configuración). Aplica la prueba de Friedman (H0: las k configuraciones tienen la misma
    distribución), la W de Kendall = χ²_F / (n (k − 1)) como tamaño del efecto (0: los bloques no
    coinciden en el orden; 1: todos ordenan igual) y, para cada par de configuraciones, Wilcoxon
    de rangos con signo bilateral, con las p ajustadas por Holm. Si todos los bloques tienen sus k
    valores iguales, o un par tiene todas sus diferencias nulas, no hay diferencia que probar:
    estadístico 0 y p = 1.

    Retorna un dict con 'friedman_estadistico', 'friedman_p', 'kendall_w', 'hay_diferencia'
    (friedman_p < alpha) y 'pares': un DataFrame con una fila por par y las columnas 'a', 'b',
    'mediana_diferencia' (a − b), 'wilcoxon_estadistico', 'p', 'p_holm' y 'significativo'.
    """
    tabla = pd.DataFrame(valores).astype(float)
    n, k = tabla.shape
    if k < 3 or n < 2:
        raise ValueError("Se necesitan al menos 3 configuraciones y 2 bloques")
    if tabla.isna().to_numpy().any():
        raise ValueError("valores no puede tener NaN")
    if np.all(np.ptp(tabla.to_numpy(), axis=1) == 0):
        chi2, p = 0.0, 1.0
    else:
        friedman = stats.friedmanchisquare(*(tabla[c] for c in tabla.columns))
        chi2, p = float(friedman.statistic), float(friedman.pvalue)
    filas = []
    for a, b in itertools.combinations(tabla.columns, 2):
        d = tabla[a] - tabla[b]
        if np.all(d == 0):
            w_est, w_p = 0.0, 1.0
        else:
            w = stats.wilcoxon(tabla[a], tabla[b], alternative="two-sided")
            w_est, w_p = float(w.statistic), float(w.pvalue)
        filas.append({"a": a, "b": b, "mediana_diferencia": float(np.median(d)),
                      "wilcoxon_estadistico": w_est, "p": w_p})
    pares = pd.DataFrame(filas)
    pares["p_holm"] = s4_4_holm(pares["p"])
    pares["significativo"] = pares["p_holm"] < alpha
    return {"friedman_estadistico": chi2, "friedman_p": p, "kendall_w": chi2 / (n * (k - 1)),
            "hay_diferencia": bool(p < alpha), "pares": pares}

In [ ]:
import sys
import unittest


class TestEvaluacionEnsamble(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        # Dos nubes en R³ centradas en ±1 que se solapan un poco: hay aciertos y errores.
        rng = np.random.default_rng(SEEDS["s4_4_evaluacion"])
        y = rng.integers(0, 2, size=400)
        X = rng.normal(size=(400, 3)) + (2 * y[:, None] - 1)
        cls.datos = (X[:300], y[:300], X[300:], y[300:])

    def test_coincide_con_las_funciones_de_4_2_y_4_3(self):
        X_tr, y_tr, X_te, y_te = self.datos
        r = s4_4_evaluar_ensamble(X_tr, y_tr, X_te, y_te, N=5, seed=1)
        u, pred = run_ensemble_uq(X_te, train_ensemble(X_tr, y_tr, N=5, seed=1), return_predictions=True)
        np.testing.assert_allclose(r["ece"], calculate_expected_calibration_error(X_te, y_te, u, pred))
        np.testing.assert_allclose(r["pearson"], uncertainty_accuracy_correlation(u, y_te, pred))
        np.testing.assert_allclose(r["u_mediana"], np.median(u))
        self.assertAlmostEqual(r["tasa_aciertos"], np.mean(pred == y_te))
        self.assertTrue(0.8 < r["tasa_aciertos"] < 1.0)
        np.testing.assert_array_equal(r["cubetas"]["count"],
                                      calibration_bins(uncertainty_to_confidence(u), pred == y_te)["count"])
        self.assertEqual(r["filas_por_miembro"], (60, 60))
        self.assertTrue(all(r[t] >= 0 for t in ("t_entrenar", "t_evaluar", "t_ece")))
        self.assertAlmostEqual(r["t_total"], r["t_entrenar"] + r["t_evaluar"] + r["t_ece"])

    def test_rango_recortado(self):
        X_tr, y_tr, X_te, y_te = self.datos
        r = s4_4_evaluar_ensamble(X_tr, y_tr, X_te, y_te, N=5, seed=1, cuantil_rango=1.0)
        self.assertAlmostEqual(r["ece_recortado"], r["ece"])  # el cuantil 100 % es el máximo
        r = s4_4_evaluar_ensamble(X_tr, y_tr, X_te, y_te, N=5, seed=1, cuantil_rango=0.9)
        u, pred = run_ensemble_uq(X_te, train_ensemble(X_tr, y_tr, N=5, seed=1), return_predictions=True)
        rango = (u.min(), np.quantile(u, 0.9))
        np.testing.assert_allclose(r["ece_recortado"],
                                   calculate_expected_calibration_error(X_te, y_te, u, pred, u_range=rango))
        # El 10 % más incierto (varianza ≥ percentil 90) recibe confianza 0 y cae en la cubeta 0, junto
        # con las observaciones de confianza menor que 0.1.
        self.assertGreaterEqual(r["cubetas_recortado"]["count"][0], max(np.sum(u >= rango[1]), 10))

    def test_reproducible_con_la_misma_semilla(self):
        X_tr, y_tr, X_te, y_te = self.datos
        a = s4_4_evaluar_ensamble(X_tr, y_tr, X_te, y_te, N=4, seed=2)
        b = s4_4_evaluar_ensamble(X_tr, y_tr, X_te, y_te, N=4, seed=2)
        for clave in ("ece", "ece_recortado", "pearson", "tasa_aciertos", "u_mediana", "u_maxima"):
            self.assertEqual(a[clave], b[clave])

    def test_valida_argumentos(self):
        X_tr, y_tr, X_te, y_te = self.datos
        for q in (0.0, 1.5):
            with self.assertRaises(ValueError):
                s4_4_evaluar_ensamble(X_tr, y_tr, X_te, y_te, N=5, seed=1, cuantil_rango=q)
        with self.assertRaises(ValueError):
            s4_4_evaluar_ensamble(X_tr, y_tr, X_te, y_te, N=301, seed=1)  # más miembros que filas


class TestHolm(unittest.TestCase):
    def test_ejemplo_calculado_a_mano(self):
        # Ordenadas: 0.01·3 = 0.03; 0.03·2 = 0.06; 0.04·1 = 0.04 → no decreciente: 0.06.
        np.testing.assert_allclose(s4_4_holm([0.01, 0.04, 0.03]), [0.03, 0.06, 0.06])

    def test_recorte_y_una_sola_p(self):
        np.testing.assert_allclose(s4_4_holm([0.6, 0.9]), [1.0, 1.0])  # 0.6 · 2 = 1.2 se recorta a 1
        np.testing.assert_allclose(s4_4_holm([0.2]), [0.2])

    def test_rechaza_valores_invalidos(self):
        for p in ([], [0.1, 1.2], [-0.1], [np.nan]):
            with self.assertRaises(ValueError):
                s4_4_holm(p)


class TestComparacionConfiguraciones(unittest.TestCase):
    def setUp(self):
        self.rng = np.random.default_rng(SEEDS["s4_4_evaluacion"] + 1)
        self.base = self.rng.normal(0.1, 0.02, size=10)

    def test_orden_constante_en_todos_los_bloques(self):
        b = self.base + 0.05 + self.rng.uniform(0, 0.01, 10)
        c = b + 0.05 + self.rng.uniform(0, 0.01, 10)
        r = s4_4_comparar(pd.DataFrame({10: self.base, 100: b, 1000: c}))
        # Rangos 1, 2 y 3 en cada bloque: χ² = 12/(n·k·(k+1)) · Σ R_j² − 3n(k + 1) = 2n = 20.
        self.assertAlmostEqual(r["friedman_estadistico"], 20.0)
        self.assertAlmostEqual(r["friedman_p"], stats.chi2.sf(20.0, 2))
        self.assertAlmostEqual(r["kendall_w"], 1.0)
        self.assertTrue(r["hay_diferencia"] and r["pares"]["significativo"].all())
        self.assertEqual(len(r["pares"]), 3)
        # Las 10 diferencias de cada par tienen el mismo signo: p exacta de Wilcoxon = 2/2¹⁰. Holm
        # multiplica la menor por 3 y, como las tres son iguales, todas quedan en 3 · 2/2¹⁰.
        np.testing.assert_allclose(r["pares"]["p"], 2 / 2**10)
        np.testing.assert_allclose(r["pares"]["p_holm"], 3 * 2 / 2**10)

    def test_aprovecha_el_emparejamiento(self):
        # Bloques muy dispersos (σ = 1) y una diferencia pequeña (+0.05) que se repite en todos: las
        # pruebas emparejadas la detectan; Mann-Whitney, sin emparejar, no la distingue del azar.
        a = self.rng.normal(0, 1, size=10)
        b = a + 0.05 + self.rng.uniform(0, 0.001, 10)
        c = b + 0.05 + self.rng.uniform(0, 0.001, 10)
        r = s4_4_comparar(pd.DataFrame({"a": a, "b": b, "c": c}))
        self.assertTrue(r["hay_diferencia"] and r["pares"]["significativo"].all())
        self.assertGreater(stats.mannwhitneyu(a, b, alternative="two-sided").pvalue, 0.05)

    def test_solo_la_tercera_configuracion_difiere(self):
        c = self.base - 0.05 - self.rng.uniform(0, 0.01, 10)
        r = s4_4_comparar(pd.DataFrame({"a": self.base, "b": self.base, "c": c}))
        self.assertTrue(r["hay_diferencia"])
        significativos = {(f.a, f.b): f.significativo for f in r["pares"].itertuples()}
        self.assertEqual(significativos, {("a", "b"): False, ("a", "c"): True, ("b", "c"): True})
        self.assertEqual(r["pares"].set_index(["a", "b"]).loc[("a", "b"), "p"], 1.0)

    def test_configuraciones_identicas(self):
        r = s4_4_comparar(pd.DataFrame({"a": self.base, "b": self.base, "c": self.base}))
        self.assertEqual((r["friedman_estadistico"], r["friedman_p"], r["kendall_w"]), (0.0, 1.0, 0.0))
        self.assertFalse(r["hay_diferencia"] or r["pares"]["significativo"].any())

    def test_ruido_sin_diferencias(self):
        r = s4_4_comparar(pd.DataFrame(self.rng.normal(size=(10, 3)), columns=[10, 100, 1000]))
        self.assertGreater(r["friedman_p"], 0.05)
        self.assertFalse(r["hay_diferencia"])

    def test_rechaza_tablas_invalidas(self):
        with self.assertRaises(ValueError):
            s4_4_comparar(pd.DataFrame({"a": self.base, "b": self.base}))  # dos configuraciones
        with self.assertRaises(ValueError):
            s4_4_comparar(pd.DataFrame({"a": [0.1], "b": [0.2], "c": [0.3]}))  # un bloque
        with self.assertRaises(ValueError):
            s4_4_comparar(pd.DataFrame({"a": [0.1, np.nan], "b": [0.2, 0.1], "c": [0.3, 0.2]}))


suite_s4_4 = unittest.TestSuite()
for caso in (TestEvaluacionEnsamble, TestHolm, TestComparacionConfiguraciones):
    suite_s4_4.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(caso))
resultado_s4_4 = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite_s4_4)
assert resultado_s4_4.wasSuccessful(), "Fallaron pruebas unitarias de §4.4"

#### ECE y tiempo de ejecución en las 10 particiones

La celda siguiente aplica el protocolo a las 10 particiones 70/30. En cada una preprocesa los datos una vez y
entrena y evalúa los tres ensambles en orden sorteado. Primero imprime el entorno de la medición; al final, el
reparto de filas entre los miembros, los cuantiles de la varianza y la media y la desviación estándar de cada
métrica por configuración. `s4_4_resultados` guarda una fila por combinación de partición y $N$, y
`s4_4_cubetas`, las cubetas de los dos ECE de cada combinación.

In [ ]:
# Entorno de la medición: los tiempos dependen del procesador y de la biblioteca BLAS.
s4_4_cpu = platform.processor() or platform.machine()
try:  # nombre comercial del procesador en macOS (sysctl) y Linux (/proc/cpuinfo)
    if sys.platform == "darwin":
        s4_4_cpu = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"], capture_output=True,
                                  text=True, check=True).stdout.strip()
    elif Path("/proc/cpuinfo").is_file():
        s4_4_cpu = next(linea.split(":", 1)[1].strip() for linea in Path("/proc/cpuinfo").read_text().splitlines()
                        if linea.startswith("model name"))
except (OSError, subprocess.SubprocessError, StopIteration):
    pass
s4_4_blas = [f"{b['internal_api']} {b['version']} ({b['num_threads']} hilos)"
             for b in threadpoolctl.threadpool_info() if b["user_api"] == "blas"]
print(f"Procesador: {s4_4_cpu} · {os.cpu_count()} núcleos lógicos · BLAS: {', '.join(s4_4_blas) or 'no detectada'}")

s4_4_l2 = USE_L2  # decisión de §4.1 (T4): sin L2
s4_4_num_particiones = quick_or_full(3, 10)
s4_4_rng = np.random.default_rng(SEEDS["s4_4_evaluacion"])  # orden de los N en cada partición
s4_4_filas, s4_4_cubetas = [], {}
s4_4_inicio = time.perf_counter()
for s4_4_i, s4_4_semilla_part in enumerate(S4_SEMILLAS_70_30[:s4_4_num_particiones]):
    s4_4_t0 = time.perf_counter()
    s4_4_tr_df, s4_4_te_df = split_unsw(s4_unsw, 0.3, s4_4_semilla_part)
    s4_4_Xtr, s4_4_Xte, s4_4_ytr, s4_4_yte = preprocess_unsw(s4_4_tr_df, s4_4_te_df, l2=s4_4_l2)
    s4_4_t_prep = time.perf_counter() - s4_4_t0
    if s4_4_i == 0:  # calentamiento de BLAS/LAPACK: no se registra
        s4_4_evaluar_ensamble(s4_4_Xtr[:5000], s4_4_ytr[:5000], s4_4_Xte[:2000], s4_4_yte[:2000], N=10, seed=0)
    s4_4_semilla_ens = SEEDS["s4_4_evaluacion"] + s4_4_i
    for s4_4_pos, s4_4_N in enumerate(s4_4_rng.permutation(S4_4_TAMANOS).tolist(), start=1):
        s4_4_r = s4_4_evaluar_ensamble(s4_4_Xtr, s4_4_ytr, s4_4_Xte, s4_4_yte, s4_4_N, s4_4_semilla_ens)
        s4_4_cubetas[(s4_4_i + 1, s4_4_N)] = {"min-max": s4_4_r.pop("cubetas"),
                                               "recortado": s4_4_r.pop("cubetas_recortado")}
        s4_4_filas_min, s4_4_filas_max = s4_4_r.pop("filas_por_miembro")
        s4_4_filas.append({"particion": s4_4_i + 1, "semilla_particion": s4_4_semilla_part,
                           "semilla_ensamble": s4_4_semilla_ens, "N": s4_4_N, "orden": s4_4_pos,
                           "D": s4_4_Xtr.shape[1], "filas_min": s4_4_filas_min, "filas_max": s4_4_filas_max,
                           "t_preprocesar": s4_4_t_prep, **s4_4_r})
    del s4_4_Xtr, s4_4_Xte  # ~0.4 GB
s4_4_resultados = pd.DataFrame(s4_4_filas)
print(f"{s4_4_num_particiones} particiones × {len(S4_4_TAMANOS)} configuraciones en "
      f"{time.perf_counter() - s4_4_inicio:.0f} s (preprocesamiento: "
      f"{s4_4_resultados.groupby('particion')['t_preprocesar'].first().mean():.2f} s por partición, "
      f"fuera de los tiempos medidos)")

# Filas por miembro y forma de la varianza de prueba (medianas sobre las particiones).
print("\nPor configuración (medianas sobre las particiones): filas por miembro, pesos por miembro (D + 1) y varianza")
print(s4_4_resultados.groupby("N").agg(
    filas_por_miembro=("filas_min", "min"), pesos_por_miembro=("D", lambda d: int(d.max()) + 1),
    u_mediana=("u_mediana", "median"), u_maxima=("u_maxima", "median"),
    u_correctas=("u_media_correctas", "median"), u_incorrectas=("u_media_incorrectas", "median"))
    .assign(maxima_sobre_mediana=lambda t: t["u_maxima"] / t["u_mediana"],
            incorrectas_sobre_correctas=lambda t: t["u_incorrectas"] / t["u_correctas"])
    .to_string(float_format=lambda v: f"{v:.4g}"))

# Media y desviación estándar sobre las particiones (prueba de 30 %).
(s4_4_resultados.groupby("N")[["ece", "ece_recortado", "pearson", "tasa_aciertos",
                               "t_entrenar", "t_evaluar", "t_ece", "t_total"]]
 .agg(["mean", "std"]).style.format("{:.4f}"))

#### Prueba estadística

La hipótesis principal es la del ECE con la normalización de D6: Friedman sobre las particiones y, por pares,
Wilcoxon con la corrección de Holm (D7). La misma prueba se repite para el ECE con rango recortado, la
correlación de Pearson y el tiempo total; son complementos para interpretar el resultado principal, no
hipótesis adicionales. En cada tabla de pares, `mediana_diferencia` es la mediana, sobre las particiones, de
la métrica de la primera configuración menos la de la segunda.

In [ ]:
# Prueba estadística (D7): Friedman sobre las particiones y, por pares, Wilcoxon con Holm.
s4_4_pruebas = {}
for s4_4_metrica, s4_4_nombre in (("ece", "ECE con la normalización min-max de D6 (principal)"),
                                  ("ece_recortado", "ECE con el rango recortado al percentil 99 (sensibilidad)"),
                                  ("pearson", "Correlación de Pearson entre varianza y acierto (complemento)"),
                                  ("t_total", "Tiempo total (complemento)")):
    s4_4_c = s4_4_comparar(s4_4_resultados.pivot(index="particion", columns="N", values=s4_4_metrica))
    s4_4_pruebas[s4_4_metrica] = s4_4_c
    print(f"{s4_4_nombre}: Friedman χ² = {s4_4_c['friedman_estadistico']:.2f}, p = {s4_4_c['friedman_p']:.3g}, "
          f"W de Kendall = {s4_4_c['kendall_w']:.2f} → "
          f"{'hay' if s4_4_c['hay_diferencia'] else 'no hay'} diferencia significativa (α = 0.05)")
    print(s4_4_c["pares"].assign(par=lambda t: "N = " + t["a"].astype(str) + " frente a N = " + t["b"].astype(str))
          [["par", "mediana_diferencia", "wilcoxon_estadistico", "p", "p_holm", "significativo"]]
          .to_string(index=False, float_format=lambda v: f"{v:.4g}"))
    print()

#### Gráficas

1. **ECE por configuración**, con las dos normalizaciones. Cada línea gris une las tres configuraciones de una
   misma partición; el marcador negro es la media y la barra, ± una desviación estándar.
2. **Tiempo por etapa** (entrenar, evaluar y total), con la misma disposición. El cálculo del ECE tarda
   milisegundos y solo aparece en la tabla y en el total.
3. **Diagramas de fiabilidad** de la primera partición: tasa de aciertos de cada cubeta frente a su
   confianza, para cada $N$ y cada normalización. Una calibración perfecta deja las barras sobre la diagonal.
   El título indica qué fracción de las observaciones cae en la cubeta más alta, $[0.9, 1]$. Las cubetas con
   menos de 30 observaciones se dibujan más claras, porque su tasa de aciertos es muy variable.

In [ ]:
# 1. ECE y 2. tiempo por configuración: líneas por partición, puntos por N y media ± desviación estándar.
s4_4_fig_ece, s4_4_ejes_ece = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)
s4_4_fig_t, s4_4_ejes_t = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
s4_4_paneles = [
    (s4_4_ejes_ece[0], "ece", "ECE (prueba de 30 %)",
     f"Normalización min-max (D6, principal)\nFriedman p = {s4_4_pruebas['ece']['friedman_p']:.3g}"),
    (s4_4_ejes_ece[1], "ece_recortado", "ECE (prueba de 30 %)",
     f"Rango recortado al percentil 99 (sensibilidad)\nFriedman p = {s4_4_pruebas['ece_recortado']['friedman_p']:.3g}"),
    (s4_4_ejes_t[0], "t_entrenar", "Segundos", "Entrenar (70 %)"),
    (s4_4_ejes_t[1], "t_evaluar", "Segundos", "Evaluar (30 %)"),
    (s4_4_ejes_t[2], "t_total", "Segundos", "Total (entrenar + evaluar + ECE)"),
]
s4_4_x = np.arange(len(S4_4_TAMANOS))
for s4_4_ax, s4_4_metrica, s4_4_etiqueta, s4_4_titulo in s4_4_paneles:
    s4_4_tabla = s4_4_resultados.pivot(index="particion", columns="N", values=s4_4_metrica)[list(S4_4_TAMANOS)]
    for _, s4_4_fila in s4_4_tabla.iterrows():
        s4_4_ax.plot(s4_4_x, s4_4_fila.to_numpy(), color="#c3c2b7", linewidth=0.8, zorder=1)
    for s4_4_j, s4_4_N in enumerate(S4_4_TAMANOS):
        s4_4_ax.scatter(np.full(len(s4_4_tabla), s4_4_x[s4_4_j]), s4_4_tabla[s4_4_N], s=36,
                        color=s4_4_colores[s4_4_N], edgecolor="white", linewidth=1, zorder=2, label=f"N = {s4_4_N}")
        s4_4_ax.errorbar(s4_4_x[s4_4_j] + 0.15, s4_4_tabla[s4_4_N].mean(), yerr=s4_4_tabla[s4_4_N].std(), fmt="o",
                         color="#0b0b0b", markersize=5, capsize=3, linewidth=1.2, zorder=3)
    s4_4_ax.set_xticks(s4_4_x, [f"N = {N}" for N in S4_4_TAMANOS])
    s4_4_ax.set_xlim(-0.4, len(S4_4_TAMANOS) - 0.4)
    s4_4_ax.set_ylabel(s4_4_etiqueta)
    s4_4_ax.set_title(s4_4_titulo, fontsize=10)
    s4_4_ax.grid(True, axis="y", color="#e8e7e2", linewidth=0.8)
    s4_4_ax.spines[["top", "right"]].set_visible(False)
    if s4_4_metrica.startswith("t_"):
        s4_4_ax.set_ylim(bottom=0)
s4_4_ejes_ece[0].legend(frameon=False, fontsize=9, loc="best")
s4_4_fig_ece.suptitle("ECE por número de miembros (líneas: misma partición; negro: media ± desviación estándar)",
                      fontsize=11)
s4_4_fig_t.suptitle(f"Tiempo de ejecución por etapa ({s4_4_cpu})", fontsize=11)

# 3. Diagramas de fiabilidad de la primera partición.
s4_4_fig_fiab, s4_4_ejes_fiab = plt.subplots(2, len(S4_4_TAMANOS), figsize=(12, 7.2), constrained_layout=True,
                                             sharex=True, sharey=True)
for s4_4_fila, (s4_4_normalizacion, s4_4_metrica) in enumerate((("min-max", "ece"), ("recortado", "ece_recortado"))):
    for s4_4_col, s4_4_N in enumerate(S4_4_TAMANOS):
        s4_4_ax = s4_4_ejes_fiab[s4_4_fila, s4_4_col]
        s4_4_b = s4_4_cubetas[(1, s4_4_N)][s4_4_normalizacion]
        s4_4_centros = (s4_4_b["edges"][:-1] + s4_4_b["edges"][1:]) / 2
        s4_4_alfas = np.where(s4_4_b["count"] >= 30, 1.0, 0.35)  # cubetas con pocas observaciones, más claras
        s4_4_ax.bar(s4_4_centros, np.nan_to_num(s4_4_b["accuracy"]), width=0.09, edgecolor="white", linewidth=1,
                    color=[matplotlib.colors.to_rgba(s4_4_colores[s4_4_N], a) for a in s4_4_alfas])
        s4_4_ax.plot([0, 1], [0, 1], color="#8a8984", linewidth=1, linestyle="--")
        s4_4_ece_1 = s4_4_resultados.query("particion == 1 and N == @s4_4_N")[s4_4_metrica].item()
        s4_4_ax.set_title(f"N = {s4_4_N}, {s4_4_normalizacion}: ECE = {s4_4_ece_1:.3f}\n"
                          f"{s4_4_b['count'][-1] / s4_4_b['count'].sum():.1%} en la cubeta [0.9, 1]", fontsize=9)
        s4_4_ax.set_xlim(0, 1)
        s4_4_ax.set_ylim(0, 1.02)
        s4_4_ax.grid(True, color="#e8e7e2", linewidth=0.8)
        s4_4_ax.set_axisbelow(True)
        s4_4_ax.spines[["top", "right"]].set_visible(False)
for s4_4_ax in s4_4_ejes_fiab[-1]:
    s4_4_ax.set_xlabel("Confianza (1 − varianza normalizada)")
for s4_4_ax in s4_4_ejes_fiab[:, 0]:
    s4_4_ax.set_ylabel("Tasa de aciertos de la cubeta")
s4_4_fig_fiab.suptitle(f"Diagramas de fiabilidad, partición 1 (semilla {S4_SEMILLAS_70_30[0]}); "
                       "diagonal: calibración perfecta", fontsize=11)
plt.show()

#### Análisis

*Números de la ejecución de T7 (2026-11-27, Linux, Intel Core i3-1115G4, OpenBLAS con 4 hilos). Se revisan
en T8 con la ejecución final; los tiempos cambiarán con la máquina.*

**ECE (métrica principal, normalización min-max de D6).** Media ± desviación estándar sobre las 10
particiones: $0.0901 \pm 0.0012$ con $N = 10$, $0.0906 \pm 0.0007$ con $N = 100$ y $0.0950 \pm 0.0013$ con
$N = 1000$. Friedman indica una diferencia significativa entre las configuraciones ($\chi^2_F = 15.2$,
$p = 0.0005$, $W = 0.76$). Por pares, con la corrección de Holm, $N = 1000$ tiene más ECE que $N = 10$ y que
$N = 100$ en las 10 particiones ($p$ ajustada $= 0.006$ en los dos casos), mientras que la diferencia entre
$N = 10$ y $N = 100$ no es significativa ($p = 0.28$). La diferencia es clara, pero pequeña: unas cinco
milésimas.

**Qué mide aquí el ECE.** Las tasas de aciertos son $0.908$, $0.908$ y $0.902$, así que las tasas de error
($0.092$, $0.092$ y $0.098$) casi coinciden con el ECE. La causa es la normalización min-max: la mayor
varianza de prueba es unas 1800, 3100 y 35 000 veces la mediana (con $N = 10$, $100$ y $1000$), de modo que
casi todas las confianzas quedan cerca de 1. En la partición 1, la cubeta $[0.9, 1]$ reúne el 99.8 %, el
99.9 % y el 97.3 % de las observaciones. En esa cubeta la confianza media es casi 1 y la tasa de aciertos
ronda 0.91, y esa brecha, que es la tasa de error, domina la suma. El ECE de D6 mide entonces, sobre todo, cuánto se
equivoca el ensamble, y $N = 1000$ queda peor en buena parte porque acierta algo menos (0.902 frente a 0.908): cada miembro tiene unas 180
filas para unos 196 pesos, interpola sus datos (solución de norma mínima, D11) y generaliza peor.

**Sensibilidad a la normalización.** Con el rango recortado al percentil 99, el ECE baja con $N = 10$ y
$N = 100$ ($0.0638 \pm 0.0049$ y $0.0657 \pm 0.0023$) y sube con $N = 1000$ ($0.1031 \pm 0.0062$). La conclusión
no cambia: hay diferencia significativa (Friedman $p = 0.0004$, $W = 0.79$), $N = 1000$ es peor que las otras
dos en las 10 particiones ($p$ ajustada $= 0.006$) y $N = 10$ y $N = 100$ no se distinguen ($p = 0.13$). Lo que
sí cambia es el valor absoluto del ECE, que depende de cómo se convierte la varianza en confianza; por eso
D6 fija una normalización y esta sección la reporta junto con la alternativa. En los diagramas de
fiabilidad de la partición 1 con el rango recortado, con $N = 10$ y $N = 100$ la tasa de aciertos de todas las
cubetas está entre 0.73 y 0.93, incluso en las de confianza baja, que reúnen cientos de observaciones: la
varianza separa poco las predicciones correctas de las incorrectas, y el puntaje subestima la probabilidad
de acierto de las observaciones más inciertas (barras sobre la diagonal). Con $N = 1000$, las cubetas
intermedias casi se vacían y sus tasas son irregulares (0.11 en la cubeta $[0.7, 0.8)$, con 174
observaciones).

**La varianza sí informa, pero poco.** La varianza media de las predicciones incorrectas es 2.1, 2.8 y 3.5
veces la de las correctas (medianas sobre las particiones), y la correlación de Pearson entre varianza y
acierto es negativa en las 10 particiones con los tres $N$, aunque débil (medias $-0.045$, $-0.052$ y
$-0.100$). Con
$N = 1000$ varía mucho de una partición a otra (desviación $0.076$), porque unas pocas varianzas enormes la
dominan; Friedman no encuentra diferencias de correlación entre configuraciones ($p = 0.41$). La escala de la
varianza crece con $N$ (mediana $0.0005$, $0.006$ y $0.13$) porque cada miembro ve menos datos: no es
comparable entre configuraciones sin normalizarla.

**Tiempo de ejecución.** Total medio de $1.51 \pm 0.06$ s con $N = 10$, $1.35 \pm 0.06$ s con $N = 100$ y
$3.29 \pm 0.10$ s con $N = 1000$, sin el preprocesamiento (0.5 s por partición). El entrenamiento domina (1.46,
1.26 y 2.61 s). La evaluación crece con $N$ (0.045, 0.090 y 0.678 s), porque calcula $N$ puntajes por cada
una de las 77 302 observaciones de prueba. El ECE tarda unos 2 ms. El orden de las tres configuraciones se
repite en todas las particiones (Friedman $p = 4.5 \cdot 10^{-5}$, $W = 1$). El entrenamiento no crece en
proporción a $N$, porque cada fila entra en un solo ajuste. En cada ajuste, `lstsq` paga además un costo que
depende solo del número de pesos (la descomposición en valores singulares de una matriz de a lo sumo unos
$196 \times 196$). Ese costo se repite $N$ veces, lo que es coherente con que $N = 1000$ tarde casi el doble en
entrenar. No se investigó por qué $N = 100$ es unos 0.15 s más rápido que $N = 10$ en esta máquina.

**Conclusión.** Con miembros de mínimos cuadrados entrenados sobre particiones disjuntas, aumentar $N$ reduce
los datos de cada miembro. $N = 10$ y $N = 100$ dan prácticamente el mismo ECE, la misma tasa de aciertos y
tiempos parecidos. $N = 1000$ es peor en ECE (con las dos normalizaciones), acierta algo menos y tarda entre
2.2 y 2.4 veces más. Para este problema basta con $N = 10$ o $N = 100$; más miembros no mejoran la calibración.

**Limitaciones.** (1) Por los duplicados (D9), cerca del 44.5 % de las filas de prueba tienen una copia en
entrenamiento, así que la tasa de aciertos y el ECE son optimistas; el efecto es el mismo para los tres $N$.
(2) El ECE de D6 queda dominado por la tasa de error (ver arriba). (3) Los tiempos son de una máquina y una
ejecución. (4) Con 10 bloques y 3 configuraciones, la $p$ de Friedman usa la aproximación $\chi^2$, que con
tan pocos datos es solo aproximada; la conclusión se apoya también en las comparaciones por pares, cuyas $p$
de Wilcoxon son exactas.